# Figure 3A: Segment-pseudotime tree

This notebook draws one panel from the accepted compact tree layout. It reads the saved tree model, display coordinates and stage densities; it does not refit the tree or change cell annotations.

Replace every path placeholder before running. The drawing function was copied from the compact renderer in `04_tree-1.ipynb` and restricted to panel A. The original A coordinates are retained.


In [ ]:
#1. =========== Inputs and output locations
from pathlib import Path
import hashlib
import math
import os
import pickle
import subprocess
import sys
import numpy as np

# Set the working root and replace every file or directory placeholder below.
ROOT = Path(".")
# Set tree_results-1.pkl, produced by the accepted tree packaging step in 04_tree-1.ipynb.
RESULT = ROOT
# Set the PDF output file path for Figure 3A.
PDF_PATH = ROOT
# Set the PNG output file path for Figure 3A.
PNG_PATH = ROOT
# Set the output directory and temporary working directory.
OUTPUT_DIR = ROOT
WORK = ROOT
# Set the Arial regular and bold font file paths.
FONT_REGULAR = ROOT
FONT_BOLD = ROOT
# Set the pdftoppm executable path used for the original 600-DPI PNG export.
PDFTOPPM = ROOT
# Set an optional ReportLab dependency directory if it is not installed in this environment.
REPORTLAB_DEPENDENCIES = ROOT


In [ ]:
#2. =========== Read the saved accepted tree
# tree_results-1.pkl contains the accepted graph, cell annotations, display coordinates,
# segment-gradient colors and stage density arrays from the preceding tree workflow.
with RESULT.open("rb") as handle:
    bundle = pickle.load(handle)
display = bundle["publication_display"]
density = bundle["stage_density_display"]
assert len(bundle["obs"]) == 7355
assert float(display["tau_css30"]) == .25
assert display["horizontal_layout"]["VGN_root_span_scale"] == .5


## Drawing function

The accepted branch geometry, colors, scatter order and within-panel spacing are retained.


In [ ]:
def render_tree_all_compact(bundle, display, density, outdir, work):
    #1. =========== Display-only settings
    size_in = (232.0/72., 7.6)
    dpi = 600
    tree_top, tree_height = 500., 420.
    panel_left = {"A": 24., "B": 256.}
    label_font, legend_font = 11.3, 7.8
    point_area, point_alpha = 1.4, .72
    legend_alpha = 1.        # Opaque nominal RGB avoids seams between transparent vector strips.
    legend_x, legend_top, legend_step = 100., 497., 12.
    swatch_width, swatch_height, gradient_steps = 20., 7., 256
    density_left, density_width = 399., 77.
    stage_x, stage_top, stage_step = 351., 497., 7.

    #2. =========== Runtime and immutable source arrays
    runtime = ROOT  # Set an optional runtime directory if the import fallback is needed.
    try:
        import reportlab
    except ImportError:
        sys.path.append(str(REPORTLAB_DEPENDENCIES))
    from reportlab.pdfgen import canvas
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.lib.colors import Color, HexColor
    from PIL import Image
    for name, font_path in (("Arial", FONT_REGULAR), ("Arial-Bold", FONT_BOLD)):
        if name not in pdfmetrics.getRegisteredFontNames():
            pdfmetrics.registerFont(TTFont(name,str(font_path)))
    outdir, work = Path(outdir), Path(work)
    outdir.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    pdf, png = PDF_PATH, PNG_PATH
    width, height = np.asarray(size_in)*72
    xy = np.asarray(display["cell_xy"])
    ny = np.asarray(display["node_y"])
    nx = {int(k):float(v) for k,v in display["node_x"].items()}
    segments = bundle["segments"]
    sx = np.asarray(display["segment_x"])
    coloring = bundle["segment_gradient_coloring"]
    rgb = np.asarray(coloring["color_rgb_uint8"])
    order = np.asarray(bundle["plot_order"])
    stages = ("E9.5","E10.5","E11.5","E13.5","E14.5","E15.5","E16.5","E17.5","E18.5")
    stage_colors = ("#f5b888","#f2a985","#ee9982","#e58780","#cd717e","#b55c7d","#914679","#653175","#3a1c71")
    palette = dict(zip(stages,stage_colors))
    stage_values = bundle["obs"].stage_standardized.astype(str).to_numpy()
    digest = lambda value: hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()
    frozen = {"cell_xy":digest(xy), "node_y":digest(ny), "rgb":digest(rgb),
              "F":digest(bundle["model"]["F"]), "B":digest(bundle["model"]["B"]),
              "t":digest(bundle["projection"]["t"])}
    maximum = float(ny.max())
    sy = tree_height/maximum
    previous_sy = (.57*8.5*72)/(1.365*maximum)
    uniform_scale = sy/previous_sy
    x_scale = (2.8*72/4)*uniform_scale
    assert len(segments)==11 and len(xy)==7355
    assert float(display["tau_css30"])==.25
    assert display["horizontal_layout"]["VGN_root_span_scale"]==.5
    radius = math.sqrt(point_area)/2
    scenes, leaders, gradient_records, physical = [], [], [], {}

    def color(value,alpha=1.):
        value=HexColor(value) if isinstance(value,str) else Color(*map(float,np.asarray(value)[:3]))
        value.alpha=alpha
        return value

    def dx(value): return 1.5+.5*(np.asarray(value)-1.5)

    def cell_map(panel,x,y):
        return panel_left[panel]+x_scale*(np.asarray(x)-.75), tree_top-sy*np.asarray(y)

    def node_map(panel,x,y): return cell_map(panel,dx(x),y)

    c = canvas.Canvas(str(pdf),pagesize=(width,height),pageCompression=1,invariant=1,
                      initialFontName="Arial",initialFontSize=8)
    c.setTitle("Figure 3A: Segment pseudotime")
    c.setAuthor("Developmental map of auditory neurons")
    c.setSubject("Figure 3A; accepted graph and original annotations preserved")

    def text(value,x,y,size=8.,align="left",font="Arial",role="label",panel=None,top=False):
        value=str(value); lines=value.split("\n"); leading=1.13*size
        asc,desc=pdfmetrics.getAscentDescent(font,size)
        baseline=y-asc if top else y
        widths=[pdfmetrics.stringWidth(v,font,size) for v in lines]
        t=c.beginText();t.setFont(font,size);t.setFillColor(color("#202020"))
        for j,(line,w) in enumerate(zip(lines,widths)):
            offset=0. if align=="left" else (-w/2 if align=="center" else -w)
            t.setTextOrigin(x+offset,baseline-j*leading);t.textOut(line)
        c.drawText(t)
        w=max(widths);left=x if align=="left" else (x-w/2 if align=="center" else x-w)
        rec={"text":value,"bbox":[float(left),float(baseline-(len(lines)-1)*leading+desc),
             float(left+w),float(baseline+asc)],"role":role,"panel":panel,"font":font,"size":size}
        scenes.append(rec);return rec

    def line(x0,y0,x1,y1,fill="#727782",lw=.7):
        c.setStrokeColor(color(fill));c.setLineWidth(lw)
        c.line(float(x0),float(y0),float(x1),float(y1))

    def dot(x,y,r,fill,stroke=None,lw=.5,alpha=1.):
        c.setFillColor(color(fill,alpha))
        if stroke is not None:c.setStrokeColor(color(stroke));c.setLineWidth(lw)
        c.circle(float(x),float(y),float(r),fill=1,stroke=int(stroke is not None))

    #3. =========== Draw the selected tree panel
    for panel in ("A",):
        for k,seg in enumerate(segments):
            a,z=int(seg["start"]),int(seg["end"])
            xa,ya=node_map(panel,sx[k],ny[a]);xz,yz=node_map(panel,sx[k],ny[z])
            line(xa,ya,xz,yz)
            if nx[a]!=sx[k]:
                x0,y0=node_map(panel,nx[a],ny[a]);line(x0,y0,xa,ya)
        px,py=cell_map(panel,xy[:,0],xy[:,1])
        c.setFillAlpha(point_alpha)
        for idx in order:
            fill=rgb[idx]/255. if panel=="A" else palette[stage_values[idx]]
            dot(px[idx],py[idx],radius,fill,alpha=point_alpha)
        c.setFillAlpha(1.)
        root=int(bundle["projection"]["root"])
        rx,ry=node_map(panel,nx[root],ny[root]);dot(rx,ry,1.7,"#ffffff","#404040",.6)
        text("Root",rx,ry+6.,7.2,align="center",role="endpoint",panel=panel)
        vg=int(bundle["vgn_tip"])
        vx,vy=node_map(panel,nx[vg],ny[vg]);dot(vx,vy,1.7,"#ffffff","#404040",.6)
        text("VGN candidate-\nenriched",panel_left[panel]+11.,vy-9.,8.,align="center",top=True,role="endpoint",panel=panel)
        # The display labels identify the same existing ends; obs/state is never edited.
        positions={"Ib":(panel_left[panel]+11.,18.),"Ic":(panel_left[panel]+.5*x_scale,66.),
                   "Ia":(panel_left[panel]+x_scale,33.),"II":(panel_left[panel]+1.5*x_scale,66.)}
        names={"Ib":"Immature Ib","Ic":"Emerging Ic\nImmature Ic","Ia":"Immature Ia","II":"Immature II"}
        for fate in ("Ib","Ic","Ia","II"):
            tip=int(bundle["figures"][fate]["tip"])
            tx,ty=node_map(panel,nx[tip],ny[tip]);dot(tx,ty,1.7,"#ffffff","#404040",.6)
            lx,ly=positions[fate]
            c.setDash(.9,1.8);line(tx,ty-2.,lx,ly+3.,"#929292",.45);c.setDash()
            leaders.append({"panel":panel,"fate":fate,"points":[float(tx),float(ty-2),float(lx),float(ly+3)]})
            text(names[fate],lx,ly,label_font,align="center",top=True,role="endpoint",panel=panel)
        gx,gy=node_map(panel,np.asarray([nx[i] for i in range(len(ny))]),ny)
        physical[panel]={"cell_xy_pt_sha256":digest(np.column_stack([px,py])),
                         "node_xy_pt_sha256":digest(np.column_stack([gx,gy])),
                         "node_paper_coordinates":{str(i):[float(gx[i]),float(gy[i])] for i in range(len(ny))},
                         "normalized_display_sha256":digest(xy)}
    #4. =========== Segment-pseudotime gradient legend
    short_labels={"early_common":"CVG common","VGN_candidate":"VGN candidate-enriched",
                  "SGN_candidate":"SGN candidate-enriched","connector":"Component connection",
                  "SGN_common":"SGN common trajectory","IbIc":"Ib/Ic shared segment",
                  "Ib":"Ib-enriched terminal","Ic":"Emerging Ic-enriched terminal",
                  "IaII":"Ia/II shared segment","Ia":"Ia-associated terminal","II":"II-associated terminal"}
    for j,k in enumerate(coloring["legend_order"]):
        row,seg=coloring["segments"][k],segments[k]
        yy=legend_top-j*legend_step
        if seg["role"]=="connector":
            line(legend_x,yy,legend_x+swatch_width,yy,"#8b8b8b",.7)
            steps=0
        else:
            a=np.asarray([int(row["start_color"][i:i+2],16) for i in (1,3,5)])/255.
            z=np.asarray([int(row["end_color"][i:i+2],16) for i in (1,3,5)])/255.
            c.setFillAlpha(legend_alpha)
            for m,v in enumerate(np.linspace(a,z,gradient_steps)):
                c.setFillColor(color(v,legend_alpha))
                c.rect(legend_x+swatch_width*m/gradient_steps,yy-swatch_height/2,
                       swatch_width/gradient_steps+.003,swatch_height,fill=1,stroke=0)
            c.setFillAlpha(1.)
            c.setStrokeColor(color("#383838"));c.setLineWidth(.45)
            c.rect(legend_x,yy-swatch_height/2,swatch_width,swatch_height,fill=0,stroke=1)
            steps=gradient_steps
        text(short_labels[seg["role"]],legend_x+swatch_width+6.,yy-2.5,legend_font,role="legend",panel="A")
        gradient_records.append({"metric_segment":int(k),"role":seg["role"],"bands":steps,
                                 "start_color":row["start_color"],"end_color":row["end_color"],
                                 "alpha":legend_alpha,"bbox":[legend_x,yy-swatch_height/2,legend_x+swatch_width,yy+swatch_height/2]})
    text("A",12.,522.,28.,role="panel_letter",panel="A")
    text("Segment pseudotime",39.,524.,15.2,role="panel_title",panel="A")

    #5. =========== Check layout and unchanged data
    for item in scenes:
        x0,y0,x1,y1=item["bbox"]
        assert min(x0,y0)>=.0 and x1<=width and y1<=height,("Text outside page",item)
    for i,a in enumerate(scenes):
        for b in scenes[i+1:]:
            x0,y0,x1,y1=a["bbox"];u0,v0,u1,v1=b["bbox"]
            assert x1<=u0 or u1<=x0 or y1<=v0 or v1<=y0,("Text overlap",a,b)
    for panel in ("A",):
        xs,ys=cell_map(panel,xy[:,0],xy[:,1])
        for item in scenes:
            if item["role"]!="legend" or item["panel"]!="A":continue
            x0,y0,x1,y1=item["bbox"]
            assert not np.any((xs>=x0-radius)&(xs<=x1+radius)&(ys>=y0-radius)&(ys<=y1+radius)),("Legend covers points",item)
    assert all(v["bands"]==256 for v in gradient_records if v["role"]!="connector")
    assert frozen["cell_xy"]==digest(xy) and frozen["node_y"]==digest(ny) and frozen["rgb"]==digest(rgb)
    assert frozen["F"]==digest(bundle["model"]["F"]) and frozen["B"]==digest(bundle["model"]["B"])
    assert frozen["t"]==digest(bundle["projection"]["t"])
    c.showPage();c.save()
    poppler=Path(os.environ.get("TREE_PDFTOPPM",str(PDFTOPPM)))
    subprocess.run([str(poppler),"-singlefile","-r",str(dpi),"-png",str(pdf),str(png.with_suffix(""))],check=True,capture_output=True)
    with Image.open(png) as image:
        pixels=list(image.size);actual_dpi=list(image.info["dpi"])
        assert all(abs(a-round(b*dpi))<=1 for a,b in zip(pixels,size_in))
        assert all(abs(v-dpi)<.05 for v in actual_dpi);image.verify()
    artifact={"pdf":str(pdf),"png":str(png),"inches":list(size_in),"pixels":pixels,"dpi":actual_dpi,
              "pdf_sha256":hashlib.sha256(pdf.read_bytes()).hexdigest(),"png_sha256":hashlib.sha256(png.read_bytes()).hexdigest(),
              "expected_clips":0,"expected_images":0,"expected_forms":0,"required_font_styles":["regular"],
              "semantic_text_objects":len(scenes)}
    spec={"layout":"Compact A/B, no axes or grid, unchanged accepted VGN/SGN topology",
          "whole_size":list(size_in),"tree_top_pt":tree_top,"tree_height_pt":tree_height,
          "x_scale_pt":x_scale,"y_scale_pt":sy,"uniform_scale_from_previous_tree":uniform_scale,
          "panel_left_pt":panel_left,"point_area":point_area,"point_alpha":point_alpha,
          "legend_alpha":legend_alpha,"legend_rendering":"Opaque nominal RGB gradients; no translucent strip seams",
          "display_xy_sha256":frozen["cell_xy"],"node_y_sha256":frozen["node_y"],
          "original_segment_rgb_sha256":frozen["rgb"],"original_t_sha256":frozen["t"],
          "VGN_preserved":True,"segment_legend":gradient_records,"scene":scenes,"leaders":leaders,
          "density_bandwidth_display":float(density["bandwidth_display"]),"density_peak_normalization":1,
          "expected_clips":0,"expected_images":0,"expected_forms":0,"fonts":["ArialMT"],"PNG_DPI":dpi,
          "other_four_tree_layouts":"Unchanged; shared normalized geometry, independent page layout"}
    return {"artifact":artifact,"scene":scenes,"physical":physical,"render_spec":spec}


In [ ]:
#4. =========== Export Figure 3A
# This call writes only the selected panel to the configured PDF and PNG paths.
rendered = render_tree_all_compact(bundle, display, density, OUTPUT_DIR, WORK)


In [ ]:
#5. =========== Preview the exported panel
from IPython.display import display as show, Image
show(Image(filename=str(PNG_PATH), width=600))



# Original shared tree analysis sources
These existing source sections are retained without changing their functions,
parameters, CLI entry points, or original joint figure layouts. They are separate
original execution contexts; this merged notebook is not a Run All driver.
Configure each ROOT storage location independently. For the original local-CVG
reference-notebook input, use this notebook containing the original reference
cell below. Different ROOT occurrences refer to different inputs or outputs.



## Original reference cell
This is the complete original reference cell. Its plain JSON memberships remain
available to the original local-CVG notebook parser; no replacement loader is added.


In [ ]:
# Frozen historical memberships are used for diagnostics, not as an SGN identity gate.
# Sources: Source_Early-1.ipynb and Source-run-log-1.txt in the embryonic integration directory.
FROZEN_E9_MEMBERSHIP = json.loads('{"cells": ["GSE178931__GSM5401072__AAAGAACCAGAGGCTA.1", "GSE178931__GSM5401072__ACAAGCTCACCAGCTG.1", "GSE178931__GSM5401072__ACACAGTCAATCCTAG.1", "GSE178931__GSM5401072__ACACCAAGTGCCTACG.1", "GSE178931__GSM5401072__ACACTGAAGAGCATCG.1", "GSE178931__GSM5401072__ACAGCCGTCCTCTTTC.1", "GSE178931__GSM5401072__ACCGTTCAGCGGTAGT.1", "GSE178931__GSM5401072__ACGATGTTCCAAGAGG.1", "GSE178931__GSM5401072__ACGCACGAGGGCAGGA.1", "GSE178931__GSM5401072__ACGGTTATCGGACCAC.1", "GSE178931__GSM5401072__ACGTCCTCAGGCATGA.1", "GSE178931__GSM5401072__AGAAATGTCTGGGAGA.1", "GSE178931__GSM5401072__AGACAGGCAGTCAGAG.1", "GSE178931__GSM5401072__AGGACTTCACCCTATC.1", "GSE178931__GSM5401072__AGGTCTATCCCACAGG.1", "GSE178931__GSM5401072__AGTCAACTCTGCAGCG.1", "GSE178931__GSM5401072__AGTGATCCAAGCAGGT.1", "GSE178931__GSM5401072__ATCCACCGTTCGGTTA.1", "GSE178931__GSM5401072__ATCCCTGCAAGTCCAT.1", "GSE178931__GSM5401072__ATCCGTCTCTGTGCGG.1", "GSE178931__GSM5401072__CAACAGTCACGTATAC.1", "GSE178931__GSM5401072__CAACCTCGTATGAGAT.1", "GSE178931__GSM5401072__CAAGAGGGTGCTAGCC.1", "GSE178931__GSM5401072__CAATACGCAAGTGGGT.1", "GSE178931__GSM5401072__CAATACGGTATTCTCT.1", "GSE178931__GSM5401072__CAATGACCACGCCAGT.1", "GSE178931__GSM5401072__CAATTTCAGACCGCCT.1", "GSE178931__GSM5401072__CACCGTTCAGGCTTGC.1", "GSE178931__GSM5401072__CACGGGTAGTCAGCGA.1", "GSE178931__GSM5401072__CAGCAGCTCCATTGTT.1", "GSE178931__GSM5401072__CAGCGTGAGACCTCCG.1", "GSE178931__GSM5401072__CAGGTATCACCTATCC.1", "GSE178931__GSM5401072__CAGTGCGGTCAATGGG.1", "GSE178931__GSM5401072__CATCGCTCACCAGTAT.1", "GSE178931__GSM5401072__CATCGTCAGTTTGGCT.1", "GSE178931__GSM5401072__CATCGTCCAAGAGGCT.1", "GSE178931__GSM5401072__CATGAGTCAAACTGCT.1", "GSE178931__GSM5401072__CATTTCACAAAGGCTG.1", "GSE178931__GSM5401072__CCAATGATCTTCACAT.1", "GSE178931__GSM5401072__CCACAAAAGTCTTCCC.1", "GSE178931__GSM5401072__CCACGTTTCCGTTTCG.1", "GSE178931__GSM5401072__CCCATTGAGTAAACAC.1", "GSE178931__GSM5401072__CCGATCTGTCTACATG.1", "GSE178931__GSM5401072__CCTCATGAGCAGTACG.1", "GSE178931__GSM5401072__CGTGAATGTACAAAGT.1", "GSE178931__GSM5401072__CTACAGACATCGGATT.1", "GSE178931__GSM5401072__CTACCTGGTTGCTGAT.1", "GSE178931__GSM5401072__CTGAGGCCAGCCCACA.1", "GSE178931__GSM5401072__CTGCGAGCATGCCGCA.1", "GSE178931__GSM5401072__CTGGTCTGTGATTCTG.1", "GSE178931__GSM5401072__CTGTAGAGTTCTGACA.1", "GSE178931__GSM5401072__CTGTGAAAGAAACTAC.1", "GSE178931__GSM5401072__CTTCCGATCCCGATCT.1", "GSE178931__GSM5401072__CTTCCTTAGTGGACTG.1", "GSE178931__GSM5401072__GAAGCGAAGATACCAA.1", "GSE178931__GSM5401072__GAATCGTGTTTCCCAC.1", "GSE178931__GSM5401072__GACGCTGAGGAGCAAA.1", "GSE178931__GSM5401072__GACGTTATCCAGTTCC.1", "GSE178931__GSM5401072__GACTGATGTAATGTGA.1", "GSE178931__GSM5401072__GAGTCATTCCGCAAAT.1", "GSE178931__GSM5401072__GATGTTGGTTGACGGA.1", "GSE178931__GSM5401072__GCACGGTCAACGGCTC.1", "GSE178931__GSM5401072__GCCAGGTCACGTGTGC.1", "GSE178931__GSM5401072__GCCTGTTCAGGTCCGT.1", "GSE178931__GSM5401072__GCGTGCATCGAGATAA.1", "GSE178931__GSM5401072__GGAAGTGCAATGGGTG.1", "GSE178931__GSM5401072__GGCTTGGAGTTAGTAG.1", "GSE178931__GSM5401072__GGGACCTGTGCGACAA.1", "GSE178931__GSM5401072__GGGCGTTCACAGTACT.1", "GSE178931__GSM5401072__GGGTCTGCAGAAGCGT.1", "GSE178931__GSM5401072__GGTAACTAGGTTAGTA.1", "GSE178931__GSM5401072__GGTCACGGTAGATCGG.1", "GSE178931__GSM5401072__GGTCTGGCAATGTCTG.1", "GSE178931__GSM5401072__GGTGGCTAGACCATAA.1", "GSE178931__GSM5401072__GGTTGTAGTGCCTATA.1", "GSE178931__GSM5401072__GTAGGTTCAAGGGTCA.1", "GSE178931__GSM5401072__GTCAGCGGTGTACGCC.1", "GSE178931__GSM5401072__GTCATCCTCTGTCCCA.1", "GSE178931__GSM5401072__GTCCACTGTGCTCTCT.1", "GSE178931__GSM5401072__GTCGAATCAATAGTCC.1", "GSE178931__GSM5401072__GTCGTTCAGGTATAGT.1", "GSE178931__GSM5401072__GTGAGCCAGGTCTTTG.1", "GSE178931__GSM5401072__GTTACGACAGAACCGA.1", "GSE178931__GSM5401072__TACGGGCTCAAGATAG.1", "GSE178931__GSM5401072__TAGAGTCCAACTGTGT.1", "GSE178931__GSM5401072__TCCACCACATTGTAGC.1", "GSE178931__GSM5401072__TCGAACATCCGATAGT.1", "GSE178931__GSM5401072__TCGAAGTAGGTGCGAT.1", "GSE178931__GSM5401072__TCGGGTGCACCGTACG.1", "GSE178931__GSM5401072__TCTTAGTAGTAGCATA.1", "GSE178931__GSM5401072__TCTTGCGCAGAAACCG.1", "GSE178931__GSM5401072__TGAATCGTCCATTGCC.1", "GSE178931__GSM5401072__TGACCCTGTAGCTTTG.1", "GSE178931__GSM5401072__TGATCTTCAGAGCTAG.1", "GSE178931__GSM5401072__TGCAGTAAGTAGACCG.1", "GSE178931__GSM5401072__TGCGATAGTTGGTACT.1", "GSE178931__GSM5401072__TGCGGCAAGCAAGGAA.1", "GSE178931__GSM5401072__TGCTGAACATGTGCCG.1", "GSE178931__GSM5401072__TGGTGATCAGAGTGTG.1", "GSE178931__GSM5401072__TGTAACGTCCTTCTTC.1", "GSE178931__GSM5401072__TGTTGGAAGTTGCCCG.1", "GSE178931__GSM5401072__TTCACGCCAGGCGATA.1", "GSE178931__GSM5401072__TTCTAACGTAGTGGCA.1", "GSE178931__GSM5401072__TTGTGGAAGAAGGTAG.1", "GSE178931__GSM5401072__TTTACCATCCATTGTT.1", "GSE178931__GSM5401072__TTTACGTTCAAGAGGC.1"], "reviewed_labels": {"GSE178931__GSM5401072__AAAGAACCAGAGGCTA.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__ACGCACGAGGGCAGGA.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__CCAATGATCTTCACAT.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__CCGATCTGTCTACATG.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__CTACAGACATCGGATT.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__CTGGTCTGTGATTCTG.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__GACTGATGTAATGTGA.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__GAGTCATTCCGCAAAT.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__GTCGTTCAGGTATAGT.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__TCGAAGTAGGTGCGAT.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__TCTTGCGCAGAAACCG.1": "Otic neurogenic / CVG candidate", "GSE178931__GSM5401072__TGCTGAACATGTGCCG.1": "Otic neurogenic / CVG candidate"}}')
FROZEN_REFERENCE_ROWS = json.loads('[{"cell_id": "GSE178931__GSM5401073__AAACGAATCCGTCACT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__AACCACAAGGCATGCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__AACCATGAGATGGTAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ACATCGACACAGAGAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ACCGTTCCACCCTGTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ACCTGAAGTGTGGACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ACCTGTCGTATCAAGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ACGGTCGGTGAGTGAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ACTGTCCGTCGAACGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__AGACAGGGTTTGAAAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__AGCATCAGTCCAGCGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__AGCTCAAAGGCTTTCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__AGTCATGTCACCGGGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ATACTTCTCTAACACG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ATCGCCTTCTTCTGGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ATGATCGTCGTAGAGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ATGCGATGTGCATCTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__ATTTCTGCACATTCTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CAAGAGGCACTGTTCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CAAGCTAAGCTAATGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CACATGAGTAGCTGCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CACTAAGAGGTACTGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CATCCCACATGACTAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CCCGGAACACACGGAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CCCTAACCAGTATACC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CCGGTGAAGCGTCTGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CCTACGTAGTGATAGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CCTCAGTTCAGGCGAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CCTCCTCGTAGGAGTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CGCGTGAAGCTGGCCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CGGAATTAGAAGCGCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CGTGATAGTCACAATC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CTAGGTATCCGGCAGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CTCAAGAGTACGAGCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CTCATGCTCCTAGCCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CTCTCGATCAGGGTAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CTGCCATTCCGTAGGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__CTTCCGAAGCTGTTCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GAAGAATCACCAGGTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GAGACCCTCTGAGCAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GAGTCTAGTCCATACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GAGTGTTGTGGTAACG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GAGTTTGTCCCAGGAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GATCACAAGGGACTGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GATGACTTCGATCCAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GATGATCCACGTACAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GATTCTTAGCTGACCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GCAACCGAGCTCGGCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GCAGCTGGTACATTGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GCCGTGAAGTGATGGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GCGATCGCATCCTAAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GCGTTTCCATATACCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GCTCAAACAATTCGTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GGACGTCGTCAAATCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GGGTAGAGTGGTCTCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GTAACCAAGTGCCTCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GTCAGCGAGTATGGAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GTCATTTGTGTAGTGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GTCTAGAGTTTGAACC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GTGCTGGGTTATTCTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GTTACAGCAACGATCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__GTTAGTGAGGTACTGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TACACCCGTTCTTAGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TCACGGGTCTGTGCGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TCATGAGTCCTACCGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TCCCATGGTTAGGGAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TGACGCGTCTTGGATG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TGAGACTTCTAGTTCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TGCATGAGTAGAGATT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TGGAACTGTCACTAGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TGGAGGATCCCTTGTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TGGATCAAGCTAATCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TGGTTAGTCAGTCATG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TTCCTTCCATCTAGAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TTCGCTGAGCCGGATA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TTCTTGAAGACTGTTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TTGTTTGTCAACCGAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TTTCCTCGTTGGGAAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__TTTGGAGCAATTTCTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AAAGTGAAGGAACGCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AACACACAGGAGTACC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AACGTCAGTCCAACGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AAGCCATAGGCACTCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AAGCCATCAAGGCAAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ACAAGCTTCTCTCCGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ACACGCGTCTCATAGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ACATCGACAATGTCAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ACGGTTATCCGTTGGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AGAGAATCAACTCCAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AGATAGAAGCCGATCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AGCGCTGGTAGGGTAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__AGTAACCTCGGTTAGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ATATCCTCAGGACTTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ATGAGGGCAGTGGTGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ATGCCTCCACGTACTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ATGCCTCGTTCTCCAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ATGGTTGGTATACAGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ATGTCTTTCGAACGCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__ATTTCACTCTATGTGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CAAGCTAAGGCCTAGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CAAGGGAGTCCTCCAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CACAACAAGTCTAGAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CACCAAAGTCCTGAAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CAGCGTGGTTGCATCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CCCTGATGTTCCGCAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CCGATGGCACTCATAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CGAGGCTCAGGTGACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CGCATGGCAGCTCATA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CGGAACCTCATTGGTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CGTAAGTTCAGGAAAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CTATCTACATCGCCTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CTGAATGCAAGCAGGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__CTGTATTTCAACTCTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GAAATGAAGTTGCATC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GACCCAGGTAAGTTAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GAGTTTGCAGCAATTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GCATCTCAGGATGGCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GCTCAAATCACCGCTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GCTGAATTCGGTATGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GGAATCTTCTTCCCGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GGAGATGGTGTGGTCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GGAGGATGTCGGAAAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GGTAGAGAGGATGCGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GTAGATCCAGCCGTCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GTAGATCCATCAGCGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GTCACGGGTGTTACAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GTGATGTTCACTGAAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GTTGAACAGAATCGTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__GTTGTAGGTTTGCCGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TACTTCACAGTCAGAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TATTGGGGTGGTTTGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCAATCTCAGCGTTGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCACGCTCAGCATGCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCACTATGTCCTTTGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCACTCGGTACAAACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCATGAGGTAGCTTTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCCCAGTAGGTGGGTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCCTGCAGTGAGACGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCGCTTGTCAGTGTGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCTAACTTCCACGTGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TCTCTGGAGACAACAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TGAATCGAGGAACGAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TGTGAGTAGTAACGAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TGTTTGTAGCCGTTAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTACGCCCACCCATAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTACGCCGTTGGGACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTCTTGACATCAGCTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTGACCCGTGCCCGTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTGATGGTCAAGAGGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTGGATGTCAAGTAAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTGGGCGCATGAGGGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTGGGTATCAAAGACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTGTTCACAATTGAAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401074__TTTCACATCAGATTGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AAGTACCAGATGGGCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACAGCCGAGGTAACTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACATCGACAAACACCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACCAACAGTCGTTTCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACCCAAACATCTCCCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACGGGTCAGACAACAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACGTAACAGTTGAAGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACGTCCTGTACAGTAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACTATTCTCTAGCATG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ACTCCCAAGATCCCAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGATAGAGTTTGAACC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGATCCATCCGCTGTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGCCACGGTCTGTCAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGGAATACAGGCACAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGGACTTAGTTGCATC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGTAGCTGTATTCCTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGTCTCCAGTCGGGAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__AGTCTCCTCTTCTGTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ATACCTTGTCGTTGCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ATCACGATCGGAGTAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__ATCCCTGCAGGATGAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CAGATTGGTACCCAGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CATGCAAAGAAGAACG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CATTTCAAGGTAGCAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CCACCATTCGGAGCAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CCACTTGCAAGTGGTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CCCTCTCGTTGATGTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CCGGGTACAAAGGAGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CCGTTCAAGAACAGGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CGAAGTTTCGCTAATG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CGAGTGCGTCGACGCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CGGGTCACACAAGTGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTCAACCCAGCTATTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTCAGGGCATGGGATG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTCCCAAGTGTACAGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTCCCTCTCTCATGGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTCTCGACAAATGCGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTGAATGGTGTCTAAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTGGACGGTTGTATGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__CTGTCGTTCGTACACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GAAACCTTCCTAACAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GAACTGTCAGGCAATG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GAAGCGAAGGCCATAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GAATAGAGTCCCTGAG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GATAGAACAATCGCCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GATCAGTTCCATTTCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GATCCCTCATGAGAAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GATGATCAGGTAAAGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GCATCTCGTGGAATGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GCCGATGAGCGTGAAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GCTACAAAGCTGGTGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GCTGCAGAGTTGGAAT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GCTGCAGTCCGAGGCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GCTTTCGGTCAAGGCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GGATGTTCAGGACTTT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GGCTTGGCATCACCAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GGGAGATAGCAACTTC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GGGAGTACAGCGAACA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GGGCCATGTTGACGGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GGTGTTAAGGAACTCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GGTTAACTCTGACAGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GTAGGAGCAATTTCCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GTCTCACAGCATTTCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GTGCAGCAGTTGTCAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GTGGAGACATAGTCGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GTGGGAACAAGTGGGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GTGTTAGGTAGAGATT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__GTTATGGCAAATTAGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TAACCAGTCTAGTGTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TAAGTCGGTAGCGTCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TACGCTCTCTAGTGTG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TACTTGTAGAAGCGGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TAGAGTCCAGGGTTGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TCAGTCCAGTGGTTAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TCATATCGTTTAGTCG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TCATGAGTCTCAGTCC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TCATTCATCTACCCAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TCGACGGCACAGCATT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TCGTGGGCACGTAACT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TCTTGCGAGCAAATCA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TGAGGTTCAGTGGTGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TGCATGAAGGTAGACC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TGCGACGTCAGACAAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TGTGATGAGTTTGAGA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TGTTGGAAGGCTATCT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TGTTTGTTCGGACCAC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TTACCATTCCGACGGT.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TTACCGCCAGCTGTGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TTCTAGTCATTGCCGG.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TTCTTCCAGGAATCGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TTGAGTGCAGACAAGC.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TTGAGTGTCGATGCTA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401075__TTGGATGTCTTCGGAA.1", "stage": "E11.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401073__AGACACTCAAGTCCCG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__AGATGCTTCTCTATAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__AGCTACAAGCTCACTA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__AGTTCCCGTTTACCTT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__ATCGGCGCAGCAGTGA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__CAATACGTCGGTTCAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__CACACAAAGGTAGTCG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__CACATGAAGCCGCACT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__CACGAATAGGGTGAAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__CGGACACCATGTCAGT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__CTATAGGTCGGAACTT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GAGCCTGCATCATGAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GAGGGATAGACCTTTG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GCAACATTCGTAGCTA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GCACGGTGTTCCACGG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GCGATCGAGGGATCGT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GCTGCAGAGAGTGACC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GTTCGCTAGGGTACAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__GTTGAACCACTTCAAG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__TACTTACTCATCTGTT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__TCGGGACAGTGCTACT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__TCTGGCTGTTGAGTCT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__TGCTTGCTCATCTCTA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__TGGATCAGTTCAGTAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__TTACGCCCAACGTAAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__TTAGGGTAGACGAGCT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__ACTGTGATCCGTTGAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__AGATGAATCACGATCA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__AGCATCACATAATCCG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__AGCTTCCAGACCTTTG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__AGTTCGATCACCGGGT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__ATCACTTAGAGTTGAT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__ATCGTAGTCATCCTAT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__ATGGGTTGTGACCTGC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__CAGCGTGGTTAGGCCC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__CGAATTGTCCGGACGT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__CGCCAGACACTACTTT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__GAGTTACAGCCTGAAG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__GCTACCTCACCAATTG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__GGGACTCGTAGTGTGG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__GGGAGTAAGTATGCAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__TAAGTCGGTCACAATC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__TCGGATAAGCTTCATG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__TCGGTCTGTAAGATCA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__TGTTCATTCGCCGAAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__TTCGCTGCATTCACCC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401074__TTCTCTCAGGATTACA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__AACAAAGGTCTTACTT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__AACTTCTGTATCGAAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__AACTTCTGTCACCTTC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__AATAGAGGTACCGTCG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__AATTCCTGTTATCTTC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__ACCCAAACACTTCAGA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__AGATGAAAGAAAGTCT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__AGGTTGTAGCAATAGT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__ATCACAGCAGGAACCA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__CGATCGGGTCTTTCTA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__CGGACACTCTCGTCAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__CGTGATAAGTATGGAT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__CTCCATGTCAAACTGC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GACCCTTAGGCATCTT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GAGAAATTCTTGGATG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GAGGGATTCCGCAACG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GATGAGGCAGAACTAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GCATTAGGTAGGTACG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GCCCAGATCTGAGATC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GCTTTCGTCTGGACTA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GGCTTGGAGTCTGCGC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GTAGAAACATAGAAAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GTGGGAACAGGTATGG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GTTGAACCAAAGGCTG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__GTTGAACTCAGCTGAT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TAACCAGGTGGCGTAA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TAAGCACCAGTGTGGA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TCCTTTCAGAGCTGAC.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TGATGCAGTCAGTTTG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TGATTTCAGAGTTGCG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TGCATGATCATGAGGG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TGCGACGTCGGCACTG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TGCTCGTAGTGTTCCA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TTACCGCAGAAGCGCT.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TTCCGGTAGCTGACAG.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TTGAACGGTACCCGCA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401075__TTGCCTGTCTTCTGTA.1", "stage": "E11.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401073__AACAAGAGTAGCGTTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__AACACACGTTGCATGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__AAGTGAATCAAACTGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__ACAACCAGTCCCTGAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__ACCCTCAAGTGATGGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__AGACACTTCGTAACAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__AGTCACATCGTACCTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__ATACCTTCACATAACC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__ATCCCTGAGTAAGACT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__ATGGATCTCAACACCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__ATGGTTGGTATCGGTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CAACGATCAAGTTGGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CAAGCTAAGACCTCAT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CAGGGCTCAGGCACTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CATGCCTTCTGTCTCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CCGCAAGAGTAAACGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CCTCCTCGTCGAAACG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CGTAATGTCTCTAAGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CTAAGTGAGCGAGAAA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CTCAAGACAATGTCAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CTCACTGTCTGACCCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CTCCTCCCAAGATGGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__CTTCAATTCTGCGGAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GAGAAATTCCGCTAGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GATCATGAGCACTCAT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GCCATTCCACCCTAAA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GGAATCTTCATTCTTG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GGCGTCACATCATTGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GGGTATTCAGAGCGTA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GTCCTCAAGCCGATCC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GTTCTATCATGGCCAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GTTGTAGGTCTCTCCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__GTTTGGAAGCTTAAGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TATTGCTGTGAGATAT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TCAAGTGCATCATCTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TCATGTTCACAAAGTA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TCCGAAATCGCAGTGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TCTCAGCGTTAAGACA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGAGGAGCATCCTGTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGATCTTGTGTGAATA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGCAGGCAGGAACGCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGGATCACACCCTATC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGTAACGCATCTAACG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGTGGCGTCGGCTTCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGTTCTACAAAGGCAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TGTTGAGAGGAGGCAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TTCATTGCATCCTTCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TTCCTTCAGCAGAAAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__TTTCAGTAGGCGACAT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__AATAGAGCAACAGATA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__AATGGAAGTGCCTATA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__ACGGGTCCATTGTCGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__ACTGCAATCTATGTGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__AGAGAGCTCCCGATCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__AGGCTGCGTTGGAGAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__ATCCATTCAAATTGCC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__ATGGGAGGTAACACCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__ATGTCCCTCTTCCCGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CACCAAAGTCTTGCTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CAGCAGCAGTTCCTGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CAGTTAGAGCTAGCCC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CCAAGCGCATCGATAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CCACGAGAGTAGACCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CCGGACAAGACGGTCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CCTAAGAAGTATCCTG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CCTAAGACAGCGCGTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CCTCAACGTGCATTTG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CGACAGCAGCGCGTTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CGAGGAACAACGCCCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CGCAGGTGTGAGGAAA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CGCATAAGTCTATGAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CGGGCATCAATGACCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CTACTATCACTGAGTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CTGAGGCTCTTAAGGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__CTGCATCGTCGGTACC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GAAGAATTCAGCTAGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GAGGCAAGTCACTTAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GAGTGAGGTCGTATGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GCACGTGTCCGAGGCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GGGAGTAGTCTAACGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GTCGTTCGTCACTCTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GTGAGCCAGAATCCCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GTGCTGGAGGTTCTTG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GTTACAGAGTCTACCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GTTCATTGTGTTCAGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GTTCGCTAGTTGCCCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__GTTGTAGCACCATTCC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TATCCTAAGTCATACC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TATTTCGGTTGATGTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TCATTACTCCGAAATC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TCCTCCCGTCTCTCCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TGGAACTCAGCATCTA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TGGGAAGCAACACGAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TGGGTTAGTATGGAGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TGGTAGTCACAAGCTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TGTTGGACACCAGTAT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TTCTGTACAATGAGCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TTCTGTAGTCCGACGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TTCTTCCGTGTTGACT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TTGGGCGGTGGGCTTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TTTCACAAGAGAGTGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TTTCATGAGCGGTATG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401074__TTTGGTTCATTAGGAA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__AATGACCTCCGTCACT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ACCCTCAAGTTTAGGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ACCCTTGAGCACTTTG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ACGATGTGTACAAACA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ACGTCCTGTGTCCCTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ACGTTCCAGGAACTAT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__AGAACCTAGTGCGCTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__AGTCATGAGGATACAT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__AGTGACTCACATTCGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ATACTTCCACTACGGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ATCACAGGTGCCGTTG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ATCGTAGAGTCAGCGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ATGATCGGTGTTCGTA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ATGGATCCACGCTTAA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ATTCTTGGTAGTACGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__ATTTACCGTTATGGTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CAACAACGTCGCAACC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CACGGGTTCTTACTGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CAGCAGCCATTCTCCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CATGCAACAATTTCTC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CATGCCTCAGCACCCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CATGCCTCATGGAAGC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CATGGATGTAGCGTAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CATTGTTTCTAATTCC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CCTGCATTCTTTGGAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CCTGTTGCACCATAAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CGAATTGAGGTACCTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CGGAACCAGGTCGTGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CGGACACAGATACATG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CGTGATACATGGGATG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CTCAGAACATCCTCAC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__CTTTCGGCATGCCGCA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GAAATGATCTGGAGAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GAGACCCCACAGACGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GCACGGTTCACACCCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GCCGATGAGCGTATGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GGTCTGGTCCCTGTTG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GTAGAGGTCGGTAACT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GTAGGTTAGAATCGCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GTCATTTAGTACCGGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__GTGTGATAGGGTCTTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TAAGTCGGTTAGGCTT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TACTGCCAGCACTCCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TAGGGTTTCAGAGTGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TATCAGGGTATGAGCG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TCACATTAGTCCGCGT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TCACGGGGTTCGAAGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TCATATCTCGTGAGAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TCATCCGGTTATCCAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TCGAAGTAGAAGTCTA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TCGAAGTAGGGCAGAG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TCTACATTCATTCACT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TGCAGTACACTCTAGA.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TGGGAAGTCCCAGTGG.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TGGGCTGCACAACGCC.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401075__TTACCGCAGTGTACCT.1", "stage": "E11.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401073__AACCAACGTCTCAGAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__AAGATAGAGAGGCCAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__ACGTTCCAGCGCGTTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__AGAAGCGGTAACCCGC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__ATCCACCCAGCGATTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__ATCGCCTTCTAGTGTG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__ATTCCATAGCATTTGC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__CACGTGGCAGTCCCGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__CAGCAATTCAGGGATG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__CATGGTAGTTCGAGCC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__CCACAAACATCATGAC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__CCACAAATCGCTGACG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__CGAGAAGCAATTTCGG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GAAGCCCAGGAATGTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GATCGTAGTGACTGTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GCACGGTGTATCGATC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GCATTAGAGCCTTGAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GCCATGGCATCGATGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GCTGGGTGTGACCGTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GGGTGTCAGCTGGCCT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GGTAGAGTCTGGCTGG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GGTGTTAAGGTCGAGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GTAGGTTCAAAGGGCT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GTCCTCAAGAAACTGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GTGAGCCTCGCGGACT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GTGCTTCTCCATCTCG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__GTTCATTCAAGCACCC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TCAAGTGAGACCAAAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TCACAAGCAAAGGTTA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TCATACTAGGTGATCG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TCATATCAGAAAGTCT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TCCTCGAAGGGTGAGG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TCGGGTGGTGGTTTGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TGACTCCAGCACTAAA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__TGGTTAGCAGCTAACT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__AAGAACAAGATGCTAA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ACATGCATCACGGGAA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ACCATTTCAGGAAGTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ACGCACGCAAGACGAC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ACGGTCGGTTTACCTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ACGTTCCAGTATTCCG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__AGAACAACACCGCTAG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__AGCCAGCGTCGTTGGC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__AGCGATTCAATCTCTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__AGGAATACAAAGCAAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__AGTGACTGTTTCGCTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__AGTGCCGGTTTGGCTA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ATAGGCTGTGCATTTG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ATCCGTCTCGGTCGAC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ATGAGGGAGCACGTCC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__ATTCGTTCAATTGCAC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__CAGCGTGAGACCGCCT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__CCCGGAATCTCGTCGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__CCCTAACCAACGGGTA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__CGGGTCATCGAACTCA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__CTGATCCAGGGAGGGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__CTTTCGGCAAGCAGGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__GAAGCCCTCCAAGCAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__GAGAAATAGGAATGTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__GCGGATCAGTATGACA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__GCTTTCGGTTATCCAG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__GTAAGTCGTACCTAGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__GTGAGCCCATCCCGTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__GTGGGAATCTAGGAAA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__TAAGCACAGGATTTCC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__TCACGCTTCTGCGGGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__TCATGGACAATCCTTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__TGACCCTAGTTAGTGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__TTGCCTGCAAGCTGTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401074__TTTGGAGGTCGTCATA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AAAGGGCGTGACTCTA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AAAGGTACATGTGGTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AAGACTCGTAGACGTG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AAGATAGTCACATCAG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__ACGTTCCTCGCTGTCT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__ACTGTCCAGATAGCTA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AGCCACGGTCGTTGCG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AGGGAGTGTCACAGTT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AGTCAACAGATAGTCA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__AGTGATCGTAGCGCTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__ATCGTAGCACATATGC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__ATCGTCCTCTCGGTAA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__ATCTTCATCGGTAAGG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__ATGAGGGGTAGACGTG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__ATTTCTGTCAACCTCC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CAACCAACACAGTCGC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CAACGGCTCAACCGAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CACACAAGTGGGTATG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CAGAGCCGTGCTATTG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CAGATTGCAAGACAAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CAGCAGCTCTTCGTGC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CAGTTCCCATCGAAGG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CCTCAACCAAAGGAGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CCTCCAACAGGAGGAG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CGAGGAACAGTTCTAG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__CTGTGGGTCCAAACCA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GAAGCGACATGGAATA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GAAGGACTCGGTGCAC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GAAGTAATCATAAGGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GATCGTAAGTGCGCTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GATGGAGTCAGGAGAC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GATTCTTCAGCAGATG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GCAGTTACAAGACAAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GGTGGCTAGACAACTA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GGTGGCTCATAGATGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__GTGTGGCTCGTTGTAG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TAAGTCGCAGAGGCTA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TAATTCCGTCCAGGTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TATATCCCACGCACCA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TCACACCAGTATGGCG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TCATTCAGTTGCTGAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TCCCAGTTCAAGCCGC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TCGAAGTGTACGGGAT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TCGACGGCAAATACGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TCTCACGGTTGGGACA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TCTGGCTTCAAACGTC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TGCACGGAGTTTGCTG.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TGGGAGACACTTGAGT.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TGTGATGGTCTACTGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TTACGTTGTAGGCTGA.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401075__TTGTTTGCATAGATCC.1", "stage": "E11.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401073__ACGTACACATCGGAAG.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__AGAGCAGTCACCGGGT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__AGATCGTCATAACTCG.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__AGCGATTGTCGTTTCC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__CAGATTGAGTCGGCCT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__GCTGGGTTCTTAGTTC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__GTAAGTCAGAATCCCT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__GTTGAACGTACAGCGA.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__TCCACCAGTTCATCTT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401073__TTCTTGATCAAGCCTA.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__AACCTTTCATTACGGT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__AACGTCAAGATCCTAC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__AATCGACCAGACCAGA.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__ATACCGAGTACCCGAC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__ATCGTAGAGTATAACG.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__ATGCATGGTACGATTC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__GAACACTAGCACTCAT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__GAATCACTCCGCCTAT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__GATAGCTCAACGATTC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401074__TGTTACTAGACTCATC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__AATGAAGAGTGGCCTC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__AGTCACAAGAGGGTCT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__ATTGGGTCAAGCGGAT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__CCTCCTCAGAGCAGTC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__CGCCATTCATCCTAAG.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__CGTTAGAAGACGATAT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__CTGCATCTCTGTACAG.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__GGAAGTGTCACAGAGG.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__GGTAGAGAGAAACTAC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__GGTAGAGAGTGTTGAA.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__TACATTCAGACATAGT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__TCCAGAAAGTTGGCTT.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__TGATCAGCAGTCGCAC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401075__TGCTTCGTCCTGTTGC.1", "stage": "E11.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__AACCCAAGTTCCACGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AACGAAACAGGTGTGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AAGTCGTCATCCTTGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AATGGAACAGATCACT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__ACCTGAAGTCGTTGCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__ACGCACGCATCCGAGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__ACTGATGAGTCCTACA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__ACTTCGCGTCTGATAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AGACACTCATCTCGTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AGCGCTGGTGTTAGCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AGTACCATCTGGCTGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AGTTCCCGTGTGCCTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AGTTCGACACATTCTT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__ATCAGGTGTGCAACGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__ATCCCTGGTTGAAGTA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__ATCGTCCTCATTCATC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CATCCCATCCTACCAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CATCGGGCAGTTGTCA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CCGTTCAAGCTGACCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CCTCATGCAACACAGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CGCCATTCAAGATGTA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CTAACTTAGGCGAAGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CTACCTGCACAAATGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CTTAGGAAGTCCCGGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CTTAGGATCATACAGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__CTTCAATCATGGGTTT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GAATCACGTAACATCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GACCTTCAGCCAGTAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GCATCTCAGTGGCAGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GCATGATGTTGCCGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GCGGATCGTCGAGATG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GGAGCAATCCGCGAGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GGCGTCATCGCTTGAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GGGTCTGCACAGTCCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GGTAATCCACGACGCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTAAGTCTCCACCCTA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTAATCGAGAATCCCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTAGATCCACTGTTCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTCACGGTCCATTGGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTCAGCGGTGCTAGCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTCTGTCTCTGGGCGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTGATGTTCCGCACTT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTGCGTGAGAGTGTGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTGGCGTAGGGACACT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTGTTAGAGAGGTATT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__GTTAGTGCAGGGAGAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TAAGCCAAGGGCAGGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TACGCTCCACATAACC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TATCAGGGTACGATGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TCAGCAAAGTACAGCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TCAGGGCGTCCGACGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TCAGGTAGTCTCACGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TCCACGTGTTTGTTCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TCCTCGATCGCCATAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TGCTCCAAGTATAGGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TGCTGAATCCATAGGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TGCTTGCTCGAGCTGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TGTACAGAGTCGGCCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TTAGTCTGTGGGTCAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TTGAGTGGTTCGATTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__TTTAGTCCATCCTTGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AACCATGAGCTTACGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AACCTTTAGATCACCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AACGGGAAGTTTCGGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AAGGAATTCCCACAGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__ACCCTCATCTTCCTAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__ACGCACGAGCAACCAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AGACCCGTCAGACCGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AGGGTGAGTGATACCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AGTCAACGTTACCCAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__AGTTAGCTCTCTGAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__ATCCGTCGTTGGCCGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__ATCTCTAAGATTCGCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__ATGTCTTAGCATATGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__ATTCTACTCGGAGTGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CACGTTCAGATCCCGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CATGCAAAGTGTAGTA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CATGCAATCCCATGGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CGGACACAGCGTTCCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CTCCTTTGTCTTGAAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CTGCTCAGTAGCTCGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CTGTAGAAGAAGCTCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CTGTATTGTTCTGAGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__CTGTGAAAGGCTCTCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GAACTGTTCAAGGACG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GACCGTGTCGTCGCTT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GATGACTAGCACTAAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GATTCTTGTCGTCTCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GCATCGGGTTTGACAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GGCTGTGCAGAGAAAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GGTAATCCATTAGGAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GGTCACGTCACGGTCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GGTGTCGTCACGAGGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GTCGTAACACATCATG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GTGCAGCTCTTTGCTA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GTGGAGACAGGGAGAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GTGTCCTCAAGAGTTA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__GTTGCTCGTTATAGAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TACCGAATCTCGGGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TCACATTAGCACTAGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TGACGCGCAGGCGTTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TGAGCGCCATTGTAGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TGGTACACACCAGGTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TTCCGGTGTTGAGAGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TTGGGATTCTCTGAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TTGTTTGAGTGGTGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__AACCACACACTTCATT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__AACCTGATCTGTCAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__AATGGAATCATCAGTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__ACACCAAGTTAGGGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__ACAGAAAAGGAACTAT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__ACGGGTCTCTCTGCTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__ACGTTCCAGACTTCCA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__ATTATCCTCCTAGCGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__ATTCCCGAGTTCACTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CAATTTCCATCCTCAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CACTAAGTCGTGTGGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CAGCACGTCTCTCTAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CATACAGTCTTTCTAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CATGCTCCACGTCGGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CCCTCTCGTCAAGTTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CCCTGATTCAAAGACA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CCTCAACTCTCGAGTA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CCTTCAGCAATAAGGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CCTTGTGGTGAATAAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CGAGAAGAGTTTGGCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CGTTCTGGTCCGGACT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CTATCCGCAGCGACAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CTATCCGGTTAGGGTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CTCCGATTCGCCAGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CTGCCTACACAACGCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__CTTCTCTCAGCGCTTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GAGTTACCAGTTGTCA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GAGTTTGAGTGCCAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GATCGTATCTCATAGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GATGACTAGGTTGCCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GATGCTAGTGTACAGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GATTCTTTCAGCGGAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GCAGGCTTCAAAGACA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GCCCGAATCTCTGCTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GGAGGTAAGAGTGAAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GGCAGTCCAGCTGAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GGTGGCTCACAACGCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GTAGAGGGTACCCAGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GTCACGGAGAAGCTCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GTGTCCTGTACTCCCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GTGTGATTCCCTTTGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GTTCATTTCGAAGGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GTTCGCTGTCCCTGTT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__GTTGCGGAGTTAACGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TACACCCGTTGAGTCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TCGACCTTCGCTTACC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TCTCACGCATCGCTCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TGGCGTGCATCCTTCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TGTGGCGTCGAACGGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TTAATCCCACAAGGTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TTACAGGTCCGCACGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TTACGCCGTAATCAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TTCATTGGTCGGAACA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TTGATGGGTATGAGGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__TTTGATCTCGTGGCGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AAAGGGCAGGTCATTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AAAGTCCAGCATGTTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AACAACCCAAGTCCCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AACTTCTCATCAGTGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AATGCCAGTTGCGGCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ACAGAAATCGCACGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ACGATCAGTCTCGGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ACGATGTAGCCGTAAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ACGATGTTCATTCTTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ACGGAAGAGTAGGCCA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ACGTAACCAGAATTCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ACTATCTCATTGGGAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AGGGCTCGTAAGGAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ATCACTTGTAACATGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__ATCGTGATCTGGTTGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CAATACGGTCTTTATC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CAGATCATCGCCACTT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CATCGGGCACAGAGAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CGAGTTAAGTCTAGAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CGATCGGCAGTTCCAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CGGAATTTCATTCGGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CTCTGGTTCCGAAATC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CTGCGAGGTTCACGAT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CTGTGGGAGGTCGCCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__CTTCAATTCCTTCTTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GAAGGGTTCCGATAGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GAATAGACAAATCGGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GCAACCGAGCTTTGTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GCAACCGTCTTGGTCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GCGTGCAAGTTACGTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GCTTTCGAGTATCTGC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GGCTTGGAGGAAGAAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GGGCTACAGGTCACCC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GGGTGTCGTCCATACA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GGTAATCCATCGGAGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GTAATCGAGGATTCCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GTGAGCCTCAGACCCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GTGGCGTGTTAAACAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GTTATGGGTAAGTTGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__GTTCATTGTGCAGTGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TAACACGCAACACACT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TAGCACACACAAGTTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TAGTGCAAGTATTCCG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TATCCTACACATTGTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TCACGGGAGCAGGGAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TCACTATGTTAAGGAT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TCAGTCCGTTATGACC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TCCAGAATCACCCTGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TCCTTCTCAGTAACAA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TCGGGCACACTACCGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TCTAACTCAACCGTAT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TGCAGGCGTTACGTAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TGGAGGAAGTGATCGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TGGCGTGAGGTGCTGA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TGTCCTGCAACTGATC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TGTGATGAGTAGATCA.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TTAGGCATCTGCGTCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TTAGTCTCAGAAATTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TTCTAACGTTCGGCTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TTCTTCCGTGATATAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TTGAGTGAGCTACTGT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__TTTACGTTCTACCCAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AGCGTCGGTCCTATAG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TCACACCAGAGGGTGG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TGCTGAACATCGCTCT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TTCATGTAGTTACGTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401077__TTTCACAAGTGAGGTC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401078__AGCGTCGAGCACGGAT.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AACGGGAGTGCTCGTG.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401079__AGACACTCAGAGGTAC.1", "stage": "E13.5", "reference_type": "CVG"}, {"cell_id": "GSE178931__GSM5401076__AAACGAATCGACGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AACACACTCGCCGATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AAGACAAGTCAGCGTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AATGAAGCAAGCGAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AATGACCGTTGCTCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACAAGCTGTGAATTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACATCGAAGAGGGTAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACATCGACATGTTCGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACATGCACATAATCCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACCACAAGTCACTCTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACCCTTGAGCTAATGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACCTGAAAGAGGTCAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACGGGTCCACTCTCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACGGTCGCAAACGGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACGTCCTAGCCATTGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACTATCTCAAGGTCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACTATGGAGGAAACGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACTATGGCATCGAAGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACTGTGACAATTCTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ACTTAGGAGAATCCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGACCATAGGTCACCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGAGAGCAGCCTGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGAGCAGCATCACCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGGAAATAGCGCACAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGGACGACAGGTACGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGGCATTTCCATTCAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGGTAGGAGCCTGGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGTAACCGTTCACGAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGTCAACCACCATATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AGTGCCGCACGTTGGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ATATCCTAGAAACCAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ATCATTCGTGATTAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ATCCGTCAGGCTCACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ATCTTCACAAATGGAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ATGGGTTGTAGCTGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ATGTCCCGTAAGATCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__ATTGTTCTCCATCCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CAAGGGAGTATACGGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CACAGGCAGCACCAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CACGTTCGTAGATGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CACTAAGAGAGCATTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CAGAGCCCATCTCCCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CATAAGCAGAGCTGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CATCGGGCACTCCACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CATGAGTTCTCTAAGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CATGCTCTCGACGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CATTCATAGTCCCGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCCGAAGTCAGAGTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCCTCAAAGCGAACTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCCTCTCTCGAAGTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCGAACGTCGCATTAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCGGGTATCTGCTAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCGTGAGCAGAGAAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCTACGTAGCAGATAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCTCCAAAGTAAATGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCTCCAACAGAGGACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CCTTCAGAGTAAATGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGAAGGAAGCCAAGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGACAGCAGGCGAACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGAGGCTGTTGCGAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGCAGGTTCGGCATAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGCAGGTTCTCCGAAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGGAACCGTGCGTGCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGGAATTCATTGGGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGGAGAACAAAGGTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CGGGACTTCTTGGTCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTATCCGGTTTCGTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTCACTGTCGAGTACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTGAGGCCATAGAATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTGCATCCATAGATGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTGCTCACAGCGCGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTGGCAGTCCACTTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTGTATTAGGGATCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTGTGGGCAGTAACAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTTACCGGTACTCGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTTAGGATCGTGACTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTTCAATAGCCAACCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTTGAGAAGGTTGACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__CTTGATTCATAGAATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GAAGGACCACGGCACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GAAGGACTCTCTAAGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GAAGGGTCACGCTATA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GAAGTAATCTCGTCAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GACCCAGAGAGTTGCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GACTCAAGTATCAGGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GACTGATTCCCATTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GATGGAGGTCAGGCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GATGTTGAGCGTCTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GATTCGACACAACCGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GATTCTTGTTGGGAAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GATTTCTAGCTCGTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GCCCGAATCGCTTGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GCGTTTCAGGCTCTAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GCTCAAACAAGCAATA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GCTCAAATCTCTCTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GCTGGGTCAGCACGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GCTTTCGTCTCCGAGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GGAGAACGTACAGTAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GGGACTCGTTATCCAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GGGAGATGTACCCGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GGGCGTTTCACGGACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GGGTGTCAGGCTAGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTAAGTCAGGGTTAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTAGTACAGCCTCAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTATTTCCAACTACGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTCATTTAGGTCCGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTCTGTCAGTTAGTAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTGAGTTCACGTTCGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTGGCGTAGCATGCAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTGTCCTGTTGAATCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTTACAGCACTAACGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTTAGTGCAGGCCTGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__GTTGCGGTCCTAGCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TAACCAGTCTAGCCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TACTTACCAAACTAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TACTTACCAGGCATTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TAGACCACAGAGACTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TAGATCGCAGTAGTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TAGTGCAAGCTACTGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TAGTGCATCATTTGTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TATACCTTCTCATTGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TATCGCCCATTGTGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCAAGCACAAGAAATC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCACTATGTGAGAACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCAGTCCGTCGCACAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCATATCAGGTTACAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCCCATGGTGTCCGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCCTTCTGTGTAACGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCGATTTCAAGACGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TCGTGGGTCAGCTAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGAATGCAGGCACTAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGAGACTAGATGTAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGATCTTGTCCTGGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGATTTCAGTATAGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGGATGTCAGTAGTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTAACGCAGAGGGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTCCACCACAGAGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTCCTGTCTCATTGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTGATGAGTTTCAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTGGCGAGTAGACAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTTACTGTGCCTACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTTGAGCACCTCTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTTGAGCATTCCTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGTTGAGTCGTGGCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTACGCCCATACCACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTACTGTCAACTCATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTACTGTCAGTGGGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTACTGTGTATGTCCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTATTGCCAGGTGTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTCACGCCATCCGTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTCCTTCTCACTGTTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTCTAGTCAACCACAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTCTCTCCACCTGCAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTGAACGCACACTGGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTTATGCCAAGATCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TTTGGTTTCGTGGGTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AACAACCTCGTAACAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AACAAGATCGTCGGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AACCACATCTTACGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AACCATGCATGAGAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AACGGGACAGACACCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AAGACAACAGCCGTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AAGCATCCATTCAGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AAGCATCTCTCTTAAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AAGGAATGTCACTCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AATCACGAGACCATAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AATGGAAAGACTAAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACACTGAAGGATGGCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACAGAAAGTCTCACGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACATTTCCACTTGTCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACCAACAAGCTGAAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACCCAAACACATTACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACCCTCAAGACAGCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACCTACCGTTCATCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACGCACGAGAACGCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACGTAACTCATCTACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACTCCCACAAATGAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ACTGCAAAGTGTGTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGACACTCAGACCTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGACCATGTGCGTCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGAGCCCCAGATACCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGATAGAGTGTATTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGCGCCAGTACCGTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGCGTATTCGATCCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGGTAGGTCGACGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGGTGTTCACCAGGTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGTGACTGTCAGTCCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGTGATCGTGCGTTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGTGTTGGTTGGAGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__AGTTCCCCATGGTACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATACTTCTCTGTACAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATCACGAAGCATGCAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATCGTCCCATTGCCGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATGGAGGCACCCTAGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATGGGAGCAACCGTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATGGTTGAGACATACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATTACCTCAACAGCCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATTACTCGTCTCGGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATTGTTCTCGTCGCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATTGTTCTCGTTGTAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATTGTTCTCTGCGGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__ATTTCACGTTAAGTCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CAACGGCTCACCGACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CAAGACTTCCAGTGCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CAAGGGACATATGCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CACCGTTTCCTGGGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CAGCAATCAAATCGGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CATACCCGTACCTATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CATCAAGGTAGTCGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CATTCATAGTGAGTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CCAATTTCATATAGCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CCACAAATCTGGGCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CCGTAGGTCCGTGGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CCTATCGCAGTAGTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CCTCAGTGTCGCACAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CGACAGCTCTGACGCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CGCAGGTGTCGCGTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CGCGTGAGTCGGCTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CGTGCTTAGTTGCCCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTACCTGCACAGCTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTACCTGGTGCCCACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTCAACCCAAGAGCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTCAAGATCGACGCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTCCAACGTGGACAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTGAGGCCACCGTGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTGCCATTCCGGTAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTGCGAGTCGTCACCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTGCTCAGTGACAGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__CTTCTCTGTATCGCTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GAATAGAGTGGGCTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GACTCAAGTGACAGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GAGAAATAGACGCTCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GAGAAATCAACAAGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GAGTGTTAGGACATCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GATAGCTTCCTTCGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GATCAGTCATGGCTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GATGACTAGTAAGGGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GATGTTGGTTCTCCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GCAGCTGGTGGACCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GCCAGTGTCGATAACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GCCCAGAAGTGCAGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GCGAGAAAGTCACGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GCTTCACCAAATAAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGAACCCTCTCATGCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGAATGGAGGTTCCGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGAGATGGTAGCTGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGAGGATTCTTGTGCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGCACGTAGAGATTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGCAGTCTCATGGGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGGATGATCTATACTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGGTTATTCCAAGCTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGTGGCTTCTGTACAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GGTTGTACAAACAGGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTCACGGTCATCTGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTCAGCGTCCACTGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTCCCATCAAGCGCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTCGCGACACGTACAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTCTCACAGTGCCGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTGCGTGGTCGTGTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTGGCGTTCAGAGTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTTACAGGTCACCGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__GTTACGACACACACGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TAATCTCCACGTCGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TACACCCTCGTTTACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TACCTGCCATCCGAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TACGGGCCAATGCTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TACGTCCGTGGGATTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TATCGCCGTCTGTCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCACATTAGCGATGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCACGGGCACTCCACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCATACTCAGCTATTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCATTACGTCCAGCAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCCACCACAGCGCGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCCTAATGTTAAAGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCCTTCTAGTGAACAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCGACCTCACACTTAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCGATTTGTTGTTGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCGCTCAGTCGGAAAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCTATACCACTAGTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCTCCGACACATATCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCTCCGATCAGGCGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCTGGCTCATATGAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCTGTCGGTGTCGCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TGAGCATTCCCTTTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TGCTTGCCAGGTCAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TGGAACTGTGTATTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TGGGAAGTCCCAATAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TGGTACATCATAGACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TGGTTAGTCCTAAACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TGTTCTACACTCCCTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTACAGGAGCACTAGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTACAGGGTCTGATAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTACCATAGCGAGTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTACCATCAGTGGCTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTACCATGTAGACTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTCCGTGTCCAAAGGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTCTAGTCACAGTACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTGACCCGTGCGTTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTGATGGCACAAAGCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTGCGTCCACTGTCGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTGCGTCTCGCGTTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTGGGTACAACAGCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTTACGTAGACCAGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTTACGTAGGTGGCTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTTACTGAGCAGTACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTTCCTCAGAATACAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTTCGATAGAATCGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AAACCCAGTTGTTGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AACCTGATCGAGATAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AACCTTTAGTCTGCGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AAGCATCCATCCGATA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AAGTCGTGTTGTGCAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AATGAAGAGGTAGCCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ACGGTTATCAGTGCGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ACTATGGAGTAAATGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ACTTTCAAGCTCAGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AGAACAACAGATTAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AGACCATAGTACCGGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AGCGTATCAGGTCCCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AGGCATTAGCAATTCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__AGGTGTTCATAACAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATACCGACAGAACATA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATCCGTCTCGTAACCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATCGGATCATCCGTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATCGGCGGTCTCAAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATGAAAGAGCTCCACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATGAAAGGTTCCGCGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATTCAGGAGCTCCACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__ATTCTTGTCATGGATC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CAAGCTACATGGCACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CAAGCTATCAGTGTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CAAGCTATCGGCTGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CAATTTCAGGTATTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CACAACAGTGAGATTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CACCAAAAGGAACGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CACCGTTTCCTTGACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CACTGAAGTGCCTACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CAGCGTGTCATTTACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CAGGCCAAGCTGAAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CAGGCCAAGGGATCAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CATGGTAAGGATGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CATTCCGCAAAGCTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CCCGAAGAGCGAACTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CCCTGATAGAACCCGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CCCTTAGTCGTAATGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CCTCAACCAAAGAACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CCTCCTCCAACCGTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CCTTCAGAGACGGTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CCTTGTGCACCTGAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CGAGGAATCTTCTGGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CGTAAGTTCGTTGTTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTCATGCTCACCCTTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTCCTCCGTGGATTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTCGAGGGTTTGGGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTGAATGAGTAGCATA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTGATCCAGATGGTAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTGCTCAAGCAACAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTTCCTTAGACATCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CTTCTAATCTTCGTAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GAACGTTTCGACCACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GAATCACTCCCTTTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GACCTTCTCCTGTAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GACTATGCACAGTATC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GACTATGTCTTGAACG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GACTGATCAAGACCGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GAGCTGCAGTGGAAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GAGGGATGTCTAACGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GATCATGAGATAGCTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GATTGGTTCGCATAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GCACATATCCTCTGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GCACGGTGTCGCGGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GCAGTTAAGCACCCAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GCAGTTAGTAAGTAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GCATCTCGTAGACTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GCTGAATGTTTGTTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GGAACCCGTATCGAAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GGACGTCGTACCTATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GGAGCAAAGACATATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GGCTTGGTCAAGTCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GGGATCCTCCGATAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GGGTATTGTTGCTTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GGTGGCTTCCTAGAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GTAGAAAGTGTCTTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GTAGGTTAGAGAAGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GTCAAACCAATCGCAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GTCGTTCTCTCCTGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GTGACGCCATCGCTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GTGGTTATCTCTTGCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__GTGTTAGGTGCAAGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TAACTTCGTTTCCAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TACAACGTCATTCGGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TACACCCGTGCACATT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TACATTCCATGACTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TACTGCCAGTATGGAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TAGAGTCTCCCATGGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TATACCTGTGCAACGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCAAGACTCAGCGTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCACAAGAGCGCAATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCAGGGCGTGGAGAAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCATCCGGTCGCCACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCATTACGTAATACCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCCATCGCACTGAGGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCCGTGTGTAAGGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCGCAGGGTGAGCGAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCGGGTGGTGCGCTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCGGGTGTCGTAGGGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCGTGCTTCCGTTTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCTTCCTGTGTTCCTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGAGGGAGTCACAGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGAGGTTCATGGGTTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGATGCATCGACGACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCAGATAGCTAAACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCATGAAGTGTTCCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCGGCACACCGTCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCGGGTGTCGATTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCGGGTTCATCACAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCTGAAGTAGGCAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCTGAATCTCGGCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCTTCGGTATAGCTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGCTTGCTCTGGTTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGGGAAGCACAGACGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGGGATTTCGCGATCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGGTAGTAGTATGGAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGTCCCAAGGTTCAGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TGTTCCGAGGCAGGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTAATCCCATATAGCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTAGGCATCATAGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTATTGCTCATGGCCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTCCTCTTCTATGCCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTGACCCTCGCCTTTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTGGGATAGCCGTCGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTGGGCGGTAGTAAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTTACTGCAGGCTTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTTCCTCTCGGTAGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TTTGACTGTGCAGATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAACGCTTCGTGACTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAAGTGACACTGAGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAATGGAAGAAGAGCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAATGGAAGCATTTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AACAACCTCGACCTAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAGACTCCAAAGGAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAGCGAGAGGAACGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAGTACCTCAGTGGGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AAGTGAACATGATGCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACACCAAGTGAGTCAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACAGGGAGTGACTCGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACATTTCAGAAAGTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACCAACAAGTCCTACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACGATCACAGGGTTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACGGTCGCAACACGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACTATTCCAAAGGCAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACTGATGTCGAACTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ACTTAGGGTAGGGTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AGGACTTGTCCTGTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AGGGCTCCATACCATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AGGTTACTCGCGTAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__AGTTCGACATTGACTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATACCGAAGGCCTTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATCACGAAGAAACACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATCCACCGTCGAGATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATCCATTAGGCGTCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATCCTATTCTTAGCAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATCGTGAAGTAGCTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATGGGTTCAGCGGTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATTATCCGTTTCCCAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATTGGGTCACCCTTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATTGGGTCACTTTATC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATTGTTCGTGTTCGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATTGTTCGTTGGGAAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__ATTTCTGTCCACGTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CAGCAGCTCACAAGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CAGGTATCAGCTGAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CATACAGCACAAATCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CATCCACAGAAACTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CATCCACAGTCTTGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CATCGTCTCCTGTACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CATGCAAGTTAAGGAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CATGGTAGTTCACCGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CATTCTACACCTCTAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CCAAGCGAGCTTCTAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CCACCATAGGCCCGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CCCATTGCAACGTATC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CCGTGAGTCGGCATCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CCGTTCAGTGCCGAAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGAGGAAGTGAGTAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGATCGGAGTGGTTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGATGCGGTCACTACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGCCAGACATGACTTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGGAACCGTACCTGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGGGACTGTATCCCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGTAAGTAGAAGCCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGTAATGTCGAAATCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CGTGTCTCAGTGCCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTCTGGTGTGACTGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTGAATGCATGCTGCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTGCTCAGTACCGGCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTTACCGAGACATATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTTCCGATCTGGCTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTTCCTTCAGCTATAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTTCGGTCACTAACCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTTCTCTTCCTCTAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CTTGAGAGTAGGCAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAAGGACAGCTGGTGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GACCCTTAGAAGGGAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GACTTCCCACTCATAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAGACCCAGGAGCTGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAGAGGTCACAGAAGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAGAGGTCAGGTTCGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAGAGGTTCACTCGAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAGTGAGGTATACGGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAGTGTTAGCATTTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAGTTGTGTGCCGTTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GCAACCGTCGTCGATA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GCATCTCGTACAAACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GCGTTTCTCAGTGTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GGAGGATCACCGTGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GGAGGATTCGCGTGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GGGATCCCACGCTTAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GGTCACGTCGATACAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GGTGTCGAGGTATCTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTAACACGTTCGAACT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTAATCGAGACGAGCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTAGAAAGTATCCCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTAGAAAGTCATGGCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTAGAAATCTGACAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTAGCTAAGCGAGTCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTCATGATCAAAGGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTCATTTAGGAGTCTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTCTCACTCGTTCATT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTGAGCCGTATCACCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTGTGATCACACCTGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GTTTACTGTAATCAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TAACGACTCGCCCAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TAAGCCAGTTATCTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TACCGGGGTCGTCGGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TACCTCGAGAGTGTGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TACTTACCACGCAAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TATCTTGCAATGAAAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TATCTTGGTACACGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCACAAGAGGTCGACA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCAGGTAGTGATGAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCAGTCCAGACCAGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCAGTGACAGAGAGGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCATTACAGCACTCCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCATTTGGTCAGATTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCCACGTCAAGGCGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCCGAAAGTGGTATGG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCCGTGTTCCTTTAGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCCTTTCTCGAGGCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCGATTTGTCTACGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCTATACGTGTCCCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCTGTCGTCCCTCAAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TCTTAGTTCGCGTTTC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGATGGTAGTAGGCCA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGATTTCGTACAGTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGCATCCAGCATGCGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGCATCCAGGTTGGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGCTCGTCATATCTCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGCTGAACAACCAATC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGGGCGTAGGCACCAA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGGGCGTGTATCACGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGGGCTGGTGGTGATG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGGGTTACAGCTGTTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGTAACGGTAGGAAAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGTAAGCCATAATGAG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGTAAGCTCCACGGAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TGTTACTTCTCACTCG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTAGGCATCCTGTACC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTCACGCTCAAGAGTA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTCCGGTGTGACTGTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTCCTAAAGCTGAGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTCCTTCGTATCAAGA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTGTTGTTCGGCTCTT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTTACTGCACAGCGCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__TTTCAGTGTATCACGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGAGGTTAGCAAATGT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGCCGAGAGCGTTCAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGCGGGTTCTTAATCC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__TGGGTTAAGGATACGC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TATGTTCTCTTCGATT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TCACTATAGCCAAGTG.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401077__TTGCGTCGTCTGCATA.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__CGTTGGGCAGCGTATT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401078__TCCATCGTCAGGAAAT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__CAGCAGCTCAAAGAAC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GAAGCGATCTACTATC.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401079__GCATCGGGTAGATCCT.1", "stage": "E13.5", "reference_type": "SGN"}, {"cell_id": "GSE178931__GSM5401076__AAACCCACAAGCGAAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AACCATGGTATGGAGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AAGAACATCAGCGTCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ACAAGCTTCGTGCGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ACAGCCGCAAGTGACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ACGATCAGTTGCCATA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ACGGAAGAGGTTACCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ACTATCTCAAGATCCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ACTCCCATCCTATTTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGAAGTAAGCGTCTCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGACCCGGTGCATTAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGACTCATCCGATGTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGCGCTGGTTCCCACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGGCATTTCTCTCTTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGGTGTTAGCGTGTCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGTAGCTAGTCTCTGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__AGTGCCGGTCAAGCCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATCAGGTGTTGTTGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATGGTTGCATGAATAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATTACCTAGATAGTGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATTCTACCAAACTAAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATTGGGTCAAGCGATG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATTGGGTCAGTCAGCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATTGGGTTCGCATTGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__ATTGTTCAGTGCCAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CAACAACAGTCCCTAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CACGAATGTATCCCAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CACTGAATCCACGTAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CAGCCAGGTGGGTCAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CAGGGCTCAATTTCGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CAGTTAGCATGCCGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CATGGATGTACAAGTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CCGATCTGTGTGTGTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CGAGGAATCACCCTTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CGTCCATGTACGAGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTAACTTAGTAAGACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTACTATAGTTCATCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTAGGTATCAAGAGTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTATAGGGTAGCTAAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTCCGATCACAAGCCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTCCTCCGTGGTCTTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTGCGAGTCACTGAAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CTGGCAGCACTATCGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GACACGCTCAACCTTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GACCCAGCAAGAAACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GACCCTTTCGCTGTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GAGGGTACATCATTTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GAGTTACAGACGAGCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GATTCGATCGAGAATA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GCAGGCTGTGAGTGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGCACGTAGGGTTAAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGCAGTCTCTAGCCTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGGATGACACTGGCGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGGATGATCACTGTTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGGCGTTCACCGTCGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGGTATTGTCGAATGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGGTCACCAAGCACAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGGTGAAGTTACGATC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGGTGTCGTTCGAACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GGTGGCTCAACGGGTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GTACAGTCAATTCACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GTTCATTTCCTTCGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__GTTTACTAGTTTGCTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TAAGCGTAGCCTGTCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TAAGTCGTCTCTAGGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TACGGGCTCATCTACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TAGACCACAATACCTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TATATCCAGGGCTAAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TCAAGCAAGCTCCGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TCACAAGCATCACCAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TCCCATGGTCCGCAGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TCGCAGGGTATCAAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TCTATACCAAGATTGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TGTTCCGTCAATCTTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TTATTGCAGTCACGAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TTCATTGAGGTTGACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TTGGTTTCATATACCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TTTATGCCACTAGAGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__TTTCCTCGTTAGGAGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AAACCCACAAACAGGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AAGCCATTCGCTCATC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AATGCCACAAGCACAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__ACTCTCGCACCGCTAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AGAAGCGAGCATCAGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AGATCCAAGGTTGGTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AGATGCTGTACCTGTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AGGATCTGTCCGATCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AGGTCTAAGGGACCAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__AGTAGTCCAAATAGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__ATCGGCGAGCGACAGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__ATCTTCACATTGTACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__ATGGTTGCATCGATCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__ATTATCCCATAGTCAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__ATTCCCGCACAAGCCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CAACAACAGAAGCTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CAACAGTGTCGGAACA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CACGTTCTCCGCATAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CACTGAATCGGAGCAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CAGTTCCTCAACGCTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CATACCCGTGCAACAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CATTCATAGTGGTCAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CATTCCGGTGGACCAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CATTTCAGTAACACGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CCAATTTTCTGCACCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CCGGTAGCAATTCTTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CCTCAGTTCTGCCCTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CCTCATGGTATGAAGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CCTCATGTCACGATCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CCTCCAAGTACGTACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CCTCCTCCAGGCATTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CGAATTGCACTGCGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CGTAGTATCCACTGAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CGTGCTTTCTCCCTAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CGTTAGACATCAGCGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CGTTAGAGTTAAGCAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTACCCAGTAATGTGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTCAAGACAAGTGCTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTCACTGCAGGAGACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTCATCGAGAAGCGGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTCGAGGGTCGGTAAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTCTCAGTCAGCAGAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTGAGCGTCAGAACCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__CTGCCTACAGCTGGTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GAAACCTTCTTCGCTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GACACGCTCAAATGAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GACCAATCAAAGCTAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GACTCAAAGTGCTCGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GAGATGGGTCACGTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GAGTCTAAGTAGACCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GATAGCTCAAATTGGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GCACGTGTCCTACTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GCAGTTAAGTTGCCTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GCCAGTGAGCCTTGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GCCGATGCAACGACAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GGAATCTAGGCTATCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GGGATGACAACCGTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GGGTATTTCAGACCTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GGGTGTCAGTAACGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GTCACGGAGTGGTTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GTCGTAATCCACTGGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GTGCAGCTCCGTTTCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GTGGAGACAGCGACAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GTGTTCCGTTCGAGCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__GTTACCCGTTCAAGGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TAAGTCGCATAGGTTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TACCTGCGTCTTCCGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TACTGCCCAACCCGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TACTTGTGTTTACGTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TAGACCACACGCAAAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TATTTCGCACGTCGTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCATCATAGATGCCGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCATGTTGTATTTCTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCATTACGTTAGGACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCATTCAGTTAGCTAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCATTCATCATGCCGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCATTGTTCTTAGCAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCCACGTGTACTCGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCCTCGACAGGCGATA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCGGGACCACTGGACC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TCTCACGTCACCTGGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGAGGTTGTATGACAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGATCAGAGCGACTGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGCATGACAAACGGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGCCGAGAGACTTCGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGGGCGTCACCGCTGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGGGCTGTCTTCTTCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGGTAGTTCACGATCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TGTAAGCAGTGGTTAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TTACCGCGTGAGTAAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TTACGTTGTCTACGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TTGGGATGTGAACCGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TTTACCAAGTGGTTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TTTACTGTCCTCACCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401077__TTTGACTTCTTAAGGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AAACGCTTCGGTCTGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AACGGGACAGACCATT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AATCGTGCAACCCTAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ACAACCATCTGCGTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ACACCAAAGCCGGATA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ACTATCTAGAGTGTTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ACTATCTAGGGCGAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ACTCTCGTCGAACGGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AGACTCACAAGTGGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AGGGCTCAGTGCCTCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AGGGTCCTCATGGGAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AGGTAGGTCGCCGAGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AGGTCTAGTAGTAAGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AGTGATCTCCGTAATG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__AGTTCGAGTACCGTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ATAGAGATCTTCCCAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ATCACGACACAATGTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ATCCCTGTCGAGCACC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ATCTCTAGTGCAACGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ATTCATCGTGACACGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__ATTCCTAAGCATCTTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CAGCACGAGCAGCAGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CAGTTAGCAAAGAACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CATGAGTCAAGATCCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CATTCTACACACCGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CCACGAGAGTTATGGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CCCGAAGTCACCGGTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CCCTTAGAGTCGAAGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CCGTTCACATAGAAAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CCTAAGAGTAGTATAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CCTACGTGTCTGCCTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CCTGTTGGTGGATACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CGAGGCTAGCCATCCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CGAGTGCTCGTCGGGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CGTAGTAGTCGAAGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTAACTTTCTGAGAAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTACATTCATAGAATG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTACATTTCAAGCTGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTACCCATCACCACAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTACTATGTTGGGACA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTAGACAGTGCCGGTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTCAGGGAGAAGGCTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTCCATGCACTGGAAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTCCCTCCATTGTAGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTGATCCTCGGACTTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTGTACCGTCACCCTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTTCAATAGGAACGAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTTCAATTCAAATGCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__CTTTCAATCCGACGGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GAAGGACCATAGGAGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GACTCTCAGCACACAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GATCAGTCAATTGTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GATGGAGGTACGATTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GATGTTGCAACCGCCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GCCCGAATCCTTTGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GCTTTCGAGAAGCTCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGAGAACCACGACAAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGCTTGGAGCCGTTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGGAAGTTCCAAGCCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGGACAAAGCGTGCCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGGAGATTCATGAGAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGGTGAAGTAATGCGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGTCTGGGTACTAAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GGTTGTAAGCCGCTTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTAACCAGTTAAGGGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTCCACTGTATACCTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTCTACCAGACCAACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTCTTTATCCTTCGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTGCACGAGTCGAGGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTGCTTCAGTCGAATA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTGGAAGCAGGCTCTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTTACGAGTTGCCGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TAAGCGTAGTAGTCTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TACCCGTTCTATCGTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TACTTGTGTGGACCAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TAGAGTCCATCGCTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TATATCCCAGCACAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCAAGTGAGGACAACC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCACACCAGCACTCCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCAGGGCGTTCGTAAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCATTACGTTCAAAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCCGGGACACTGTGTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCGTGCTGTCCAAGAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCTCAGCAGTAAGGGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCTGCCACACATGACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TCTGTCGGTCTCAGGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TGAGACTAGATAGCTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TGAGACTAGGTAAGAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TGCAGTAAGTAGGCCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TGCCGAGCAGCATTGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TGCGATAAGAAGTGTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TGTTACTAGAGTGTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TGTTGGATCTAGTACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TTCTTGAAGTTGAATG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TTGGGATGTCGAATGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__TTTGACTGTCTTACTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AACAACCCAAGCCATT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AAGACAATCATGAAAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AATCGACTCCCAACTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AATTTCCAGCATGCGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ACACCAAAGTGGATTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ACATTTCTCCAGCCTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ACCGTTCTCTTCCCAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ACGCACGGTACCCACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ACTTAGGAGTGGTGGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AGAAATGCATAGGAGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AGGATCTTCTGTAAGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AGGTTACGTGTATTCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AGTAGTCCAGCTTCGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__AGTTAGCAGTATCTGC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ATATCCTCAGACATCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ATCACGACATCGGCCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ATCTCTAAGTACTGGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ATGATCGTCGGACGTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ATTACCTGTGGCAGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ATTCGTTCACCTGTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__ATTGTTCGTGGGTTGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CAACCTCAGAACGCGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CATCGGGCAACGATCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CATTCATGTCGCTTAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCACAAAAGTTCTACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCACGAGGTCGTTGCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCCAACTGTTGCTCGG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCCTTAGGTCTCAGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCGAACGCACTCTAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCGGGTAGTTTCGGCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCTATCGTCATGGGAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCTCAACGTCCATACA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CCTGTTGTCTCAGTCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CGAAGGAAGAGTTGAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CGACAGCGTACTGTTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CGTAAGTAGGCCGCTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CGTAATGCATGTTACG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CGTTAGATCCATTTAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTACCCAAGTACTGTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTCAATTAGGTATCTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTCAGAATCTCTTAAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTCATGCTCTTCCCAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTGAGCGTCTCTCGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTGCATCGTCACCGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTTCCGATCACTCACC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTTGATTCATGTTCAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__CTTTCAAAGACGGAAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GAAGGGTAGTCAGGGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GAGGGTAGTATGTCTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GAGGGTATCATGCTAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GAGTCATAGGGACACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GATCCCTCAGCGACCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GATTCGATCCTAGAGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GCCAGTGTCCGTGTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GCCCGAACACCTATCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GCGTGCATCACTACTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GCTTTCGTCCGGCTTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GGACGTCGTAAGGAGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GGCGTCATCGGTCATA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GGGTAGAAGTGCAGGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GTAATGCTCACGGGAA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GTAGTACCAGTAGTTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GTCGAATAGCATATGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GTGAGGAAGCTGTCCG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GTTGCGGTCGCATTAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GTTGTGATCGGTAACT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TAACGACTCACTGTCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TCACACCCATTAGGCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TCATGCCCAGTTTCAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TCCTGCACATCTTTCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TCGCACTGTCCTGGTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TCGGGCAAGCTGTGCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TCGGTCTTCCACCCTA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TCTGGCTAGACCAGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGAATCGGTACGCGTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGACAGTGTGACTATC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGCAGTACAAATCGTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGCTGAAGTCCACTCT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGGAACTCAGAGCTAG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGGGAGATCTATACTC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGTCAGACAGATTCGT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGTCCACCAGCCGTTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TGTTCTATCGACGCTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TTGCATTCACATTCGA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TTGTGGACACGGCCAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TTTACCAGTACGACTT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TTTCCTCCATCCGGCA.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__TTTGATCCAGAGACTG.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401078__GTCACGGAGGAAAGAC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GGAGGATGTGCCTAAT.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401079__GTAGAAACAATGGCCC.1", "stage": "E13.5", "reference_type": "VGN"}, {"cell_id": "GSE178931__GSM5401076__CCACAAACAACCTATG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401076__CGTAGTAAGTGAACAT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401076__TAACCAGTCCTCGCAT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401076__TAAGTCGGTTCTTGCC.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401076__TCAGTTTAGGATTTGA.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401076__TTCACGCCAATCGCCG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401077__AAACCCAAGCTTTCTT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401077__AAGAACAGTTTCGTAG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401077__CCTACGTCATCCAACA.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401077__CTCATCGAGCGGCTCT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401077__GAGCTGCTCAAGTCGT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401077__GTCTTTAAGCGTCAAG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401077__TCATGGAAGGTAGACC.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__AGGATCTTCTCGACCT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__CATCAAGCAATGCTCA.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__CATTCCGGTTCCGCAG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__CTACCTGAGCTCTATG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__GCAGCCACACAAATCC.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__GTAGGAGCAACGATCT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__GTGGGAAGTTTACTTC.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401078__TGCCGAGAGCCAAGTG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__CCCATTGTCGGTTAGT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__CGAGGAAAGCGTATGG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__CTCATCGCACATTCTT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__GACCGTGTCTTGAGCA.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__GCAGTTATCCTTACCG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__GGTGGCTAGCCGATAG.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__GTAGAAACACCAACAT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__TCCTGCACAACGCATT.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401079__TGAGGGATCTACCACC.1", "stage": "E13.5", "reference_type": "GG"}, {"cell_id": "GSE178931__GSM5401076__AAAGGATGTTTAGACC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__AAGATAGGTGGCCACT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__AAGTGAAGTACAATAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__AGTCTCCAGAGTGGCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__ATCATTCTCGAGGCAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__ATGGGAGTCGGCCCAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__ATTATCCAGAATAACC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CAAGACTCACGGTGAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CATGGTACAGGCAATG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CCACAAAGTCAGGTAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CCCATTGGTCAGTTTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CCGGTAGAGAGTACCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CCGGTAGAGGCCACTC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CCTGTTGGTTACACTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CGCAGGTCAGCTATAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CTATCCGCAGGAGGAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__CTCATTAGTACGGCAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GACGTTATCTGCAGCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GAGCCTGTCTTGATTC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GAGTCATTCACCGGTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GAGTTGTCATCTAGAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GATCGTAAGCCACTCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GCAGCTGTCCCATACC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GCATCTCCACGTATAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GGTAATCTCGCCAGTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GTTACGAAGCGAAACC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__GTTGTCCAGTCAATCC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TAACGACTCTCGCCTA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TAACTTCCAGTGTATC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TAAGTCGGTGAGTAAT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TCCTCTTAGCATCAAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TCCTTTCCACTCGATA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TGAATCGAGCATTTCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TGACGCGTCAGAGCAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401076__TTAATCCTCTCTTCAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__AAACGAAGTCACATTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__ACCCTCACAGTAACCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__AGAGAGCGTATGCTTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__AGGTAGGAGCATGTTC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__AGTACCATCGAGCCAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__AGTAGTCTCCGTAGGC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__ATCACGATCGTGGCTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__ATCATTCCATCGGAGA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__ATGAAAGGTCAAGGCA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__ATTCATCGTCAGTCCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__ATTCTACTCAGACTGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__CCATCACCACACCTAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__CCGGACAAGCCATTGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__CCTTGTGTCCTATTTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__CGAATTGTCAACACGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__CTCCCAAAGAGCGACT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__GAATCGTAGTCCGCGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__GAGTGAGGTCCACTCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__GCATTAGTCACCCTGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__GCCATTCTCACACCGG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__TCGCTCACAGAGAGGG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__TCTATACAGCTTCTAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__TGAGCGCAGTGCGTCC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__TGGCGTGTCGGAGTAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__TGTCCTGTCAAACCCA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__TGTGATGAGTAACGTA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401077__TGTGCGGCAGCCTACG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__AACGTCAGTAATTAGG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__AAGAACATCGCTGTCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__AGACAGGAGTTGCCCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__AGATAGAGTCACCCTT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__ATCCATTGTCACTTCC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__ATGCATGAGTATCTGC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__ATTCTTGCAGCCCACA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__ATTTCACAGAACTCCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CAATACGTCATGCCCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CACGTTCGTACAAGCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CAGGTATCATACGCAT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CCTAAGAGTCGGAAAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CCTCCAAGTGTAGCAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CGTTGGGGTCTTTCTA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CTCTCGAGTACCGCGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CTGGACGAGTTCCAGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__CTTCTAAAGATTGTGA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__GCCAGCAAGGTAATCA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__GGAGAACTCTCAACCC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__GTAGTACCATGGGTTT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__TAATTCCGTGTATTCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__TCATGTTCACGCCACA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__TCTTAGTAGTTAGAAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__TGGTGATTCGTTAGTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__TTGCTGCAGCACTCTA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__TTGGATGGTGTATACC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401078__TTTGACTCACATCATG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__AAACGAAAGTCTCGTA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__AAAGTCCCAGGCGATA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__AACCTGAAGGATATGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__ACCAAACTCGCTACGG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__ACCCAAATCGAAGAAT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__ACGTAGTGTCTCACAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__AGACCCGGTAACATCC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__AGCCACGAGTATGGCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__AGGCATTTCCTCAGAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__ATCATTCAGTCCTGCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CAACAGTTCTATCCAT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CATAAGCGTCTGTCCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CATGGATGTCATCGCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CCCGAAGGTCCAGCAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CCGTAGGTCGCATTAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CCTAACCAGTAGCCAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CGCATGGTCAGAGTGG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CTCAGAAGTACAAGCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__CTGCGAGCACCTAAAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__GGAAGTGAGTGCCTCG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__GGGAAGTCAGTAGAAT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__GGGAGATGTCGTCTCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__GTACAACGTGCAAGAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__GTGTGGCGTCAGCTTA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__GTTAGTGGTGCTCTCT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__GTTCTATAGGTTAAAC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TACGGGCCAAGAGCTG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TCGAAGTCATAACCCA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TCGACGGCACTGGAAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TCGGGCAAGTCTGCGC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TGACTCCAGTTCAACC.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TGCGGCAGTTTACCAG.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TGTCCACTCATCCCGT.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TTGATGGTCTAGCCAA.1", "stage": "E13.5", "reference_type": "mixed"}, {"cell_id": "GSE178931__GSM5401079__TTTAGTCCAGAAACCG.1", "stage": "E13.5", "reference_type": "mixed"}]')

def read_cases():
    return [FROZEN_E9_MEMBERSHIP]

def reference_labels():
    result = pd.DataFrame(FROZEN_REFERENCE_ROWS).set_index('cell_id')
    assert result.index.is_unique and len(result) == 1978
    assert result.groupby('stage').size().to_dict() == {'E11.5':645, 'E13.5':1333}
    return result

def prepare(a):
    assert a.obs_names.is_unique and a.var_names.is_unique
    assert np.all(a.X.data >= 0) and np.allclose(a.X.data, np.round(a.X.data))
    assert a.obs.qc_pass.fillna(False).all()
    a.var_names = a.var['source_feature_id'].astype(str)
    a.var_names_make_unique()
    a.var.index.name = 'gene_symbol_unique'
    a.layers['counts'] = a.X.copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    return a


def feature_eligible(a, exclude_cc=False):
    excluded = np.zeros(a.n_vars, bool)
    for key in ('mt_feature','ERCC_feature','technical_feature','reporter_or_vector_feature'):
        excluded |= a.var[key].fillna(False).to_numpy(bool)
    names = a.var_names.astype(str)
    excluded |= np.asarray(names.str.lower().str.startswith(('mt-','ercc-')))
    if exclude_cc:
        excluded |= names.isin(CC_GENES)
    return ~excluded


def cluster_branch(a, n_hvg, prefix, resolution, n_pcs=30, neighbors=20, exclude_cc=True, umap=False):
    eligible = feature_eligible(a, exclude_cc=exclude_cc)
    b = a[:, eligible].copy()
    sc.pp.highly_variable_genes(b, n_top_genes=n_hvg, flavor='seurat', subset=True)
    sc.pp.scale(b, max_value=10)
    pcs = min(n_pcs, b.n_obs - 2, b.n_vars - 2)
    sc.tl.pca(b, n_comps=pcs, svd_solver='arpack', random_state=SEED)
    sc.pp.neighbors(b, n_neighbors=min(neighbors, b.n_obs-1), n_pcs=pcs, random_state=SEED)
    sc.tl.leiden(b, resolution=resolution, key_added='cluster', random_state=SEED, flavor='igraph', n_iterations=2)
    a.obs[prefix] = b.obs['cluster'].astype(str).to_numpy()
    a.obsm[prefix+'_pca'] = b.obsm['X_pca'].copy()
    if umap:
        sc.tl.umap(b, min_dist=.35, random_state=SEED)
        a.obsm[prefix+'_umap'] = b.obsm['X_umap'].copy()
    return b


def marker_summary(a, key, extra=()):
    genes = list(dict.fromkeys([g for panel in MARKERS.values() for g in panel] + list(extra)))
    genes = [g for g in genes if g in a.var_names]
    x = a[:, genes].X.toarray()
    rows = []
    for group in sorted(a.obs[key].unique(), key=lambda v: int(v) if str(v).isdigit() else str(v)):
        m = a.obs[key].astype(str).eq(str(group)).to_numpy()
        for j,g in enumerate(genes):
            rows.append((group,int(m.sum()),g,float(x[m,j].mean()),float((x[m,j]>0).mean())))
    return pd.DataFrame(rows,columns=['group','n','gene','mean_log1p_cp10k','fraction_detected'])


## Original standalone source: tree_core.py
The following raw Python source is an original standalone program, not a notebook
execution cell. Run it in its original CLI context when rebuilding this stage.
Where the original program invokes another source file by subprocess, manually
export that required original source section and configure its ROOT location.
No worker-export cell or replacement dispatcher is provided here.



## Original standalone source: joint-fit.py
The following raw Python source is an original standalone program, not a notebook
execution cell. Run it in its original CLI context when rebuilding this stage.
Where the original program invokes another source file by subprocess, manually
export that required original source section and configure its ROOT location.
No worker-export cell or replacement dispatcher is provided here.



## Original standalone source: local_analysis.py
The following raw Python source is an original standalone program, not a notebook
execution cell. Run it in its original CLI context when rebuilding this stage.
Where the original program invokes another source file by subprocess, manually
export that required original source section and configure its ROOT location.
No worker-export cell or replacement dispatcher is provided here.



## Original standalone source: local_fit.py
The following raw Python source is an original standalone program, not a notebook
execution cell. Run it in its original CLI context when rebuilding this stage.
Where the original program invokes another source file by subprocess, manually
export that required original source section and configure its ROOT location.
No worker-export cell or replacement dispatcher is provided here.



## Original accepted global-tree package
The following eight original notebook cells retain the original package and
joint tree figures. Supply their accepted upstream component files in the
original context. The original functions and export calls are unchanged.


In [ ]:
#1. =========== Shared paths, fixed accepted model and runtime
from pathlib import Path
import os, sys, json, pickle, hashlib, warnings, math, subprocess, io, textwrap
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path('.')
FIGURE_DIR = ROOT
DATA_DIR = ROOT
WORK = ROOT
RUNTIME = ROOT
BUNDLE_PATH = ROOT
VALIDATION_PATH = ROOT
SOURCE_MODEL = ROOT
SOURCE_PREPARED = ROOT
SOURCE_PNG = ROOT
CODEX_RUNTIME = ROOT
VALIDATOR = ROOT
VALIDATOR_PYTHON = CODEX_RUNTIME/'python/python.exe'
PDFTOPPM = CODEX_RUNTIME/'native/poppler/Library/bin/pdftoppm.exe'
FONT_DIR = ROOT
RANDOM_SEED, DPI = 42, 600
FATES = ['Ia','Ib','Ic','II']
WHOLE_SIZE, PATH_SIZE = (10.8,8.5), (7.,8.5)

for folder in [FIGURE_DIR, DATA_DIR, RUNTIME]: folder.mkdir(parents=True,exist_ok=True)
sys.dont_write_bytecode = True
os.environ.update(TMP=str(RUNTIME), TEMP=str(RUNTIME), PYTHONDONTWRITEBYTECODE='1')
try:
    import reportlab, pypdf
except ImportError:
    # Pure-Python fallback. NumPy, pandas and Pillow have already loaded from this kernel.
    sys.path.append(str(CODEX_RUNTIME/'python/Lib/site-packages'))
    import reportlab, pypdf
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.colors import HexColor, Color
from pypdf import PdfReader
from pypdf.generic import ContentStream


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def array_sha(value):
    return hashlib.sha256(np.asarray(value).tobytes()).hexdigest()


def read_pickle(path):
    with warnings.catch_warnings(), Path(path).open('rb') as stream:
        warnings.simplefilter('ignore',DeprecationWarning)
        return pickle.load(stream)
previous_validation = json.loads(VALIDATION_PATH.read_text(encoding='utf-8')) if VALIDATION_PATH.exists() else {}


In [ ]:
#2. =========== Package the accepted all-cell tree and all five figure selections
if BUNDLE_PATH.exists():
    bundle = read_pickle(BUNDLE_PATH)
else:
    accepted = read_pickle(SOURCE_MODEL)
    prepared = read_pickle(SOURCE_PREPARED)
    obs, model, projection = accepted['obs'].copy(), accepted['model'], accepted['projection']
    assert len(obs)==7355 and obs.index.equals(prepared['obs'].index)
    assert model['F'].shape==(30,153) and model['B'].shape==(153,153)
    assert model['parameters']['seed']==1 and model['parameters']['steps']==1000
    children = {node:[] for node in range(len(model['B']))}
    for i,j in projection['edges']: children[int(i)].append(int(j))
    def leaves(node):
        return [node] if not children[node] else [tip for child in sorted(children[node]) for tip in leaves(child)]
    root = int(projection['root'])
    tips = leaves(root)
    lx = {node:i for i,node in enumerate(tips)}
    node_x = {node:(min(lx[t] for t in leaves(node))+max(lx[t] for t in leaves(node)))/2
              for node in range(len(model['B']))}
    first_fork = root
    while len(children[first_fork])==1: first_fork=children[first_fork][0]
    assert (root,first_fork)==(36,38)
    edge_lookup = {tuple(edge):i for i,edge in enumerate(projection['edges'])}
    segments, edge_x = [], {}
    for start in np.flatnonzero(model['B'].sum(0)!=2):
        for child in children[int(start)]:
            path=[int(start),child]
            while model['B'][path[-1]].sum()==2: path.append(children[path[-1]][0])
            end=path[-1]
            segments.append(dict(start=int(start),end=end,path=path,x=node_x[end]))
            for i,j in zip(path,path[1:]): edge_x[(i,j)]=node_x[end]
    xy=accepted['plot_coordinates'].copy()
    rng=np.random.default_rng(RANDOM_SEED)
    expected_xy=np.column_stack([[edge_x[tuple(e)] for e in projection['edges'][projection['edge_id']]],projection['t']])
    expected_xy[:,0]+=rng.uniform(-.028,.028,len(obs))
    np.testing.assert_array_equal(xy,expected_xy)
    plot_order=rng.permutation(len(obs))
    figures={'all':dict(mask=np.ones(len(obs),bool),cells=len(obs),size=WHOLE_SIZE)}
    totals={f:int(obs.state.eq('Immature Type '+f+' SGN').sum()) for f in FATES}
    expected_counts={'Ia':2672,'Ib':1597,'Ic':1130,'II':2233}
    for fate in FATES:
        match=accepted['diagnostics']['fate_matching'][fate]
        tip=int(match['tip']); path=[tip]
        while path[-1]!=first_fork: path.append(int(projection['parent'][path[-1]]))
        path=path[::-1]
        ids=np.array([edge_lookup[(i,j)] for i,j in zip(path,path[1:])],int)
        mask=np.isin(projection['edge_id'],ids)
        terminal=[tip]
        while terminal[-1]!=root:
            terminal.append(int(projection['parent'][terminal[-1]]))
            if model['B'][terminal[-1]].sum()!=2: break
        term_ids=[edge_lookup[(j,i)] for i,j in zip(terminal,terminal[1:])]
        term_mask=np.isin(projection['edge_id'],term_ids)
        typed=obs.state.eq('Immature Type '+fate+' SGN').to_numpy()
        assert int(mask.sum())==expected_counts[fate]
        assert int((term_mask&typed).sum())==match['count']
        figures[fate]=dict(mask=mask,cells=int(mask.sum()),size=PATH_SIZE,start=first_fork,tip=tip,
            node_ids=path,edge_ids=ids,mask_sha256=array_sha(mask),
            terminal_mask=term_mask,terminal_typed=int((term_mask&typed).sum()),
            path_typed=int((mask&typed).sum()),typed_total=totals[fate],
            states=obs.loc[mask,'state'].value_counts().to_dict(),
            stages=obs.loc[mask,'stage_standardized'].value_counts().to_dict(),
            studies=obs.loc[mask,'study_id'].value_counts().to_dict())
    cvg_states=['CVG neuroblast','O-2: delaminating otic/CVG neuroblast',
        'O-3: developing CVG neuron (not SGN-specific)','Undifferentiated / proliferative CVG','Late CVG']
    assert not any(np.any(figures[f]['mask']&obs.state.isin(cvg_states)) for f in FATES)
    union=np.logical_or.reduce([figures[f]['mask'] for f in FATES])
    assert int(union.sum())==5023
    bundle=dict(schema_version=1,obs=obs,model=model,projection=projection,plot_coordinates=xy,
        X_css30=prepared['spaces']['css30'].copy(),node_x=node_x,segments=segments,plot_order=plot_order,
        first_fork=first_fork,tips=tips,figures=figures,accepted_key=accepted['key'],
        original_png=SOURCE_PNG.read_bytes(),diagnostics=accepted['diagnostics'],
        original_sources={str(p):sha(p) for p in [SOURCE_MODEL,SOURCE_PREPARED,SOURCE_PNG]},
        fitted_source_h5ad=prepared['source'],fitted_source_sha256=prepared['source_sha256'],
        provenance=dict(input_cells=7355,method='SimplePPT, independent all-cell cold start',
            actual_fit=dict(nodes=160,lam=3.,sigma=.095,max_steps=1000,seed=1,metric='euclidean'),
            fitted_space='HVG3000 CSS97 -> PCA30 -> centered global RMS normalization',
            node_contraction_tolerance=1e-8, no_grouped_fitting=True,no_attachment=True,
            no_fixed_endpoints=True,root_rule='Lowest mean cell age among occupied tips after fitting',
            path_selection='Original assigned edges along the first-fork-to-tip path; annotations are not a filter',
            geometry='Original F, B, edge projection, t and display coordinates retained exactly',
            all_png='Byte-for-byte accepted original; all PDF is a vector reconstruction from the same geometry',
            paper_reference='https://github.com/LouisFaure/sgnfates_paper/blob/main/04.Per-trajectory_analysis.ipynb'),
        model_hash=hashlib.sha256(model['F'].tobytes()+model['B'].tobytes()).hexdigest(),
        coordinates_hash=array_sha(xy))
    with BUNDLE_PATH.open('wb') as stream: pickle.dump(bundle,stream,protocol=5)

obs=bundle['obs']; model=bundle['model']; projection=bundle['projection']; xy=bundle['plot_coordinates']
assert hashlib.sha256(model['F'].tobytes()+model['B'].tobytes()).hexdigest()==bundle['model_hash']
assert array_sha(xy)==bundle['coordinates_hash']
assert bundle['figures']['Ic']['terminal_typed']==43 and bundle['figures']['Ic']['path_typed']==164
print(pd.DataFrame({f:{'path_cells':bundle['figures'][f]['cells'],
    'typed_on_path':bundle['figures'][f]['path_typed'],
    'typed_on_terminal':bundle['figures'][f]['terminal_typed'],
    'typed_total':bundle['figures'][f]['typed_total']} for f in FATES}).T.to_string())

In [ ]:
#3. =========== Editable vector drawing helpers and fixed appearance
FONT_SIZE, LABEL_SIZE, LEGEND_SIZE = 9.,8.,7.6
PATH_POINT_ALPHA = .88  # Appearance only; original source xy is never changed.
WHOLE_POINT_AREA, WHOLE_POINT_ALPHA = 1.4,.72
PATH_POINT_AREA = WHOLE_POINT_AREA  # Shared marker size preserves the whole-cell band width.
STATES=['CVG neuroblast','O-2: delaminating otic/CVG neuroblast',
    'O-3: developing CVG neuron (not SGN-specific)','Undifferentiated / proliferative CVG','Late CVG',
    'Differentiating SGN','Undifferentiated SGN','Type Ia + Type II precursor candidate',
    'Type Ib + Type Ic precursor candidate','Emerging Type Ic','Immature Type Ia SGN',
    'Immature Type Ib SGN','Immature Type Ic SGN','Immature Type II SGN']
STATE_LABELS=['CVG neuroblast','Delaminating otic/CVG neuroblast (O2)',
    'Developing CVG neuron (O3; not SGN-specific)','Undiff. / proliferative CVG','Late CVG',
    'Differentiating SGN','Undifferentiated SGN','Ia/II precursor candidate','Ib/Ic precursor candidate',
    'Emerging Ic','Immature Ia','Immature Ib','Immature Ic','Immature II']
STATE_COLORS=['#8A6E9C','#B49164','#5A89AD','#5F738A','#7AA8A4','#C6A068','#6C7590',
    '#A18AB2','#4F9695','#A8AE6D','#D66E73','#339EB8','#72AA8F','#A363A8']
STAGES=['E9.5','E10.5','E11.5','E13.5','E14.5','E15.5','E16.5','E17.5','E18.5']
STAGE_COLORS=['#f5b888','#f2a985','#ee9982','#e58780','#cd717e','#b55c7d','#914679','#653175','#3a1c71']
PALETTE=dict(zip(STATES,STATE_COLORS)); STAGE_PALETTE=dict(zip(STAGES,STAGE_COLORS))
RAMPS={'Ia':['#fff5f0','#fee0d2','#fcbba1','#fc9272','#fb6a4a','#ef3b2c','#cb181d','#a50f15','#67000d'],
       'Ib':['#f7fbff','#deebf7','#c6dbef','#9ecae1','#6baed6','#4292c6','#2171b5','#08519c','#08306b'],
       'Ic':['#f7fcf5','#e5f5e0','#c7e9c0','#a1d99b','#74c476','#41ab5d','#238b45','#006d2c','#00441b'],
       'II':['#fcfbfd','#efedf5','#dadaeb','#bcbddc','#9e9ac8','#807dba','#6a51a3','#54278f','#3f007d']}
for name,file in [('Arial','arial.ttf'),('Arial-Bold','arialbd.ttf')]:
    assert (FONT_DIR/file).exists()
    if name not in pdfmetrics.getRegisteredFontNames(): pdfmetrics.registerFont(TTFont(name,str(FONT_DIR/file)))
SCENES={}


def rgb(value):
    return HexColor(value) if isinstance(value,str) else Color(*value)


def draw_text(c,text,x,y,size=FONT_SIZE,font='Arial',color='#252525',align='left',angle=0,role='label'):
    lines=str(text).split('\n'); leading=size*1.18
    widths=[pdfmetrics.stringWidth(line,font,size) for line in lines]
    asc,desc=pdfmetrics.getAscentDescent(font,size)
    textobj=c.beginText(); textobj.setFont(font,size); textobj.setFillColor(rgb(color))
    for i,(line,width) in enumerate(zip(lines,widths)):
        dx=0 if align=='left' else (-width/2 if align=='center' else -width)
        theta=math.radians(angle); co,si=math.cos(theta),math.sin(theta)
        bx=x+dx*co+i*leading*si; by=y+dx*si-i*leading*co
        textobj.setTextTransform(co,si,-si,co,bx,by); textobj.textOut(line)
    c.drawText(textobj)
    left=0 if align=='left' else (-max(widths)/2 if align=='center' else -max(widths))
    corners=np.array([[left,desc-(len(lines)-1)*leading],[left,asc],
                      [left+max(widths),desc-(len(lines)-1)*leading],[left+max(widths),asc]])
    theta=math.radians(angle); rot=np.array([[math.cos(theta),math.sin(theta)],[-math.sin(theta),math.cos(theta)]])
    corners=corners@rot+np.array([x,y])
    c._scene.append(dict(text=str(text),bbox=[float(corners[:,0].min()),float(corners[:,1].min()),
        float(corners[:,0].max()),float(corners[:,1].max())],role=role))


def circle(c,x,y,r,fill,stroke=None,width=.5):
    k=r*4*(math.sqrt(2)-1)/3
    p=c.beginPath(); p.moveTo(x+r,y)
    p.curveTo(x+r,y+k,x+k,y+r,x,y+r)
    p.curveTo(x-k,y+r,x-r,y+k,x-r,y)
    p.curveTo(x-r,y-k,x-k,y-r,x,y-r)
    p.curveTo(x+k,y-r,x+r,y-k,x+r,y); p.close()
    c.setFillColor(rgb(fill))
    if stroke is not None: c.setStrokeColor(rgb(stroke)); c.setLineWidth(width)
    c.drawPath(p,fill=1,stroke=int(stroke is not None))


def line(c,x0,y0,x1,y1,color='#59616A',width=1.):
    c.setStrokeColor(rgb(color)); c.setLineWidth(width)
    c.line(x0,y0,x1,y1)


def chart_map(size,box):
    w,h=np.array(size)*72; bx,by,bw,bh=box
    xmax=3.5; xmin=-.5
    ymax=float(projection['root_distance'].max())*1.14
    ymin=-float(projection['root_distance'].max())*.06
    def transform(x,y): return bx*w+bw*w*(np.asarray(x)-xmin)/(xmax-xmin), by*h+bh*h*(ymax-np.asarray(y))/(ymax-ymin)
    return transform,(bx*w,by*h,bw*w,bh*h)


def draw_axes(c,size,box,title=None):
    w,h=np.array(size)*72; transform,rect=chart_map(size,box); x0,y0,pw,ph=rect
    for tick in range(7):
        _,yp=transform(0,tick)
        line(c,x0,yp,x0+pw,yp,'#E9E9E9',.4)
        line(c,x0-3,yp,x0,yp,'#63676B',.65)
        draw_text(c,str(tick),x0-6,yp-2.8,8,align='right',role='tick')
    line(c,x0,y0,x0,y0+ph,'#63676B',.8)
    draw_text(c,'Distance from root (CSS30 units)',x0-33,y0+ph/2,9,align='center',angle=90,role='axis')
    if title: draw_text(c,title,x0+pw/2,y0+ph+17,12,align='center',role='panel_title')
    return transform


def skeleton(c,transform,color='#59616A',width=1.):
    nx=bundle['node_x']; dist=projection['root_distance']
    for seg in bundle['segments']:
        i,j,xx=seg['start'],seg['end'],seg['x']
        x0,y0=transform(xx,dist[i]); x1,y1=transform(xx,dist[j]); line(c,x0,y0,x1,y1,color,width)
        if nx[i]!=xx:
            xa,ya=transform(nx[i],dist[i]); line(c,xa,ya,x0,y0,color,width)


def ramp_color(fate,fraction):
    anchors=np.array([[int(v[i:i+2],16)/255 for i in (1,3,5)] for v in RAMPS[fate]])
    at=float(np.clip(fraction,0,1))*(len(anchors)-1)
    low=min(int(at),len(anchors)-2); weight=at-low
    return tuple(anchors[low]*(1-weight)+anchors[low+1]*weight)


def make_canvas(path,size):
    c=canvas.Canvas(str(path),pagesize=tuple(np.array(size)*72),pageCompression=1,
        invariant=1,initialFontName='Arial',initialFontSize=FONT_SIZE)
    c.setTitle(path.stem); c.setAuthor('Developmental map of auditory neurons')
    c.setSubject('Fixed all-cell SimplePPT tree; editable vectors and live Arial text')
    c._scene=[]
    return c


def finish_canvas(c,key,size):
    w,h=np.array(size)*72
    for item in c._scene:
        x0,y0,x1,y1=item['bbox']
        assert x0>=0 and y0>=0 and x1<=w and y1<=h, ('Clipped text',item)
    endpoints=[item for item in c._scene if item['role']=='endpoint']
    for i,a in enumerate(endpoints):
        for z in endpoints[i+1:]:
            x0,y0,x1,y1=a['bbox']; a0,b0,a1,b1=z['bbox']
            assert x1<=a0 or a1<=x0 or y1<=b0 or b1<=y0,('Overlapping endpoint labels',a,z)
    SCENES[key]=c._scene
    c.showPage(); c.save()

In [ ]:
#4. =========== Compact whole-cell tree and stage distributions
# Display parameters affect only this whole-cell figure. Distances and assignments are fixed.
TREE_BOXES = [(0.62/10.8,.19,2.8/10.8,.57), (3.95/10.8,.19,2.8/10.8,.57)]
DENSITY_LEFT, DENSITY_WIDTH = 7.10*72, .78*72
LEGEND_LEFT, LEGEND_TOP = 8.15*72, .79*8.5*72
WHOLE_END_SIZE, STATE_LEGEND_SIZE = 7.3, 7.3
DENSITY_BANDWIDTH, DENSITY_GRID_SIZE, DENSITY_ALPHA = .12, 768, .20
HORIZONTAL_BRANCH_SCALE, CELL_BAND_WIDTH_SCALE = .5, 1.75  # Relative to the preceding compact whole-cell view.
TREE_X_CENTER = 1.5
END_LABEL_T = {'Ic':5.9,'Ia':6.4}  # Text placement only; fitted tips and cell t stay fixed.

# Manually specified display colors, reusing the accepted annotation color families.
MILESTONE_COLORS = {36:'#8A6E9C',38:'#6C7590',96:'#4F9695',92:'#A18AB2',
    114:'#339EB8',86:'#72AA8F',63:'#D66E73',41:'#A363A8'}
GRADIENT_LEVELS, GRADIENT_LEGEND_STEPS = 1000, 256
GRADIENT_LEGEND_WIDTH, GRADIENT_LEGEND_HEIGHT = 17., 7.
SEGMENT_LEGEND_ORDER = [0,1,5,6,2,3,4]
SEGMENT_LABELS = {0:'Root to first branch point',1:'Ib/Ic shared segment',2:'Ia/II shared segment',
    3:'Ia-associated terminal segment',4:'II-associated terminal segment',
    5:'Ib-enriched terminal segment',6:'Emerging Ic-enriched terminal'}
STAGE_LEGEND_TOP = 245.62799999999984



def stage_densities():
    t=np.asarray(projection['t'],float); limit=float(projection['root_distance'].max())
    grid=np.linspace(0,limit,DENSITY_GRID_SIZE); values={}; counts={}
    for stage in STAGES:
        v=t[obs.stage_standardized.eq(stage).to_numpy()]
        assert len(v)>0
        # Reflection at both fitted endpoints retains mass near the root and final tip.
        centers=np.concatenate([v,-v,2*limit-v])
        density=np.exp(-.5*((grid[:,None]-centers[None,:])/DENSITY_BANDWIDTH)**2).sum(1)
        density/=len(v)*DENSITY_BANDWIDTH*np.sqrt(2*np.pi)
        density/=np.trapezoid(density,grid)
        assert np.all(np.isfinite(density)) and np.all(density>=0)
        assert abs(np.trapezoid(density,grid)-1)<1e-12
        values[stage]=density; counts[stage]=int(len(v))
    return dict(grid=grid,values=values,counts=counts,
        method='Gaussian KDE; reflection at 0 and maximum fitted root distance',
        bandwidth_css30=DENSITY_BANDWIDTH,normalization='Unit integral separately within each stage',
        display_scale='Each curve divided by its own peak; relative density, peak=1',
        display_values={s:v/v.max() for s,v in values.items()},
        density_sha256={s:array_sha(v) for s,v in values.items()})


def draw_density(c,transform,item):
    grid=item['grid']; maximum=1.
    _,yy=transform(np.zeros(len(grid)),grid)
    # Reverse layering keeps narrow early distributions visible on the common baseline.
    for stage in reversed(STAGES):
        xx=DENSITY_LEFT+DENSITY_WIDTH*item['display_values'][stage]
        p=c.beginPath(); p.moveTo(DENSITY_LEFT,float(yy[0]))
        for x,y in zip(xx,yy): p.lineTo(float(x),float(y))
        p.lineTo(DENSITY_LEFT,float(yy[-1])); p.close()
        c.setFillColor(rgb(STAGE_PALETTE[stage])); c.setFillAlpha(DENSITY_ALPHA)
        c.drawPath(p,fill=1,stroke=0)
        c.setFillAlpha(1); c.setStrokeColor(rgb(STAGE_PALETTE[stage])); c.setLineWidth(.75)
        p=c.beginPath(); p.moveTo(float(xx[0]),float(yy[0]))
        for x,y in zip(xx[1:],yy[1:]): p.lineTo(float(x),float(y))
        c.drawPath(p,fill=0,stroke=1)
    _,yt=transform(0,0); _,yb=transform(0,float(grid[-1]))
    line(c,DENSITY_LEFT,yt,DENSITY_LEFT,yb,'#8B8B8B',.45)
    axis_y=.19*8.5*72
    line(c,DENSITY_LEFT,axis_y,DENSITY_LEFT+DENSITY_WIDTH,axis_y,'#63676B',.5)
    draw_text(c,'0',DENSITY_LEFT,axis_y-11,7,align='center',role='tick')
    draw_text(c,'1',DENSITY_LEFT+DENSITY_WIDTH,axis_y-11,7,align='center',role='tick')
    draw_text(c,'Relative density',DENSITY_LEFT+DENSITY_WIDTH/2,axis_y-24,7.5,align='center',role='axis')
    draw_text(c,'Stage distribution',DENSITY_LEFT+DENSITY_WIDTH/2,.76*8.5*72+17,9.5,align='center',role='panel_title')
    draw_text(c,'Per-stage peak = 1',DENSITY_LEFT+DENSITY_WIDTH/2,.76*8.5*72+4,6.6,align='center',role='legend')



def whole_display_geometry():
    # Source xy and graph are never edited; marker size stays fixed.
    points_per_x=WHOLE_SIZE[0]*72*TREE_BOXES[0][2]/4
    diameter=math.sqrt(WHOLE_POINT_AREA)
    lookup={tuple(e):i for i,e in enumerate(projection['edges'])}
    edge_segment=np.full(len(projection['edges']),-1,int)
    for k,seg in enumerate(bundle['segments']):
        for i,j in zip(seg['path'],seg['path'][1:]): edge_segment[lookup[(i,j)]]=k
    assert np.all(edge_segment>=0)
    segment_ids=edge_segment[projection['edge_id']]
    display_xy=xy.copy(); widths=[]; connectors=[]
    for k,seg in enumerate(bundle['segments']):
        mask=segment_ids==k; jitter=xy[mask,0]-seg['x']
        span=float(np.ptp(jitter))*points_per_x
        assert span>0
        old_width=span+diameter
        jitter_factor=(CELL_BAND_WIDTH_SCALE*old_width-diameter)/span
        center=TREE_X_CENTER+HORIZONTAL_BRANCH_SCALE*(seg['x']-TREE_X_CENTER)
        display_xy[mask,0]=center+jitter_factor*jitter
        new_width=float(np.ptp(display_xy[mask,0]))*points_per_x+diameter
        ratio=new_width/old_width
        assert abs(ratio-CELL_BAND_WIDTH_SCALE)<1e-12
        widths.append(dict(segment=k,start=seg['start'],end=seg['end'],cells=int(mask.sum()),
            old_visible_width_pt=old_width,new_visible_width_pt=new_width,
            visible_width_ratio=ratio,jitter_factor=jitter_factor))
        nx=bundle['node_x'][seg['start']]
        if nx!=seg['x']:
            old=abs(nx-seg['x'])*points_per_x
            a=TREE_X_CENTER+HORIZONTAL_BRANCH_SCALE*(nx-TREE_X_CENTER)
            new=abs(a-center)*points_per_x
            assert abs(new/old-HORIZONTAL_BRANCH_SCALE)<1e-12
            connectors.append(dict(segment=k,old_length_pt=old,new_length_pt=new,ratio=new/old))
    np.testing.assert_array_equal(display_xy[:,1],xy[:,1])
    return dict(cell_xy=display_xy,segment_ids=segment_ids,cell_band_checks=widths,
        horizontal_connector_checks=connectors,marker_diameter_pt=diameter,
        source_coordinates_sha256=bundle['coordinates_hash'])



def milestone_gradient(pst,start,end):
    # Equivalent to LinearSegmentedColormap.from_list(N=1000), then to_hex.
    pst=np.asarray(pst,dtype=np.float64)
    assert np.isfinite(pst).all() and np.all((pst>=0)&(pst<=1))
    c0=np.array([int(start[j:j+2],16) for j in (1,3,5)],float)/255
    c1=np.array([int(end[j:j+2],16) for j in (1,3,5)],float)/255
    q=((GRADIENT_LEVELS-1)*np.linspace(0.,1.,GRADIENT_LEVELS))/(GRADIENT_LEVELS-1)
    lut=np.clip(q[:,None]*(c1-c0)[None,:]+c0,0,1)
    lut[0],lut[-1]=c0,c1
    scaled=pst*GRADIENT_LEVELS
    scaled[scaled==GRADIENT_LEVELS]=GRADIENT_LEVELS-1
    indices=scaled.astype(np.int16)
    values=np.rint(lut[indices]*255).astype(np.uint8)
    hexes=np.array(['#'+''.join(f'{int(v):02x}' for v in row) for row in values])
    return hexes,values,indices


def segment_gradient_colors(segment_ids):
    t=np.asarray(projection['t'],float); fraction=np.empty(len(t),float)
    color_hex=np.empty(len(t),dtype='<U7'); color_rgb=np.empty((len(t),3),np.uint8)
    lookup_index=np.empty(len(t),np.int16); segments=[]
    assert set(np.unique(segment_ids))==set(range(len(bundle['segments'])))
    for k,seg in enumerate(bundle['segments']):
        mask=segment_ids==k; local_t=t[mask]; tmin=float(local_t.min()); tmax=float(local_t.max())
        assert tmax>tmin
        pst=(local_t-tmin)/(tmax-tmin)
        start=MILESTONE_COLORS[seg['start']]; end=MILESTONE_COLORS[seg['end']]
        hh,rr,ii=milestone_gradient(pst,start,end)
        fraction[mask]=pst; color_hex[mask]=hh; color_rgb[mask]=rr; lookup_index[mask]=ii
        assert hh[np.argmin(local_t)]==start.lower() and hh[np.argmax(local_t)]==end.lower()
        segments.append(dict(id=k,start=seg['start'],end=seg['end'],cells=int(mask.sum()),
            label=SEGMENT_LABELS[k],start_color=start,end_color=end,t_min=tmin,t_max=tmax,
            fraction_min=float(pst.min()),fraction_max=float(pst.max()),unique_colors=len(np.unique(hh))))
    assert sum(s['cells'] for s in segments)==len(obs)==7355
    return dict(method='scFates v0.4.0 within-segment t min-max, linear endpoint RGB colormap, 1000-entry LUT, to_hex rounding',
        source_t='projection.t; independent of display positions and endpoint label offsets',
        node_colors=MILESTONE_COLORS,levels=GRADIENT_LEVELS,segment_ids=segment_ids.copy(),
        fraction=fraction,lookup_index=lookup_index,color_hex=color_hex,color_rgb_uint8=color_rgb,
        segments=segments,legend_order=SEGMENT_LEGEND_ORDER,
        node_color_source='Explicit display settings reusing the existing project colors; annotations are not changed',
        colors_sha256=array_sha(color_rgb),fraction_sha256=array_sha(fraction),
        literature=['https://github.com/LouisFaure/sgnfates_paper/blob/main/make_fig2.ipynb',
            'https://github.com/LouisFaure/scFates/blob/v0.4.0/scFates/plot/utils.py#L176-L206'])


def draw_segment_legend(c,x,y,coloring):
    draw_text(c,'Tree segment (n)',x,y,8.5,'Arial-Bold',role='legend')
    draw_text(c,'Within-segment pseudotime: start to end',x,y-13,6.6,role='legend')
    y-=33
    for k in SEGMENT_LEGEND_ORDER:
        seg=coloring['segments'][k]
        # Independent solid vector strips avoid raster gradients and PDF shading objects.
        fractions=np.linspace(0,1,GRADIENT_LEGEND_STEPS)
        hexes,_,_=milestone_gradient(fractions,seg['start_color'],seg['end_color'])
        c.setFillAlpha(WHOLE_POINT_ALPHA)
        for i,color in enumerate(hexes):
            c.setFillColor(rgb(color))
            c.rect(x+GRADIENT_LEGEND_WIDTH*i/GRADIENT_LEGEND_STEPS,y-1,
                GRADIENT_LEGEND_WIDTH/GRADIENT_LEGEND_STEPS+.005,GRADIENT_LEGEND_HEIGHT,fill=1,stroke=0)
        c.setFillAlpha(1);c.setStrokeColor(rgb('#222222'));c.setLineWidth(.45)
        c.rect(x,y-1,GRADIENT_LEGEND_WIDTH,GRADIENT_LEGEND_HEIGHT,fill=0,stroke=1)
        label=textwrap.fill(f'{seg["label"]} ({seg["cells"]:,})',31)
        assert len(label.split('\n'))<=2
        draw_text(c,label,x+GRADIENT_LEGEND_WIDTH+7,y,STATE_LEGEND_SIZE,role='legend')
        y-=24
    draw_text(c,'n = all cells assigned to the segment',x,y+1,6.6,role='legend')
    assert y>STAGE_LEGEND_TOP+10


def draw_all():
    size=WHOLE_SIZE; w,h=np.array(size)*72
    c=make_canvas(ROOT,size)
    display_geometry=whole_display_geometry(); display_xy=display_geometry['cell_xy']
    coloring=segment_gradient_colors(display_geometry['segment_ids'])
    endpoints={'Ia':'Ia-associated end','II':'II-associated end','Ib':'Ib-enriched end','Ic':'Emerging Ic-enriched end'}
    for box,field,palette,title in zip(TREE_BOXES,['segment_gradient','stage_standardized'],[None,STAGE_PALETTE],['Segment pseudotime','Developmental stage']):
        base_transform=draw_axes(c,size,box,title)
        def transform(x,y):
            return base_transform(TREE_X_CENTER+HORIZONTAL_BRANCH_SCALE*(np.asarray(x)-TREE_X_CENTER),y)
        skeleton(c,transform)
        c.setFillAlpha(WHOLE_POINT_ALPHA)
        for idx in bundle['plot_order']:
            xp,yp=base_transform(*display_xy[idx])
            color=coloring['color_hex'][idx] if field=='segment_gradient' else palette[obs.iloc[idx][field]]
            circle(c,float(xp),float(yp),math.sqrt(WHOLE_POINT_AREA)/2,color)
        c.setFillAlpha(1)
        xp,yp=transform(bundle['node_x'][projection['root']],0)
        circle(c,float(xp),float(yp),1.5,'#FFFFFF','#333333',.6)
        draw_text(c,'Root',float(xp),float(yp)+8,WHOLE_END_SIZE,align='center',role='endpoint')
        for fate in FATES:
            item=bundle['figures'][fate]; tip=item['tip']
            xp,yp=transform(bundle['node_x'][tip],projection['root_distance'][tip])
            circle(c,float(xp),float(yp),1.5,'#FFFFFF','#333333',.6)
            top=float(yp)-(3 if fate=='Ic' else 9)
            label_x,_=base_transform(bundle['node_x'][tip],projection['root_distance'][tip])
            if fate in END_LABEL_T:
                _,label_y=base_transform(bundle['node_x'][tip],END_LABEL_T[fate])
                top=float(label_y)+6.5
            c.setDash(1,1.5)
            if fate=='Ic':
                _,bend_y=base_transform(0,5.45)
                line(c,float(xp),float(yp)-2,float(xp),float(bend_y),'#999999',.35)
                line(c,float(xp),float(bend_y),float(label_x),top+2,'#999999',.35)
            else:
                line(c,float(xp),float(yp)-2,float(label_x),top+2,'#999999',.35)
            c.setDash()
            draw_text(c,f'{endpoints[fate]}\nImmature {fate}: {item["terminal_typed"]}/{item["typed_total"]}',
                float(label_x),top-6.5,WHOLE_END_SIZE,align='center',role='endpoint')
    density=stage_densities(); draw_density(c,base_transform,density)
    draw_text(c,'All-cell joint SimplePPT',.05*w,.956*h,18,'Arial-Bold',role='title')
    draw_text(c,'7,355 cells together  |  css30  |  N=160  |  lambda=3  |  sigma=0.095  |  seed=1',.05*w,.914*h,9.5)
    draw_text(c,'153 displayed nodes  |  3 branch points  |  4 non-root tips  |  CVG on initial path: 563/563',.05*w,.877*h,9)
    draw_text(c,'Paired topology observed; terminal cell assignment does not pass QA',.05*w,.836*h,9.5,color='#B94B43')
    x=LEGEND_LEFT; draw_segment_legend(c,x,LEGEND_TOP,coloring)
    y=STAGE_LEGEND_TOP
    draw_text(c,'Developmental stage (n)',x,y,8.5,'Arial-Bold',role='legend'); y-=17
    arrow_top=y+3; arrow_bottom=y-8*10.5+3
    line(c,x-9,arrow_top,x-9,arrow_bottom,'#454545',.75)
    p=c.beginPath(); p.moveTo(x-11.4,arrow_bottom+4); p.lineTo(x-9,arrow_bottom); p.lineTo(x-6.6,arrow_bottom+4)
    c.setStrokeColor(rgb('#454545')); c.setLineWidth(.75); c.drawPath(p,stroke=1,fill=0)
    for stage in STAGES:
        circle(c,x+3.5,y+2,3,STAGE_PALETTE[stage])
        draw_text(c,f'{stage} ({density["counts"][stage]:,})',x+13,y,7.5,role='legend'); y-=10.5
    assert y>95, ('Legend exceeds available height',y)
    draw_text(c,'Every fit uses all 7,355 cells. No grouping, attachment, fixed endpoints or post-fit branch rewiring.',.065*w,.117*h,7.5)
    draw_text(c,'Tip counts: annotated subtype on terminal / all of that subtype. Stage curves: each peak = 1; common bandwidth 0.12.',.065*w,.082*h,7.2,color='#555A61')
    draw_text(c,'Root chosen by lowest mean age after fitting. Vertical distances retained. Curve widths do not compare cell abundance.',.065*w,.047*h,7.5,color='#555A61')
    finish_canvas(c,'all',size)
    subprocess.run([str(PDFTOPPM),'-singlefile','-r',str(DPI),'-png',
        str(ROOT),str(ROOT)],check=True,capture_output=True)
    bundle['stage_density']=density
    bundle['segment_gradient_coloring']=coloring
    bundle['whole_display_geometry']=display_geometry
    bundle['figures']['all']['size']=WHOLE_SIZE
    bundle['render_spec']=dict(whole_size=WHOLE_SIZE,tree_boxes=TREE_BOXES,
        old_tree_width_in=3.64,axes_width_in=2.8,new_tree_width_in=2.8*HORIZONTAL_BRANCH_SCALE,
        horizontal_width_ratio=2.8*HORIZONTAL_BRANCH_SCALE/3.64,
        branch_spacing_scale=HORIZONTAL_BRANCH_SCALE,cell_band_width_scale=CELL_BAND_WIDTH_SCALE,
        cell_band_width_definition="Point-center span plus unchanged circular marker diameter",
        endpoint_labels="Thin dotted leaders, with Ic and Ia text moved below the cell bands; leaders are not graph edges",
        endpoint_label_text_t=END_LABEL_T,
        vertical_axis_size_in=.57*8.5,original_source_coordinates_unchanged=True,
        raw_annotation_palette=PALETTE,stage_palette=STAGE_PALETTE,
        left_panel_color='Within-segment pseudotime gradient',milestone_colors=MILESTONE_COLORS,
        segment_legend='Seven vector gradients matching cell colors; start to end',
        gradient_levels=GRADIENT_LEVELS,gradient_legend_steps=GRADIENT_LEGEND_STEPS,
        gradient_legend_width_pt=GRADIENT_LEGEND_WIDTH,gradient_legend_height_pt=GRADIENT_LEGEND_HEIGHT,
        stage_legend='Ordered circles, all nine stages, early-to-late arrow',
        density_left_pt=DENSITY_LEFT,density_width_pt=DENSITY_WIDTH,
        density_bandwidth_css30=DENSITY_BANDWIDTH,density_grid_size=DENSITY_GRID_SIZE,
        density_fill_alpha=DENSITY_ALPHA,whole_end_font_size=WHOLE_END_SIZE,
        state_legend_font_size=STATE_LEGEND_SIZE,whole_point_area=WHOLE_POINT_AREA,
        whole_point_alpha=WHOLE_POINT_ALPHA)
    bundle['schema_version']=3
    bundle['provenance']['left_panel_color']='Within-segment pseudotime gradients; original 14 cell annotations retained unchanged'
    bundle['provenance']['all_png']='Compact layout rendered from editable vector PDF; original_png retained only as historical source'


In [ ]:
#5. =========== Four path-selected SGN trajectories on the unchanged gray tree
PATH_BOX = (.62/PATH_SIZE[0],.19,2.8/PATH_SIZE[0],.57)

def draw_path(fate):
    size=PATH_SIZE; w,h=np.array(size)*72
    item=bundle['figures'][fate]; mask=item['mask']; start=item['start']; tip=item['tip']
    c=make_canvas(FIGURE_DIR/f'tree_{fate}.pdf',size)
    base_transform=draw_axes(c,size,PATH_BOX)
    def transform(x,y):
        return base_transform(TREE_X_CENTER+HORIZONTAL_BRANCH_SCALE*(np.asarray(x)-TREE_X_CENTER),y)
    skeleton(c,transform,color='#B4B4B4',width=1.1)
    offset=float(projection['root_distance'][start])
    length=float(projection['root_distance'][tip])-offset
    display_xy=bundle['whole_display_geometry']['cell_xy']
    c.setFillAlpha(PATH_POINT_ALPHA)
    for idx in bundle['plot_order'][mask[bundle['plot_order']]]:
        xp,yp=base_transform(*display_xy[idx]); fraction=(projection['t'][idx]-offset)/length
        circle(c,float(xp),float(yp),math.sqrt(PATH_POINT_AREA)/2,ramp_color(fate,fraction))
    c.setFillAlpha(1); c.setStrokeAlpha(1)
    root=int(projection['root']); nx=bundle['node_x']; dist=projection['root_distance']
    xp,yp=transform(nx[root],dist[root]); circle(c,float(xp),float(yp),1.5,'#FFFFFF','#777777',.5)
    draw_text(c,'Root',float(xp),float(yp)+8,8,color='#777777',align='center',role='endpoint')
    for target in FATES:
        tt=bundle['figures'][target]['tip']; xp,yp=transform(nx[tt],dist[tt])
        color='#252525' if target==fate else '#888888'
        circle(c,float(xp),float(yp),1.5,'#FFFFFF',color,.5)
        draw_text(c,target+' end',float(xp),float(yp)-16,8,
            'Arial-Bold' if target==fate else 'Arial',color,align='center',role='endpoint')
    xp,yp=transform(nx[start],dist[start])
    circle(c,float(xp),float(yp),2.1,'#FFFFFF','#444444',.7)
    draw_text(c,'Path start',float(xp)+6,float(yp)+7,7.8)
    draw_text(c,f'{fate} trajectory on the joint tree',.065*w,.956*h,17,'Arial-Bold',role='title')
    draw_text(c,f'First branch point to {fate} end | {item["cells"]:,} path-selected cells',.065*w,.914*h,10)
    draw_text(c,'Same display geometry as tree_all; original annotations retained',.065*w,.877*h,9.5)
    draw_text(c,'Cells selected by tree path; color shows path pseudotime',.065*w,.836*h,9,color='#555A61')
    lx=.72*w; ly=.75*h
    circle(c,lx+3,ly,2.3,ramp_color(fate,.65),'#333333',.2)
    draw_text(c,f'Selected {fate} path\n({item["cells"]:,} cells)',lx+14,ly-2,8,role='legend')
    line(c,lx-1,ly-35,lx+8,ly-35,'#B4B4B4',1.1)
    draw_text(c,'Other tree segments',lx+14,ly-38,8,role='legend')
    bx,by,bw,bh=lx+1,.39*h,9.,.22*h
    steps=96
    for k in range(steps):
        c.setFillColor(rgb(ramp_color(fate,(k+.5)/steps)))
        c.rect(bx,by+bh*k/steps,bw,bh/steps+.02,fill=1,stroke=0)
    for fraction in [0,.5,1.]:
        yy=by+bh*fraction; line(c,bx+bw,yy,bx+bw+3,yy,'#555555',.5)
        draw_text(c,f'{fraction*length:.2f}',bx+bw+6,yy-2.8,8,role='tick')
    draw_text(c,'Path pseudotime\n(CSS30 distance)',bx,by+bh+19,8,role='legend')
    typed=f'Immature {fate}'
    draw_text(c,f'Original {typed}:\nOn entire path: {item["path_typed"]}/{item["typed_total"]}\nOn terminal segment:\n{item["terminal_typed"]}/{item["typed_total"]}',
        lx,by-29,8,role='legend')
    draw_text(c,'Path membership follows the accepted tree; it is not a selection of all subtype-annotated cells.',.065*w,.117*h,7.5)
    draw_text(c,'Shared upstream cells can appear in both paths of a pair. No local refit or attachment is used.',.065*w,.082*h,7.5,color='#555A61')
    foot='The Ic end is enriched for Emerging Ic; original Immature Ic agreement remains limited.' if fate=='Ic' else 'This view highlights a fitted route and does not establish cell fate.'
    draw_text(c,foot,.065*w,.047*h,7.5,color='#555A61')
    bundle.setdefault('path_render_spec',{})[fate]=dict(axes_box=PATH_BOX,axes_width_in=2.8,
        display_xy_sha256=array_sha(display_xy),point_area=PATH_POINT_AREA,
        branch_spacing_scale=HORIZONTAL_BRANCH_SCALE,cell_band_width_scale=CELL_BAND_WIDTH_SCALE,
        geometry='Same node/cell paper coordinates as the first panel of tree_all',
        color='Existing path-specific pseudotime ramp',path_mask_sha256=array_sha(mask))
    # Exact physical coordinate check, not only the shared topology.
    reference,_=chart_map(WHOLE_SIZE,TREE_BOXES[0])
    px,py=base_transform(display_xy[:,0],display_xy[:,1]); rx,ry=reference(display_xy[:,0],display_xy[:,1])
    np.testing.assert_allclose(px,rx,atol=1e-12,rtol=0);np.testing.assert_allclose(py,ry,atol=1e-12,rtol=0)
    finish_canvas(c,fate,size)
    subprocess.run([str(PDFTOPPM),'-singlefile','-r',str(DPI),'-png',
        str(FIGURE_DIR/f'tree_{fate}.pdf'),str(FIGURE_DIR/f'tree_{fate}')],check=True,capture_output=True)

In [ ]:
#6. =========== Export without fitting any tree
REEXPORT_PATHS = True  # Redraw all four using the shared whole-cell display geometry.
assert PDFTOPPM.exists() and VALIDATOR.exists() and VALIDATOR_PYTHON.exists()
if not all((FIGURE_DIR/f'tree_all.{ext}').exists() for ext in ['pdf','png']):
    draw_all()
for fate in FATES:
    if REEXPORT_PATHS or not all((FIGURE_DIR/f'tree_{fate}.{ext}').exists() for ext in ['png','pdf']):
        draw_path(fate)
assert hashlib.sha256(model['F'].tobytes()+model['B'].tobytes()).hexdigest()==bundle['model_hash']
assert array_sha(xy)==bundle['coordinates_hash']
with BUNDLE_PATH.open('wb') as stream: pickle.dump(bundle,stream,protocol=5)
print('Updated all four trajectory trees using the exact tree_all display geometry; whole-cell figure retained.')


In [ ]:
#7. =========== Validate editable PDFs, 600-DPI PNGs and all saved selections
keys=['all']+FATES
pdf_paths=[FIGURE_DIR/f'tree_{key}.pdf' for key in keys]
pdf_run=subprocess.run([str(VALIDATOR_PYTHON),'-B',str(VALIDATOR),*map(str,pdf_paths),
    '--expected-clips','0','--max-clips','0','--pdftoppm',str(PDFTOPPM),
    '--render-dpi','150','--render-dir',str(ROOT)],capture_output=True,text=True,encoding='utf-8')
assert pdf_run.returncode==0,pdf_run.stdout+'\n'+pdf_run.stderr
png_reports=[]
for group,size in [(['all'],WHOLE_SIZE),(FATES,PATH_SIZE)]:
    run=subprocess.run([str(VALIDATOR_PYTHON),'-B',str(VALIDATOR),
        *[str(FIGURE_DIR/f'tree_{key}.png') for key in group],
        '--expected-size-in',f'{size[0]}x{size[1]}','--target-dpi',str(DPI)],
        capture_output=True,text=True,encoding='utf-8')
    assert run.returncode==0,run.stdout+'\n'+run.stderr
    png_reports.append(run.stdout)
artifacts={}
for key in keys:
    pdf=FIGURE_DIR/f'tree_{key}.pdf'; png=FIGURE_DIR/f'tree_{key}.png'
    reader=PdfReader(pdf); assert len(reader.pages)==1
    page=reader.pages[0]; content=ContentStream(page.get_contents(),reader)
    operators=[op for args,op in content.operations]
    assert b'W' not in operators and b'W*' not in operators and b'Do' not in operators
    assert b'h' in operators and b'c' in operators and b'Tj' in operators
    extracted=' '.join(page.extract_text().split())
    for item in SCENES.get(key,[]):
        assert ' '.join(item['text'].split()) in extracted,('Missing live text',key,item['text'])
    with Image.open(png) as image:
        pixels=list(image.size); dpi=list(image.info['dpi'])
        assert image.size==tuple(round(v*DPI) for v in bundle['figures'][key]['size'])
    if key not in SCENES:
        assert sha(pdf)==previous_validation['artifacts'][key]['pdf_sha256']
    artifacts[key]=dict(pdf=str(pdf),png=str(png),pixels=pixels,dpi=dpi,
        pdf_sha256=sha(pdf),png_sha256=sha(png),clips=0,max_clip_depth=0,
        raster_images=0,form_calls=0,fonts=['ArialMT','Arial-BoldMT'],
        expected_semantic_text_objects=len(SCENES[key]) if key in SCENES else previous_validation['artifacts'][key]['expected_semantic_text_objects'],live_labels_verified=True)
original_sources_unchanged={path:sha(path)==value for path,value in bundle['original_sources'].items() if Path(path).exists()}
assert all(original_sources_unchanged.values())
assert array_sha(xy)==bundle['coordinates_hash']
summary=dict(input_cells=len(obs),model_hash=bundle['model_hash'],coordinates_hash=bundle['coordinates_hash'],
    source_key=bundle['accepted_key'],provenance=bundle['provenance'],
    paths={f:{k:v for k,v in bundle['figures'][f].items() if k not in ['mask','terminal_mask','edge_ids']}
           for f in FATES},
    artifacts=artifacts,PDF_automatic_validation=pdf_run.stdout,PNG_automatic_validation=png_reports,
    original_sources_unchanged=original_sources_unchanged,all_original_png_exact=False,original_png_retained_as_historical_source=True,
    source_geometry_exact=True,figures=5,png_files=5,pdf_files=5,
    gradient_coloring={k:v for k,v in bundle['segment_gradient_coloring'].items() if k not in ['segment_ids','fraction','lookup_index','color_hex','color_rgb_uint8']},
    path_render_spec=bundle['path_render_spec'],
    render_spec=bundle['render_spec'],display_geometry_checks={k:v for k,v in bundle['whole_display_geometry'].items() if k not in ['cell_xy','segment_ids']},stage_density=dict(method=bundle['stage_density']['method'],bandwidth_css30=DENSITY_BANDWIDTH,counts=bundle['stage_density']['counts'],normalization=bundle['stage_density']['normalization'],display_scale=bundle['stage_density']['display_scale'],display_peaks={s:float(v.max()) for s,v in bundle['stage_density']['display_values'].items()},areas={s:float(np.trapezoid(v,bundle['stage_density']['grid'])) for s,v in bundle['stage_density']['values'].items()}),
    data_file_contract='tree_results.pkl includes the unchanged model and selections, current rendering settings, per-cell segment-gradient colors and stage-density arrays; tree_validation.json includes checks',
    software=dict(python=sys.version,numpy=np.__version__,pandas=pd.__version__,reportlab=reportlab.Version,pypdf=pypdf.__version__),
    notebook=dict(path=str(ROOT),execution='A/B section (first eight code cells) executed in order; subsequent C-G cells unchanged'))
VALIDATION_PATH.write_text(json.dumps(summary,indent=2),encoding='utf-8')
# Check A/B outputs without limiting other analyses in this shared directory.
assert BUNDLE_PATH.is_file() and VALIDATION_PATH.is_file()
print(pdf_run.stdout.splitlines()[-1])
print('Validated the whole-cell figure and all four synchronized trajectory trees.')

In [ ]:
#8. =========== Preview the five figures without saving extra images
from IPython.display import display, Image as NotebookImage
for key in keys:
    with Image.open(FIGURE_DIR/f'tree_{key}.png') as image:
        image.thumbnail((950,750))
        preview=io.BytesIO(); image.save(preview,format='PNG')
    display(NotebookImage(data=preview.getvalue()))


## Original conditional graft and accepted displays
These original cells retain the registered local component, accepted component
checks and original joint displays. They keep the original cached-result and
original component-rebuild options. No new graft entry point is added.


In [ ]:
#1. =========== Paths, runtime and accepted parameters
from pathlib import Path
from copy import deepcopy
import os, sys, json, pickle, hashlib
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from IPython.display import display as show, Image

PROJECT = ROOT
RESULT = ROOT
CHECK = ROOT
TREE_DIR = ROOT
COMPOSITE_DIR = ROOT
WORK = ROOT
STAGES = ("E9.5", "E10.5", "E11.5", "E13.5", "E14.5", "E15.5", "E16.5", "E17.5", "E18.5")
STAGE_COLORS = ("#f5b888", "#f2a985", "#ee9982", "#e58780", "#cd717e", "#b55c7d", "#914679", "#653175", "#3a1c71")
TAU = .25                    # Accepted uniform complete-branch log1p length scale.
VGN_ROOT_SPAN_SCALE = .5      # Accepted VGN-to-root horizontal display span.
VERIFY_COMPONENT_REBUILD = False # Render archived accepted components without rebuilding.
REBUILD_HEATMAP_COLORS = False # Optional H5 reconstruction; no expression refitting.
(ROOT).mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(ROOT)
os.environ["TEMP"] = str(ROOT)
os.environ["TMP"] = str(ROOT)
# Load binary libraries from the analysis environment before pure-Python PDF fallbacks.
PDF_DEPENDENCIES = ROOT
sys.path.append(str(PDF_DEPENDENCIES))
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "pandas:", pd.__version__)


In [ ]:
"""Rebuild the explicitly conditional CVG graft from archived fitted components."""
from copy import deepcopy
import hashlib
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.special import softmax


def array_hash(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()


def rooted_graph(F, B, root):
    n = B.shape[0]
    parent = np.full(n, -1, int)
    distance = np.full(n, np.nan)
    parent[root] = root
    distance[root] = 0.
    order = [int(root)]
    for u in order:
        for v in np.flatnonzero(B[u]):
            if v == parent[u]:
                continue
            assert parent[v] == -1, 'Graph has a cycle'
            parent[v] = u
            distance[v] = distance[u] + np.linalg.norm(F[:, v] - F[:, u])
            order.append(int(v))
    assert len(order) == n, 'Disconnected graph'
    assert np.count_nonzero(np.triu(B, 1)) == n - 1
    edges = np.array([(int(parent[v]), int(v)) for v in order[1:]], int)
    return parent, distance, edges


def graph_path(B, start, end):
    prev = {int(start): None}
    todo = [int(start)]
    for u in todo:
        for v in np.flatnonzero(B[u]):
            if int(v) not in prev:
                prev[int(v)] = u
                todo.append(int(v))
    path = [int(end)]
    while path[-1] != start:
        path.append(prev[path[-1]])
    return path[::-1]


def nearest_edge_projection(X, F, edges):
    a = F[:, edges[:, 0]].T
    d = F[:, edges[:, 1]].T - a
    den = (d*d).sum(1)
    assert np.all(den > 1e-20)
    alpha = np.clip(np.einsum('ned,ed->ne', X[:, None, :] - a, d)/den, 0., 1.)
    residual = ((X[:, None, :] - a - alpha[:, :, None]*d)**2).sum(2)
    eid = residual.argmin(1)
    return eid, alpha[np.arange(len(X)), eid], np.sqrt(residual[np.arange(len(X)), eid])


def build_stitched(old, local):
    selected_key = tuple(local['local_tree_render']['selected_key'])
    selected = next(r for r in local['local_fit']['records'] if tuple(r['key']) == selected_key)
    assert selected['audit']['natural_Y']
    local_ids = pd.Index(local['discovery']['cvg_ids'])
    ix = old['obs'].index.get_indexer(local_ids)
    assert len(ix) == 478 and np.all(ix >= 0) and len(np.unique(ix)) == 478
    Z = (local['discovery']['cvg_pca'][selected['hvg']] - selected['center'])/selected['global_rms']
    native_F = selected['model']['F']
    local_B = selected['model']['B']
    R = softmax(-cdist(Z, native_F.T)/selected['sigma'], axis=1)
    mapped_F = (R.T @ old['X_css30'][ix]/R.sum(0)[:, None]).T
    native_root = int(selected['audit']['primitive_orientation']['root'])
    native_sgn = int(local['local_tree_render']['matching']['SGN precursor candidate']['tip'])
    native_vgn = int(local['local_tree_render']['matching']['VGN-like CVG candidate']['tip'])
    native_fork = int(np.flatnonzero(local_B.sum(0) > 2)[0])
    old_stem = graph_path(old['model']['B'], old['projection']['root'], old['first_fork'])
    anchor_dist = np.linalg.norm(old['model']['F'][:, old_stem] - mapped_F[:, native_sgn, None], axis=0)
    anchor = int(old_stem[int(anchor_dist.argmin())])
    cut = old_stem[:old_stem.index(anchor)]
    kept = [i for i in range(old['model']['B'].shape[0]) if i not in set(cut)]
    old_map = {u: i for i, u in enumerate(kept)}
    local_map = {u: len(kept)+u for u in range(len(local_B))}
    F = np.concatenate([old['model']['F'][:, kept], mapped_F], axis=1)
    B = np.zeros((F.shape[1], F.shape[1]), np.int8)
    B[:len(kept), :len(kept)] = old['model']['B'][np.ix_(kept, kept)]
    B[len(kept):, len(kept):] = local_B
    B[local_map[native_sgn], old_map[anchor]] = B[old_map[anchor], local_map[native_sgn]] = 1
    root, first_fork = local_map[native_root], old_map[old['first_fork']]
    parent, rd, edges = rooted_graph(F, B, root)
    lookup = {tuple(sorted(e)): i for i, e in enumerate(edges)}
    n = len(old['obs'])
    eid = np.full(n, -1, int)
    alpha = np.full(n, np.nan)
    residual = np.full(n, np.nan)
    component = np.full(n, 'retained_old_projection', object)
    local_mask = np.zeros(n, bool)
    local_mask[ix] = True
    old_edges_for_cells = old['projection']['edges'][old['projection']['edge_id']]
    retained_mask = np.isin(old_edges_for_cells, kept).all(1) & ~local_mask
    # A cell exactly at the retained cut node may be encoded on the removed edge.
    # Canonicalize that zero-distance endpoint to a retained incident edge.
    endpoint_at_anchor = (((old_edges_for_cells[:,0] == anchor) & (old['projection']['alpha'] == 0)) |
                          ((old_edges_for_cells[:,1] == anchor) & (old['projection']['alpha'] == 1)))
    boundary_mask = endpoint_at_anchor & ~retained_mask & ~local_mask
    retained_mask |= boundary_mask
    scoped_mask = ~retained_mask & ~local_mask

    def assign(rows, source_edges, source_alpha, mapping):
        for j, row in enumerate(rows):
            a, b = (mapping[int(u)] for u in source_edges[j])
            new_eid = lookup[tuple(sorted((a, b)))]
            eid[row] = new_eid
            alpha[row] = source_alpha[j] if edges[new_eid, 0] == a else 1-source_alpha[j]

    retained = np.flatnonzero(retained_mask)
    regular_retained = np.flatnonzero(retained_mask & ~boundary_mask)
    assign(regular_retained, old_edges_for_cells[regular_retained], old['projection']['alpha'][regular_retained], old_map)
    boundary_eid = next(i for i,(a,b) in enumerate(edges) if a == old_map[anchor] and b < len(kept))
    eid[boundary_mask] = boundary_eid
    alpha[boundary_mask] = 0.
    residual[retained] = old['projection']['residual'][retained]
    native_projection = selected['audit']['edge_projection']
    assign(ix, native_projection['edges'][native_projection['edge_id']], native_projection['alpha'], local_map)
    component[ix] = 'native_local_edge_and_alpha'
    scoped = np.flatnonzero(scoped_mask)
    allowed = np.flatnonzero(np.isin(edges, list(local_map.values())).any(1))
    scoped_eid, scoped_alpha, scoped_resid = nearest_edge_projection(old['X_css30'][scoped], F, edges[allowed])
    eid[scoped] = allowed[scoped_eid]
    alpha[scoped] = scoped_alpha
    residual[scoped] = scoped_resid
    component[scoped] = 'interface_common_CSS_nearest_edge'
    mapped_proj = (1-alpha[ix, None])*F[:, edges[eid[ix], 0]].T + alpha[ix, None]*F[:, edges[eid[ix], 1]].T
    residual[ix] = np.linalg.norm(old['X_css30'][ix]-mapped_proj, axis=1)
    edge_len = np.linalg.norm(F[:, edges[:, 1]]-F[:, edges[:, 0]], axis=0)
    t = rd[edges[eid, 0]] + alpha*edge_len[eid]
    delta = float(rd[old_map[anchor]]-old['projection']['root_distance'][anchor])

    # Preserve old leaf ordering; add one distinct VGN candidate leaf on the left.
    node_x = {old_map[u]: float(old['node_x'][u]) for u in kept}
    node_x[local_map[native_vgn]] = -1.
    node_x[local_map[native_sgn]] = float(old['node_x'][anchor])
    leaf_ranges = {}
    for u in reversed(list(np.argsort(rd))):
        children = np.flatnonzero(parent == u)
        children = children[children != u]
        if len(children):
            lower = min(leaf_ranges[int(c)][0] for c in children)
            upper = max(leaf_ranges[int(c)][1] for c in children)
        else:
            lower = upper = node_x[int(u)]
        leaf_ranges[int(u)] = (lower,upper)
        if int(u) not in node_x:
            node_x[int(u)] = float((lower+upper)/2)

    # Keep degree-two interface milestones explicit for transparent metric registration.
    segment_specs = [
        (local_map[native_root], local_map[native_fork], 'CVG common', 'early_common'),
        (local_map[native_fork], local_map[native_vgn], 'VGN candidate-enriched arm', 'VGN_candidate'),
        (local_map[native_fork], local_map[native_sgn], 'SGN candidate-enriched arm', 'SGN_candidate'),
        (local_map[native_sgn], old_map[anchor], 'Component connection', 'connector'),
        (old_map[anchor], first_fork, 'SGN common trajectory', 'SGN_common')]
    roles = {(38,96):('Ib/Ic shared trajectory','IbIc'), (38,92):('Ia/II shared trajectory','IaII'),
             (92,63):('Ia terminal segment','Ia'), (92,41):('II terminal segment','II'),
             (96,114):('Ib terminal segment','Ib'), (96,86):('Ic terminal segment','Ic')}
    for s in old['segments'][1:]:
        label, role = roles[(s['start'],s['end'])]
        segment_specs.append((old_map[s['start']], old_map[s['end']], label, role))
    segments, edge_segment = [], np.full(len(edges),-1,int)
    for si,(start,end,label,role) in enumerate(segment_specs):
        path = graph_path(B,start,end)
        eids = np.array([lookup[tuple(sorted(pair))] for pair in zip(path[:-1],path[1:])],int)
        edge_segment[eids] = si
        segments.append(dict(id=si,start=start,end=end,path=path,x=float(node_x[end]),edge_ids=eids,role=role,label=label))
    assert np.all(edge_segment >= 0)
    segment_ids = edge_segment[eid]
    old_sid = old['whole_display_geometry']['segment_ids']
    old_centers = np.array([old['segments'][int(s)]['x'] for s in old_sid])
    old_display_centers = 1.5+.5*(old_centers-1.5)
    offsets = old['whole_display_geometry']['cell_xy'][:,0]-old_display_centers
    centers = np.array([segments[int(s)]['x'] for s in segment_ids])
    xy = np.column_stack([1.5+.5*(centers-1.5)+offsets,t])
    xy[retained,0] = old['whole_display_geometry']['cell_xy'][retained,0]
    old_colors = old['segment_gradient_coloring']['node_colors']
    node_colors = {old_map[k]:v for k,v in old_colors.items() if k in old_map}
    def hex_rgb(h):return np.array([int(h[i:i+2],16) for i in (1,3,5)])
    frac = old['projection']['root_distance'][anchor]/old['projection']['root_distance'][old['first_fork']]
    interface_rgb = np.rint((1-frac)*hex_rgb(old_colors[old['projection']['root']])+frac*hex_rgb(old_colors[old['first_fork']])).astype(int)
    interface_color = '#'+''.join(f'{v:02X}' for v in interface_rgb)
    node_colors.update({root:old_colors[old['projection']['root']],local_map[native_fork]:'#7AA8A4',
                        local_map[native_vgn]:'#5A89AD',local_map[native_sgn]:interface_color,old_map[anchor]:interface_color})
    figures = deepcopy(old['figures'])
    for typ in ('Ia','Ib','Ic','II'):
        f=figures[typ]
        f['start']=old_map[f['start']]; f['tip']=old_map[f['tip']]
        f['node_ids']=[old_map[u] for u in f['node_ids']]
        f['edge_ids']=np.array([lookup[tuple(sorted((old_map[a],old_map[b])))] for a,b in old['projection']['edges'][old['figures'][typ]['edge_ids']]],int)
    source_old_keys=('schema_version','obs','model','projection','X_css30','node_x','segments','plot_order','first_fork','tips','figures','accepted_key','diagnostics','provenance','render_spec','whole_display_geometry','segment_gradient_coloring','path_render_spec')
    out=dict(schema_version='CVG-stitch-1',obs=old['obs'].copy(deep=True),X_css30=old['X_css30'].copy(),
             model=dict(F=F,B=B,parameters=dict(method='conditional component graft',local_selected_key=selected_key,common_space='shared-cell barycentric CSS30 transfer',bridge_nodes=(local_map[native_sgn],old_map[anchor]))),
             projection=dict(root=root,edges=edges,edge_id=eid,alpha=alpha,t=t,root_distance=rd,parent=parent,residual=residual),
             node_x=node_x,segments=segments,plot_order=old['plot_order'].copy(),first_fork=first_fork,tips=[old_map[u] for u in old['tips']],
             vgn_tip=local_map[native_vgn],local_root=root,local_fork=local_map[native_fork],local_sgn_exit=local_map[native_sgn],interface_anchor=old_map[anchor],
             figures=figures,render_spec=dict(status='Awaiting stitched rendering; original templates archived only'),path_render_spec={},
             node_colors=node_colors,old_distance_offset=delta,
             whole_display_geometry=dict(cell_xy=xy,segment_ids=segment_ids,source_coordinates_sha256=array_hash(xy),method='one shared geometry; old retained x unchanged; common CSS geodesic y'),
             local_candidate_calls=local['obs'].loc[local_ids,'candidate_call'].copy(),
             projection_component=pd.Series(component,index=old['obs'].index,name='projection_component'),
             archived_inputs=dict(old={k:deepcopy(old[k]) for k in source_old_keys},local=deepcopy(local)),
             registration=dict(native_F=native_F.copy(),native_B=local_B.copy(),mapped_F_css30=mapped_F,soft_memberships=R,weight_sums=R.sum(0),effective_cell_numbers=R.sum(0)**2/(R*R).sum(0),
                               shared_cell_ids=local_ids,selected_key=selected_key,native_root=native_root,native_fork=native_fork,native_sgn_exit=native_sgn,native_vgn_tip=native_vgn,
                               old_anchor=anchor,old_stem=old_stem,anchor_distance_by_stem_node=dict(zip(old_stem,anchor_dist.tolist())),
                               bridge_css30_length=float(np.linalg.norm(F[:,local_map[native_sgn]]-F[:,old_map[anchor]])),
                               old_to_new=old_map,local_to_new=local_map,removed_old_nodes=cut,kept_old_nodes=kept,
                               native_local_projection=deepcopy(native_projection),scope_interface_cell_ids=old['obs'].index[scoped].copy(),
                               scope_native_cell_ids=local_ids.copy(),scope_retained_cell_ids=old['obs'].index[retained].copy()),
             provenance=dict(method='Registered component graft, not all-cell refitting',
                             annotation_policy='All 7355 original IDs, annotation columns and values unchanged; candidate calls stored separately',
                             projection_policy='478 retain native local edges/alpha; 98 cut-interface nonlocal cells nearest-edge projected in common CSS; 6779 retain old projections',
                             caution='Selected local natural Y occurs in 6/30 fits. Identity/age/library confounding remains. Candidate enrichment does not establish single-cell fate.',
                             source_old_accepted_key=old['accepted_key']))
    assert old['obs'].equals(out['obs']) and out['obs'].index.is_unique and len(out['obs'])==7355
    assert (len(ix),len(scoped),len(retained))==(478,98,6779)
    np.testing.assert_array_equal(F[:,:len(kept)],old['model']['F'][:,kept])
    np.testing.assert_allclose(t[retained],old['projection']['t'][retained]+delta,rtol=0,atol=1e-13)
    np.testing.assert_array_equal(xy[retained,0],old['whole_display_geometry']['cell_xy'][retained,0])
    for typ in ('Ia','Ib','Ic','II'):
        mask=np.isin(eid,figures[typ]['edge_ids'])
        np.testing.assert_array_equal(mask,old['figures'][typ]['mask'])
    assert np.count_nonzero(B.sum(0)>2)==4 and np.count_nonzero(B.sum(0)==1)==6
    out['build_validation']=dict(cells=7355,nodes=len(B),edges=len(edges),branch_points=4,nonroot_tips=5,
                                annotations_unchanged=True,old_downstream_node_coordinates_unchanged=True,
                                native_local_edge_alpha_preserved=True,projection_scope_counts=dict(native_local=478,interface_reprojected=98,retained=6779),
                                old_downstream_t_offset=delta,four_SGN_path_masks_identical=True,
                                terminal_typed={typ:f"{figures[typ]['terminal_typed']}/{figures[typ]['typed_total']}" for typ in ('Ia','Ib','Ic','II')})
    out['registration']['cut_node_endpoint_canonicalized_cell_ids'] = old['obs'].index[boundary_mask].copy()
    return out


In [ ]:
#3. =========== Accepted display transformation
def digest(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()

def compress_complete_branches(bundle, tau=TAU):
    assert tau > 0
    F, B = bundle['model']['F'], bundle['model']['B']
    p = bundle['projection']
    root, parent = int(p['root']), p['parent']
    rd, edges = p['root_distance'], p['edges']
    degree = B.sum(0)
    milestones = set(np.flatnonzero(degree != 2).tolist()) | {root}
    edge_lookup = {tuple(sorted(map(int,e))):i for i,e in enumerate(edges)}
    edge_branch = np.full(len(edges),-1,int)
    node_y = np.full(len(B),np.nan)
    node_y[root] = 0.
    branches = []
    for start in sorted(milestones,key=lambda i:rd[i]):
        children = np.flatnonzero(parent == start)
        children = children[children != start]
        for child in children:
            path = [int(start),int(child)]
            while path[-1] not in milestones:
                nxt = np.flatnonzero(parent == path[-1])
                assert len(nxt) == 1
                path.append(int(nxt[0]))
            end = path[-1]
            eids = np.array([edge_lookup[tuple(sorted(pair))] for pair in zip(path[:-1],path[1:])])
            L = float(np.linalg.norm(F[:,edges[eids,1]]-F[:,edges[eids,0]],axis=0).sum())
            np.testing.assert_allclose(L,rd[end]-rd[start],rtol=0,atol=1e-13)
            new_length = float(tau*np.log1p(L/tau))
            factor = new_length/L
            assert np.isfinite(node_y[start]) and np.all(edge_branch[eids] == -1)
            node_y[path[1:]] = node_y[start]+(rd[path[1:]]-rd[start])*factor
            edge_branch[eids] = len(branches)
            branches.append(dict(id=len(branches),start=int(start),end=end,path=path,edge_ids=eids,
                                 original_length=L,display_length=new_length,factor=factor,
                                 display_start=float(node_y[start]),display_end=float(node_y[end])))
    assert len(branches)==9 and np.all(edge_branch>=0) and np.isfinite(node_y).all()
    eid,alpha,t = p['edge_id'],p['alpha'],p['t']
    cell_branch = edge_branch[eid]
    cell_y = (1-alpha)*node_y[edges[eid,0]]+alpha*node_y[edges[eid,1]]
    independent_y = np.array([v['display_start'] for v in branches])[cell_branch] + (
        t-np.array([rd[v['start']] for v in branches])[cell_branch])*np.array([v['factor'] for v in branches])[cell_branch]
    np.testing.assert_allclose(cell_y,independent_y,rtol=0,atol=1e-13)
    assert np.min(cell_y)>=-1e-14
    for branch in branches:
        values=cell_y[cell_branch==branch['id']]
        assert values.min()>=branch['display_start']-1e-13 and values.max()<=branch['display_end']+1e-13
    metric_to_branch=[]
    for segment in bundle['segments']:
        ids=np.unique(edge_branch[segment['edge_ids']])
        assert len(ids)==1
        metric_to_branch.append(int(ids[0]))
    assert metric_to_branch[2]==metric_to_branch[3]==metric_to_branch[4]
    xy=bundle['whole_display_geometry']['cell_xy'].copy()
    xy[:,1]=cell_y
    return dict(method='L_display=tau*log1p(L_CSS/tau); maximal root/fork/tip chains; within-chain linear arc-length map',
                tau_css30=tau,branches=branches,edge_branch=edge_branch,cell_branch=cell_branch,
                metric_segment_to_complete_branch=metric_to_branch,node_y=node_y,cell_xy=xy,
                original_t_sha256=digest(t),display_xy_sha256=digest(xy),
                original_root_distance_sha256=digest(rd),original_model_F_sha256=digest(F),original_model_B_sha256=digest(B))

def adjust_vgn_spacing(bundle, display, span_scale=VGN_ROOT_SPAN_SCALE):
    assert 0 < span_scale <= 1
    node_x={int(k):float(v) for k,v in bundle['node_x'].items()}
    root=int(bundle['projection']['root']);vg=int(bundle['vgn_tip'])
    branch=next(b for b in display['branches'] if b['end']==vg)
    raw_shift=(1-span_scale)*(node_x[root]-node_x[vg])
    display_shift=.5*raw_shift  # Existing raw-to-display x scale.
    moved_nodes=branch['path'][1:]
    for node in moved_nodes:node_x[node]+=raw_shift
    segment_x=np.array([float(s['x']) for s in bundle['segments']])
    moved_segments=[i for i,s in enumerate(bundle['segments']) if s['role']=='VGN_candidate']
    assert len(moved_segments)==1
    segment_x[moved_segments]+=raw_shift
    mask=display['cell_branch']==branch['id']
    display['cell_xy'][mask,0]+=display_shift
    np.testing.assert_array_equal(display['cell_xy'][~mask,0],bundle['whole_display_geometry']['cell_xy'][~mask,0])
    old_span=.5*(bundle['node_x'][root]-bundle['node_x'][vg])
    new_span=.5*(node_x[root]-node_x[vg])
    assert abs(new_span/old_span-span_scale)<1e-14 and int(mask.sum())==175
    display.update(node_x=node_x,segment_x=segment_x,display_xy_sha256=digest(display['cell_xy']),
                   horizontal_layout=dict(method='Translate the complete VGN arm toward the fixed root',
                       VGN_root_span_scale=span_scale,original_span=old_span,new_span=new_span,
                       displayed_x_shift=display_shift,raw_x_shift=raw_shift,
                       moved_node_ids=moved_nodes,moved_cell_mask=mask,moved_cells=int(mask.sum()),
                       moved_metric_segment_ids=moved_segments,root_node_fixed=root))
    return display

def stage_density(bundle, display):
    grid_size = 768
    old_h = float(bundle['stage_density']['bandwidth_css30'])
    limit = float(display['node_y'].max())
    h = old_h*limit/float(bundle['projection']['root_distance'].max())
    grid = np.linspace(0,limit,grid_size)
    values,counts={},{}
    for stage in STAGES:
        y=display['cell_xy'][bundle['obs'].stage_standardized.eq(stage).to_numpy(),1]
        centers=np.r_[y,-y,2*limit-y]
        value=np.exp(-.5*((grid[:,None]-centers[None,:])/h)**2).sum(1)/(len(y)*h*np.sqrt(2*np.pi))
        value/=np.trapezoid(value,grid)
        values[stage],counts[stage]=value,int(len(y))
        assert abs(np.trapezoid(value,grid)-1)<1e-12
    return dict(grid=grid,values=values,counts=counts,bandwidth_display=h,
                bandwidth_rule='Original CSS bandwidth times new/old root-distance span',
                original_bandwidth_css30=old_h,display_values={s:v/v.max() for s,v in values.items()},
                method='Gaussian KDE on transformed cell y, reflected at 0 and maximum node y; each stage unit area; displayed peak=1')

In [ ]:
"""Read frozen expression results and package the accepted display without refitting."""
from pathlib import Path
import io
import json
import hashlib
from copy import deepcopy
import numpy as np
import pandas as pd


def sha_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def array_sha(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()


def load_heatmaps(project, bundle):
    import h5py
    from matplotlib import colormaps, colors
    project=Path(project)
    source=ROOT
    contract_path=ROOT
    contract=json.loads(contract_path.read_text(encoding='utf-8'))
    assert sha_file(source)==contract['h5_sha256']
    stage_palette=dict(zip(contract['render']['stages'],contract['render']['stage_colors']))
    titles={'C':'CVG to Uns.','D':'Ia path','E':'Ib path','F':'Ic-associated path','G':'II path'}
    fates={'D':'Ia','E':'Ib','F':'Ic','G':'II'}
    result={}
    with h5py.File(source,'r') as store:
        def text(key):return store[key][()].tobytes().decode('utf-8')
        def table(key):return pd.read_csv(io.StringIO(text(key)),sep='\t' if key.endswith('.tsv') else ',',keep_default_na=False)
        labels_all=table('global/Labels.tsv')
        for panel in 'CDEFG':
            keys=[f'panels/{panel}/fitted',f'panels/{panel}/stable_cell_order',
                  f'panels/{panel}/tables/displayed-genes.tsv',f'panels/{panel}/tables/fitted-genes.tsv',
                  f'panels/{panel}/tables/cells.csv',f'panels/{panel}/tables/figure-qa.json','global/Labels.tsv']
            for key in keys:
                value=store[key][()];expected=contract['dataset_manifest'][key]
                assert list(value.shape)==expected['shape'] and value.dtype.str==expected['dtype']
                assert hashlib.sha256(value.tobytes(order='C')).hexdigest()==expected['sha256'],key
            genes=table(f'panels/{panel}/tables/displayed-genes.tsv').sort_values('display_row',kind='stable').reset_index(drop=True)
            fit_genes=table(f'panels/{panel}/tables/fitted-genes.tsv')
            source_obs=table(f'panels/{panel}/tables/cells.csv')
            order=store[f'panels/{panel}/stable_cell_order'][()]
            np.testing.assert_array_equal(order,np.argsort(source_obs.t.to_numpy(),kind='stable'))
            obs=source_obs.iloc[order].reset_index(drop=True)
            rows=pd.Index(fit_genes.gene_id).get_indexer(genes.gene_id)
            assert np.all(rows>=0) and not genes.gene_id.duplicated().any()
            fitted=store[f'panels/{panel}/fitted'][()][np.ix_(rows,order)]
            minimum=fitted.min(1);maximum=fitted.max(1)
            np.testing.assert_allclose(minimum,genes.color_min.to_numpy(float),atol=1e-12,rtol=0)
            np.testing.assert_allclose(maximum,genes.color_max.to_numpy(float),atol=1e-12,rtol=0)
            normalized=(fitted-minimum[:,None])/(maximum-minimum)[:,None]
            qa=json.loads(text(f'panels/{panel}/tables/figure-qa.json'))
            bounds=np.percentile(normalized,[2,98])
            np.testing.assert_allclose(bounds,qa['color_limits'],atol=1e-12,rtol=0)
            rgb=colormaps['RdBu_r'](colors.Normalize(*bounds)(normalized),bytes=True)[...,:3]
            origin=float(contract['panels'][panel]['path_origin'])
            obs['path_t']=obs.t-origin
            pt_limits=[float(np.percentile(obs.path_t,2)),float(obs.path_t.max())]
            pt_rgb=colormaps[contract['render']['pseudotime_cmaps'][panel]](
                colors.Normalize(*pt_limits,clip=True)(obs.path_t.to_numpy()),bytes=True)[...,:3]
            age_rgb=np.rint(np.array([colors.to_rgb(stage_palette[s]) for s in obs.stage])*255).astype(np.uint8)
            selected=labels_all.loc[labels_all.panel==panel].copy()
            labels=genes[['gene_id','symbol','display_row']].merge(
                selected.drop(columns='symbol'),on='gene_id',how='inner',validate='one_to_one').sort_values('display_row').reset_index(drop=True)
            assert labels.symbol.tolist()==qa['labels'] and set(labels.symbol)==set(contract['panels'][panel]['labels'])
            assert len(genes)==contract['panels'][panel]['displayed_genes']
            assert obs.cell_id.is_unique and len(obs)==contract['panels'][panel]['cells']
            indices=bundle['obs'].index.get_indexer(obs.cell_id)
            assert np.all(indices>=0)
            tree_mask=np.zeros(len(bundle['obs']),bool);tree_mask[indices]=True
            if panel!='C':np.testing.assert_array_equal(tree_mask,bundle['figures'][fates[panel]]['mask'])
            tree_rgb=np.full((len(bundle['obs']),3),255,np.uint8);tree_rgb[indices]=pt_rgb
            if panel=='C':
                np.testing.assert_array_equal(tree_mask,np.isin(bundle['whole_display_geometry']['segment_ids'],range(5)))
            result[panel]=dict(title=titles[panel],genes=genes,labels=labels,obs=obs,rgb=np.ascontiguousarray(rgb),
                normalized=normalized,stage_strip_rgb=age_rgb,pseudotime_strip_rgb=pt_rgb,
                color_limits=bounds.tolist(),pt_limits=pt_limits,pseudotime_cmap=contract['render']['pseudotime_cmaps'][panel],
                tree_mask=tree_mask,tree_rgb=tree_rgb,cells=len(obs),genes_count=len(genes),
                source_original_pseudotime=True,source_qa=qa,
                source_fitted_subset_sha256=array_sha(fitted),normalized_sha256=array_sha(normalized),rgb_sha256=array_sha(rgb),
                source_gene_order_sha256=hashlib.sha256('\n'.join(genes.gene_id).encode()).hexdigest(),
                source_cell_order_sha256=hashlib.sha256('\n'.join(obs.cell_id).encode()).hexdigest(),
                note='Original early-cell set; expression fitted on original pseudotime' if panel=='C' else 'Original selected cells and expression fits; only the tree display is transformed')
    return result,dict(source_h5=str(source),source_h5_sha256=contract['h5_sha256'],source_validation=str(contract_path),
                       source_validation_sha256=sha_file(contract_path),mapping={'C':'Common early cells','D':'Ia','E':'Ib','F':'Ic','G':'II'},
                       heatmap_policy='Only the expression matrix is an opaque RGB image; all other publication elements are vector/text',
                       label_policy='Gene symbols only, with one editable ArialItalic text object per gene; evidence retained in result tables')


def prepare_publication(project, accepted, protected_hashes):
    """The accepted snapshot is read once; the published package is self-contained."""
    base=deepcopy(accepted['source_bundle'])
    display=deepcopy(accepted['display'])
    density=deepcopy(accepted['stage_density'])
    np.testing.assert_array_equal(display['cell_xy'][:,1],accepted['display']['cell_xy'][:,1])
    assert display['tau_css30']==.25 and display['horizontal_layout']['VGN_root_span_scale']==.5
    heatmaps,provenance=load_heatmaps(project,base)
    base['linear_display_geometry']=base['whole_display_geometry']
    base['whole_display_geometry']=dict(cell_xy=display['cell_xy'].copy(),
        segment_ids=base['linear_display_geometry']['segment_ids'].copy(),
        method=display['method'],source_coordinates_sha256=display['display_xy_sha256'],
        horizontal_layout=deepcopy(display['horizontal_layout']))
    base['publication_display']=display
    base['stage_density_display']=density
    base['heatmaps']=heatmaps
    base['heatmap_publication_provenance']=provenance
    base['publication_protected_file_sha256']=protected_hashes
    base['publication_status']='Accepted tau0.25 and VGN-root span0.5; original graph, annotations and expression fits preserved'
    base['accepted_display_snapshot']=dict(node_y_sha256=array_sha(display['node_y']),
        cell_xy_sha256=array_sha(display['cell_xy']),source_F_sha256=array_sha(base['model']['F']),
        source_B_sha256=array_sha(base['model']['B']),original_t_sha256=array_sha(base['projection']['t']))
    # Retain historical templates only in the archived inputs, not as current verification.
    for stale in ('render_stitch','render_spec_stitch','path_render_spec'):
        base.pop(stale,None)
    base['render_spec']={'status':'Pending publication rendering','method':'Accepted display geometry'}
    return base


In [ ]:
#5. =========== Read the self-contained accepted result package
bundle = pickle.load(RESULT.open("rb"))
assert "publication_display" in bundle and "heatmaps" in bundle
for path, expected in bundle["publication_protected_file_sha256"].items():
    assert sha_file(path) == expected, f"Protected original changed: {path}"
if VERIFY_COMPONENT_REBUILD:
    rebuilt = build_stitched(**bundle["archived_inputs"])
    for key in ("F", "B"):
        np.testing.assert_array_equal(rebuilt["model"][key], bundle["model"][key])
    for key in ("edges", "edge_id", "alpha", "t", "root_distance"):
        np.testing.assert_array_equal(rebuilt["projection"][key], bundle["projection"][key])
linear_input = dict(bundle)
linear_input["whole_display_geometry"] = bundle["linear_display_geometry"]
accepted_display = adjust_vgn_spacing(linear_input, compress_complete_branches(linear_input, TAU), VGN_ROOT_SPAN_SCALE)
assert digest(accepted_display["cell_xy"]) == bundle["accepted_display_snapshot"]["cell_xy_sha256"]
assert digest(accepted_display["node_y"]) == bundle["accepted_display_snapshot"]["node_y_sha256"]
bundle["publication_display"] = accepted_display
bundle["stage_density_display"] = stage_density(linear_input, accepted_display)
if REBUILD_HEATMAP_COLORS:
    rebuilt_heatmaps, provenance = load_heatmaps(PROJECT, linear_input)
    for panel in "CDEFG":
        for key in ("rgb", "stage_strip_rgb", "pseudotime_strip_rgb", "tree_mask", "tree_rgb"):
            np.testing.assert_array_equal(rebuilt_heatmaps[panel][key], bundle["heatmaps"][panel][key])
        assert rebuilt_heatmaps[panel]["obs"].cell_id.tolist() == bundle["heatmaps"][panel]["obs"].cell_id.tolist()
        assert rebuilt_heatmaps[panel]["genes"].gene_id.tolist() == bundle["heatmaps"][panel]["genes"].gene_id.tolist()
print("Accepted components and display verified. Heatmap row/cell order and fits retained.")


In [ ]:
"""Publish the accepted branch-wise log1p display as five editable tree figures.

This module only renders caller-supplied data. It never fits, reprojects, relabels,
rescales branches, or changes the accepted display geometry. Importing it writes
nothing; ``render_trees`` is the explicit export entry point.
"""
from pathlib import Path
import hashlib
import math
import os
import subprocess
import sys

import numpy as np


FATES = ("Ia", "Ib", "Ic", "II")
STAGES = ("E9.5", "E10.5", "E11.5", "E13.5", "E14.5", "E15.5", "E16.5", "E17.5", "E18.5")
STAGE_COLORS = ("#f5b888", "#f2a985", "#ee9982", "#e58780", "#cd717e", "#b55c7d", "#914679", "#653175", "#3a1c71")
RAMPS = {
    "Ia": ("#fff5f0", "#fee0d2", "#fcbba1", "#fc9272", "#fb6a4a", "#ef3b2c", "#cb181d", "#a50f15", "#67000d"),
    "Ib": ("#f7fbff", "#deebf7", "#c6dbef", "#9ecae1", "#6baed6", "#4292c6", "#2171b5", "#08519c", "#08306b"),
    "Ic": ("#f7fcf5", "#e5f5e0", "#c7e9c0", "#a1d99b", "#74c476", "#41ab5d", "#238b45", "#006d2c", "#00441b"),
    "II": ("#fcfbfd", "#efedf5", "#dadaeb", "#bcbddc", "#9e9ac8", "#807dba", "#6a51a3", "#54278f", "#3f007d"),
}


def _array_sha(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()


def _hex_rgb(value):
    return np.array([int(value[j:j + 2], 16) for j in (1, 3, 5)], float) / 255.


def _ramp_lut(fate, fractions):
    """Match LinearSegmentedColormap.from_list(..., N=1000), including indexing."""
    fractions = np.asarray(fractions, float)
    index = np.minimum((np.clip(fractions, 0., 1.) * 1000).astype(int), 999)
    at = index / 999. * 8.
    low = np.minimum(at.astype(int), 7)
    weight = at - low
    stops = np.array([_hex_rgb(s) for s in RAMPS[fate]])
    return stops[low] * (1. - weight[..., None]) + stops[low + 1] * weight[..., None]


def render_trees(bundle, display, density, outdir, work):
    """Export all/Ia/Ib/Ic/II PDFs and 600-DPI PNGs from one accepted display.

    PDF geometry is entirely vector, with live embedded Arial text, no clipping,
    no Form XObjects, and no images. PNGs are rendered directly by Poppler from
    these PDFs. Structural certification and visual review are caller steps.
    """
    runtime = ROOT
    try:
        import reportlab  # noqa: F401
    except ImportError:
        sys.path.append(str(runtime / "python/Lib/site-packages"))
    from reportlab.pdfgen import canvas
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.lib.colors import Color, HexColor
    from PIL import Image
    from matplotlib.ticker import MaxNLocator

    outdir, work = Path(outdir), Path(work)
    outdir.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    pdftoppm = Path(os.environ.get("TREE_PDFTOPPM", str(runtime / "native/poppler/Library/bin/pdftoppm.exe")))
    assert pdftoppm.is_file(), pdftoppm
    for name, filename in (("Arial", "arial.ttf"), ("Arial-Bold", "arialbd.ttf")):
        if name not in pdfmetrics.getRegisteredFontNames():
            fontfile = ROOT / filename
            assert fontfile.is_file(), fontfile
            pdfmetrics.registerFont(TTFont(name, str(fontfile)))

    DPI, POINT_AREA, WHOLE_ALPHA, PATH_ALPHA = 600, 1.4, .72, .88
    WHOLE_SIZE, PATH_SIZE = (10.8, 8.5), (7., 8.5)
    AXES_WIDTH, AXES_HEIGHT, AXES_BOTTOM = 2.8 * 72, .57 * 8.5 * 72, .205 * 8.5 * 72
    XMIN, XMAX, X_CENTER, X_SCALE = -.5, 3.5, 1.5, .5
    obs, projection, segments = bundle["obs"], bundle["projection"], bundle["segments"]
    nx = {int(k): float(v) for k, v in display["node_x"].items()}
    sx = np.asarray(display["segment_x"], float)
    ny = np.asarray(display["node_y"], float)
    xy = np.asarray(display["cell_xy"], float)
    original_xy = xy.copy()
    rd, times = np.asarray(projection["root_distance"], float), np.asarray(projection["t"], float)
    maximum = float(ny.max())
    ymin, ymax = -.045 * maximum, 1.32 * maximum
    order = np.asarray(bundle["plot_order"], int)
    coloring = bundle["segment_gradient_coloring"]
    rgb_values = np.asarray(coloring["color_rgb_uint8"], np.uint8)
    palettes = dict(zip(STAGES, STAGE_COLORS))
    stage_values = obs.stage_standardized.astype(str).to_numpy()
    assert xy.shape == (len(obs), 2) and np.isfinite(xy).all()
    assert len(ny) == len(rd) and len(sx) == len(segments)
    assert np.array_equal(np.sort(order), np.arange(len(obs)))
    assert rgb_values.shape == (len(obs), 3)
    assert float(display["tau_css30"]) == .25
    assert display["horizontal_layout"]["VGN_root_span_scale"] == .5
    assert sum(int(s["cells"]) for s in coloring["segments"]) == len(obs)
    ticks = MaxNLocator(nbins=6).tick_values(0., maximum)
    ticks = ticks[(ticks >= 0.) & (ticks <= maximum * 1.05)]
    artifacts, scenes, physical = {}, {}, {}
    radius = math.sqrt(POINT_AREA) / 2.

    def color(value):
        if isinstance(value, str):
            return HexColor(value)
        return Color(*map(float, np.asarray(value)[:3]))

    def dx(raw):
        return X_CENTER + X_SCALE * (np.asarray(raw) - X_CENTER)

    def cell_map(left, x, y):
        return (left * 72 + AXES_WIDTH * (np.asarray(x) - XMIN) / (XMAX - XMIN),
                AXES_BOTTOM + AXES_HEIGHT * (ymax - np.asarray(y)) / (ymax - ymin))

    def node_map(left, raw_x, y):
        return cell_map(left, dx(raw_x), y)

    def draw_text(c, value, x, y, size=8., font="Arial", fill="#252525", align="left",
                  valign="baseline", angle=0., role="label", leading=None, panel=None):
        """One semantic BT/ET object, with an auditable physical bounding box."""
        value = str(value)
        lines = value.split("\n")
        leading = 1.2 * size if leading is None else float(leading)
        widths = [pdfmetrics.stringWidth(s, font, size) for s in lines]
        asc, desc = pdfmetrics.getAscentDescent(font, size)
        if valign == "top":
            y -= asc
        elif valign == "center":
            y -= (asc + desc - (len(lines) - 1) * leading) / 2.
        elif valign != "baseline":
            raise ValueError(valign)
        theta = math.radians(angle)
        co, si = math.cos(theta), math.sin(theta)
        text = c.beginText()
        text.setFont(font, size)
        text.setFillColor(color(fill))
        for i, (string, width) in enumerate(zip(lines, widths)):
            offset = 0. if align == "left" else (-width / 2. if align == "center" else -width)
            text.setTextTransform(co, si, -si, co, x + offset * co + i * leading * si,
                                  y + offset * si - i * leading * co)
            text.textOut(string)
        c.drawText(text)
        width = max(widths)
        left = 0. if align == "left" else (-width / 2. if align == "center" else -width)
        corners = np.array([[left, desc - (len(lines) - 1) * leading], [left, asc],
                            [left + width, desc - (len(lines) - 1) * leading], [left + width, asc]])
        corners = corners @ np.array([[co, si], [-si, co]]) + [x, y]
        c._scene.append(dict(text=value, role=role, panel=panel, font=font, fontsize_pt=float(size),
                             bbox=[float(corners[:, 0].min()), float(corners[:, 1].min()),
                                   float(corners[:, 0].max()), float(corners[:, 1].max())]))

    def line(c, x0, y0, x1, y1, fill="#626971", width=1.):
        c.setStrokeColor(color(fill))
        c.setLineWidth(width)
        c.line(float(x0), float(y0), float(x1), float(y1))

    def circle(c, x, y, r, fill, stroke=None, width=.5):
        k = r * 4 * (math.sqrt(2) - 1) / 3
        path = c.beginPath()
        path.moveTo(x + r, y)
        path.curveTo(x + r, y + k, x + k, y + r, x, y + r)
        path.curveTo(x - k, y + r, x - r, y + k, x - r, y)
        path.curveTo(x - r, y - k, x - k, y - r, x, y - r)
        path.curveTo(x + k, y - r, x + r, y - k, x + r, y)
        path.close()
        c.setFillColor(color(fill))
        if stroke is not None:
            c.setStrokeColor(color(stroke))
            c.setLineWidth(width)
        c.drawPath(path, fill=1, stroke=int(stroke is not None))

    def axes(c, left, title="", panel="left"):
        x0, y0 = left * 72, AXES_BOTTOM
        for tick in ticks:
            _, yp = cell_map(left, 0., tick)
            line(c, x0, yp, x0 + AXES_WIDTH, yp, "#e8e8e8", .45)
            line(c, x0 - 3., yp, x0, yp, "#6c7074", .6)
            draw_text(c, f"{tick:.1f}", x0 - 6.5, yp, 7.5, align="right", valign="center", role="tick", panel=panel)
        line(c, x0, y0, x0, y0 + AXES_HEIGHT, "#6c7074", .7)
        draw_text(c, "Branch-wise log1p display distance", x0 - 33, y0 + AXES_HEIGHT / 2.,
                  8.5, align="center", valign="center", angle=90., role="axis", panel=panel)
        if title:
            draw_text(c, title, x0 + AXES_WIDTH / 2., y0 + AXES_HEIGHT + 17.,
                      11.5, align="center", role="panel_title", panel=panel)

    def skeleton(c, left, highlighted=False):
        fill = "#b4b4b4" if highlighted else "#626971"
        for k, seg in enumerate(segments):
            a, z, raw_x = int(seg["start"]), int(seg["end"]), sx[k]
            xa, ya = node_map(left, raw_x, ny[a])
            xz, yz = node_map(left, raw_x, ny[z])
            line(c, xa, ya, xz, yz, fill, 1.)
            if nx[a] != raw_x:
                x0, y0 = node_map(left, nx[a], ny[a])
                line(c, x0, y0, xa, ya, fill, 1.)

    def local_labels(c, left, compact=False, panel="left"):
        root, vg, sg = int(projection["root"]), int(bundle["vgn_tip"]), int(bundle["local_sgn_exit"])
        xp, yp = node_map(left, nx[root], ny[root])
        circle(c, xp, yp, math.sqrt(8.) / 2., "#ffffff", "#404040", .6)
        tx, ty = node_map(left, nx[root], ny[root] - .042 * maximum)
        draw_text(c, "Root", tx, ty, 7.2, align="center", valign="top", role="endpoint", panel=panel)
        xp, yp = node_map(left, nx[vg], ny[vg])
        circle(c, xp, yp, math.sqrt(8.) / 2., "#ffffff", "#555555", .6)
        tx, ty = node_map(left, nx[vg], ny[vg] + .020 * maximum)
        draw_text(c, "VGN candidate-\nenriched", tx, ty, 6.8, align="center", valign="top", role="endpoint", panel=panel)
        if not compact:
            tx, ty = cell_map(left, float(dx(nx[sg])) + .16, ny[sg] - .012 * maximum)
            draw_text(c, "SGN candidate route", tx, ty, 6.5, valign="top", fill="#62686e", panel=panel)

    def whole_endpoints(c, left, panel):
        rows = {"Ib": (.28, .07), "Ic": (1.05, .15), "Ia": (2.10, .23), "II": (2.95, .07)}
        names = {"Ia": "Ia-associated end", "Ib": "Ib-enriched end", "Ic": "Emerging Ic-enriched end", "II": "II-associated end"}
        for fate in FATES:
            item = bundle["figures"][fate]
            tip = int(item["tip"])
            xp, yp = node_map(left, nx[tip], ny[tip])
            circle(c, xp, yp, math.sqrt(8.) / 2., "#ffffff", "#404040", .6)
            lx, pad = rows[fate]
            tx, ty = cell_map(left, lx, maximum * (1. + pad))
            # Matplotlib scales the accepted (2, 3) dash pattern by lw=.4.
            c.setDash(.8, 1.2)
            line(c, xp, yp, tx, ty, "#999999", .4)
            c.setDash()
            draw_text(c, f'{names[fate]}\nImmature {fate}: {item["terminal_typed"]}/{item["typed_total"]}',
                      tx, ty, 6.8, align="center", valign="top", role="endpoint", panel=panel)

    def footer(c, width, extra):
        height = 8.5 * 72
        draw_text(c, "Display transform only: original fitted graph, cell assignments, annotations and pseudotime retained.",
                  .065 * width, .075 * height, 7.)
        draw_text(c, "Complete root/fork/tip branches use one tau; all five views share identical transformed coordinates.",
                  .065 * width, .045 * height, 7., fill="#555a61")
        draw_text(c, extra, .065 * width, .017 * height, 7., fill="#555a61")

    def new_canvas(key, inches):
        path = outdir / f"tree_{key}.pdf"
        c = canvas.Canvas(str(path), pagesize=tuple(np.array(inches) * 72), pageCompression=1,
                          invariant=1, initialFontName="Arial", initialFontSize=8)
        c.setTitle(f"Branch-wise log1p tree: {key}")
        c.setAuthor("Developmental map of auditory neurons")
        c.setSubject("Accepted display transform; original graph and pseudotime retained; live Arial and vector geometry")
        c._scene = []
        return c

    def physical_record(key):
        px, py = cell_map(.62, xy[:, 0], xy[:, 1])
        nodes = np.arange(len(ny))
        gx, gy = node_map(.62, np.array([nx[int(k)] for k in nodes]), ny)
        return dict(cell_xy_pt_sha256=_array_sha(np.column_stack([px, py])),
                    node_xy_pt_sha256=_array_sha(np.column_stack([gx, gy])),
                    axes_left_inches=.62, axes_bottom_inches=AXES_BOTTOM / 72,
                    axes_width_inches=AXES_WIDTH / 72, axes_height_inches=AXES_HEIGHT / 72,
                    xlim=[XMIN, XMAX], ylim=[ymin, ymax], point_diameter_pt=2 * radius,
                    node_paper_coordinates={str(k): [float(gx[k]), float(gy[k])] for k in nodes})

    def finish(c, key, inches):
        width, height = np.array(inches) * 72
        for item in c._scene:
            a, b, z, q = item["bbox"]
            assert a >= -.1 and b >= -.1 and z <= width + .1 and q <= height + .1, ("Text outside page", key, item)
        end_labels = [item for item in c._scene if item["role"] == "endpoint"]
        for j, a in enumerate(end_labels):
            for b in end_labels[j + 1:]:
                x0, y0, x1, y1 = a["bbox"]
                a0, b0, a1, b1 = b["bbox"]
                assert x1 <= a0 or a1 <= x0 or y1 <= b0 or b1 <= y0, ("Endpoint overlap", key, a, b)
        scenes[key] = c._scene
        physical[key] = physical_record(key)
        if key != "all":
            assert physical[key] == physical["all"], "Five views must share the same physical tree geometry"
        c.showPage()
        c.save()
        pdf, png = outdir / f"tree_{key}.pdf", outdir / f"tree_{key}.png"
        subprocess.run([str(pdftoppm), "-singlefile", "-r", str(DPI), "-png", str(pdf), str(png.with_suffix(""))],
                       check=True, capture_output=True)
        with Image.open(png) as image:
            expected_pixels = tuple(round(s * DPI) for s in inches)
            assert image.size == expected_pixels, (key, image.size, expected_pixels)
            actual_dpi = list(map(float, image.info["dpi"]))
            assert all(abs(v - DPI) < .05 for v in actual_dpi)
            image.verify()
        artifacts[key] = dict(pdf=str(pdf), png=str(png), inches=list(inches), pixels=list(expected_pixels), dpi=actual_dpi,
                              pdf_sha256=hashlib.sha256(pdf.read_bytes()).hexdigest(),
                              png_sha256=hashlib.sha256(png.read_bytes()).hexdigest(),
                              expected_clips=0, expected_images=0, expected_forms=0,
                              semantic_text_objects=len(scenes[key]))

    def density_panel(c):
        left, width = 7.10 * 72, .78 * 72
        _, yy = cell_map(.62, 0., np.asarray(density["grid"]))
        for stage in reversed(STAGES):
            xx = left + width * np.asarray(density["display_values"][stage])
            path = c.beginPath()
            path.moveTo(left, float(yy[0]))
            for x, y in zip(xx, yy):
                path.lineTo(float(x), float(y))
            path.lineTo(left, float(yy[-1]))
            path.close()
            c.setFillColor(color(palettes[stage]))
            c.setFillAlpha(.20)
            c.drawPath(path, fill=1, stroke=0)
            c.setFillAlpha(1.)
            path = c.beginPath()
            path.moveTo(float(xx[0]), float(yy[0]))
            for x, y in zip(xx[1:], yy[1:]):
                path.lineTo(float(x), float(y))
            c.setStrokeColor(color(palettes[stage]))
            c.setLineWidth(.7)
            c.drawPath(path, fill=0, stroke=1)
        line(c, left, AXES_BOTTOM, left + width, AXES_BOTTOM, "#6c7074", .7)
        for v in (0, 1):
            x = left + width * v
            line(c, x, AXES_BOTTOM, x, AXES_BOTTOM - 2, "#6c7074", .6)
            draw_text(c, str(v), x, AXES_BOTTOM - 11, 6.8, align="center", role="tick", panel="density")
        draw_text(c, "Relative density", left + width / 2, AXES_BOTTOM - 23, 7., align="center", role="axis", panel="density")
        draw_text(c, "Stage distribution", left + width / 2, AXES_BOTTOM + AXES_HEIGHT + 17, 9., align="center", role="panel_title", panel="density")
        draw_text(c, "Each peak = 1", left + width / 2, AXES_BOTTOM + 1.017 * AXES_HEIGHT, 6.5, align="center", role="legend", panel="density")

    def whole_legend(c):
        left, width = 8.15 * 72, 2.5 * 72
        def xx(x): return left + width * x
        def yy(y): return AXES_BOTTOM + AXES_HEIGHT * y
        draw_text(c, "Tree segment (n)", xx(0), yy(1.035), 8.5, "Arial-Bold", role="legend", panel="legend")
        draw_text(c, "Original within-segment color gradients", xx(0), yy(1.004), 6.3, role="legend", panel="legend")
        y = .95
        for k in coloring["legend_order"]:
            row, seg = coloring["segments"][k], segments[k]
            if seg["role"] == "connector":
                line(c, xx(0), yy(y), xx(.09), yy(y), "#8b8b8b", 1.)
            else:
                a, z = _hex_rgb(row["start_color"]), _hex_rgb(row["end_color"])
                c.setFillAlpha(WHOLE_ALPHA)
                for j in range(256):
                    c.setFillColor(color(a * (1 - j / 255.) + z * j / 255.))
                    c.rect(xx(.09 * j / 256), yy(y - .008), .09 * width / 256 + .005,
                           .016 * AXES_HEIGHT, fill=1, stroke=0)
                c.setFillAlpha(1.)
                c.setStrokeColor(color("#333333"))
                c.setLineWidth(.4)
                c.rect(xx(0), yy(y - .008), .09 * width, .016 * AXES_HEIGHT, fill=0, stroke=1)
            draw_text(c, f'{row["label"]} ({row["cells"]:,})', xx(.13), yy(y), 6.5,
                      valign="center", role="legend", panel="legend")
            y -= .047
        draw_text(c, "n = cells on each original metric segment", xx(0), yy(y - .003), 6.2, role="legend", panel="legend")
        y -= .08
        draw_text(c, "Developmental stage (n)", xx(0), yy(y), 8.1, "Arial-Bold", role="legend", panel="legend")
        y -= .045
        for stage in STAGES:
            circle(c, xx(.018), yy(y + .004), math.sqrt(20) / 2, palettes[stage])
            draw_text(c, f'{stage} ({density["counts"][stage]:,})', xx(.10), yy(y), 7.,
                      valign="center", role="legend", panel="legend")
            y -= .027
        assert y > 0

    # The whole view uses original per-cell segment RGB, including the same alpha.
    c = new_canvas("all", WHOLE_SIZE)
    width, height = np.array(WHOLE_SIZE) * 72
    for left, mode, title in ((.62, "segment", "Segment pseudotime"), (3.95, "stage", "Developmental stage")):
        panel = mode
        axes(c, left, title, panel)
        skeleton(c, left)
        c.setFillAlpha(WHOLE_ALPHA)
        for idx in order:
            xp, yp = cell_map(left, xy[idx, 0], xy[idx, 1])
            fill = rgb_values[idx] / 255. if mode == "segment" else palettes[stage_values[idx]]
            circle(c, float(xp), float(yp), radius, fill)
        c.setFillAlpha(1.)
        local_labels(c, left, panel=panel)
        whole_endpoints(c, left, panel)
    density_panel(c)
    whole_legend(c)
    draw_text(c, "Branch-wise log1p tree", .05 * width, .944 * height, 17., "Arial-Bold", role="title")
    draw_text(c, f'{len(obs):,} cells | {len(display["branches"])} complete branches | tau = {display["tau_css30"]:g} | VGN-root horizontal span x{display["horizontal_layout"]["VGN_root_span_scale"]:g}',
              .05 * width, .902 * height, 8.8)
    draw_text(c, "CVG candidate branches remain provisional; original SGN terminal assignments are unchanged.",
              .05 * width, .867 * height, 8.8, fill="#b94b43")
    footer(c, width, f'Stage KDE recomputed on display y; bandwidth = {density["bandwidth_display"]:.4f} display units. Widths do not show abundance.')
    finish(c, "all", WHOLE_SIZE)

    for fate in FATES:
        c = new_canvas(fate, PATH_SIZE)
        width, height = np.array(PATH_SIZE) * 72
        axes(c, .62)
        skeleton(c, .62, highlighted=True)
        item = bundle["figures"][fate]
        mask = np.asarray(item["mask"], bool)
        assert mask.shape == (len(obs),) and int(mask.sum()) == int(item["cells"])
        start, tip = int(item["start"]), int(item["tip"])
        origin, length = float(rd[start]), float(rd[tip] - rd[start])
        selected = order[mask[order]]
        fraction = (times[selected] - origin) / length
        assert np.all((fraction >= -1e-12) & (fraction <= 1 + 1e-12))
        path_colors = _ramp_lut(fate, fraction)
        c.setFillAlpha(PATH_ALPHA)
        for idx, fill in zip(selected, path_colors):
            xp, yp = cell_map(.62, xy[idx, 0], xy[idx, 1])
            circle(c, float(xp), float(yp), radius, fill)
        c.setFillAlpha(1.)
        local_labels(c, .62, compact=True)
        for target in FATES:
            target_tip = int(bundle["figures"][target]["tip"])
            xp, yp = node_map(.62, nx[target_tip], ny[target_tip])
            fill = "#252525" if target == fate else "#888888"
            circle(c, xp, yp, math.sqrt(8.) / 2, "#ffffff", fill, .5)
            draw_text(c, target + " end", xp, yp - 12., 8., "Arial-Bold" if target == fate else "Arial",
                      fill, align="center", valign="top", role="endpoint", panel="left")
        xp, yp = node_map(.62, nx[start], ny[start])
        circle(c, xp, yp, math.sqrt(17.) / 2, "#ffffff", "#444444", .7)
        draw_text(c, "Path start", xp + 5, yp + 8, 7.8, panel="left")
        lx, lw = .70 * width, .27 * width
        def legend_y(y): return AXES_BOTTOM + AXES_HEIGHT * y
        circle(c, lx + .02 * lw, legend_y(.97), math.sqrt(18.) / 2,
               _ramp_lut(fate, np.array([.65]))[0], "#333333", .2)
        draw_text(c, f'Selected {fate} path\n({item["cells"]:,} cells)', lx + .10 * lw, legend_y(.975),
                  8., valign="top", role="legend", panel="legend")
        line(c, lx - .01 * lw, legend_y(.85), lx + .05 * lw, legend_y(.85), "#b4b4b4", 1.)
        draw_text(c, "Other tree segments", lx + .10 * lw, legend_y(.85), 8.,
                  valign="center", role="legend", panel="legend")
        bx, by, bw, bh = .71 * width, .390 * height, .018 * width, .22 * height
        colors = _ramp_lut(fate, np.linspace(0., 1., 256))
        for j, fill in enumerate(colors):
            c.setFillColor(color(fill))
            c.rect(bx, by + bh * j / 256, bw, bh / 256 + .005, fill=1, stroke=0)
        for fraction in (0., .5, 1.):
            yy = by + bh * fraction
            line(c, bx + bw, yy, bx + bw + 3.5, yy, "#6c7074", .6)
            draw_text(c, f"{fraction * length:.2f}", bx + bw + 7, yy, 7.8,
                      valign="center", role="tick", panel="legend")
        draw_text(c, "Original path distance\n(CSS30; color only)", .71 * width, .655 * height,
                  8., valign="top", role="legend", panel="legend")
        draw_text(c, f'Original Immature {fate}:\nOn path: {item["path_typed"]}/{item["typed_total"]}\nOn terminal:\n{item["terminal_typed"]}/{item["typed_total"]}',
                  lx, legend_y(.24), 8., valign="top", role="legend", panel="legend")
        draw_text(c, f"{fate} path on the log1p display", .065 * width, .944 * height, 16., "Arial-Bold", role="title")
        draw_text(c, f'tau = {display["tau_css30"]:g} | {item["cells"]:,} path-selected cells | same geometry as the whole tree',
                  .065 * width, .902 * height, 8.7)
        draw_text(c, "Gray skeleton: complete stitched graph | color: original path pseudotime",
                  .065 * width, .867 * height, 8.3, fill="#555a61")
        footer(c, width, "Candidate enrichment does not establish cell fate.")
        finish(c, fate, PATH_SIZE)

    np.testing.assert_array_equal(xy, original_xy)
    spec = dict(whole_size=list(WHOLE_SIZE), path_size=list(PATH_SIZE),
                axes_left_inches=[.62, 3.95], axes_bottom_inches=AXES_BOTTOM / 72,
                axes_width_inches=AXES_WIDTH / 72, axes_height_inches=AXES_HEIGHT / 72,
                xlim=[XMIN, XMAX], ylim=[ymin, ymax], y_inverted=True, tick_values=ticks.tolist(),
                raw_x_center=X_CENTER, raw_x_scale=X_SCALE, cell_xy_already_display_scaled=True,
                tau_css30=float(display["tau_css30"]), VGN_root_span_scale=.5,
                display_xy_sha256=_array_sha(xy), node_y_sha256=_array_sha(ny),
                original_t_sha256=_array_sha(times), original_segment_rgb_sha256=_array_sha(rgb_values),
                point_area=POINT_AREA, point_diameter_pt=2 * radius,
                whole_point_alpha=WHOLE_ALPHA, path_point_alpha=PATH_ALPHA,
                path_gradient_levels=1000, path_gradient_stops={k: list(v) for k, v in RAMPS.items()},
                stage_palette=palettes, density_bandwidth_display=float(density["bandwidth_display"]),
                all_views_share_geometry=True, expected_clips=0, expected_images=0, expected_forms=0,
                fonts=["ArialMT", "Arial-BoldMT"], PNG_DPI=DPI,
                rendering="ReportLab direct vector paths and semantic live text; Poppler PNG")
    return dict(artifacts=artifacts, scenes=scenes, physical_coordinate_checks=physical,
                physicalmetadata=physical, render_spec=spec)


def render_tree_all_compact(bundle, display, density, outdir, work):
    #1. =========== Display-only settings
    size_in = (6.8, 7.6)
    dpi = 600
    tree_top, tree_height = 500., 420.
    panel_left = {"A": 24., "B": 256.}
    label_font, legend_font = 11.3, 7.8
    point_area, point_alpha = 1.4, .72
    legend_alpha = 1.        # Opaque nominal RGB avoids seams between transparent vector strips.
    legend_x, legend_top, legend_step = 100., 497., 12.
    swatch_width, swatch_height, gradient_steps = 20., 7., 256
    density_left, density_width = 399., 77.
    stage_x, stage_top, stage_step = 351., 497., 7.

    #2. =========== Runtime and immutable source arrays
    runtime = ROOT
    try:
        import reportlab
    except ImportError:
        sys.path.append(str(runtime/"python/Lib/site-packages"))
    from reportlab.pdfgen import canvas
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.lib.colors import Color, HexColor
    from PIL import Image
    for name, filename in (("Arial", "arial.ttf"), ("Arial-Bold", "arialbd.ttf")):
        if name not in pdfmetrics.getRegisteredFontNames():
            pdfmetrics.registerFont(TTFont(name,str(ROOT/filename)))
    outdir, work = Path(outdir), Path(work)
    outdir.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    pdf, png = outdir/"tree_all.pdf", outdir/"tree_all.png"
    width, height = np.asarray(size_in)*72
    xy = np.asarray(display["cell_xy"])
    ny = np.asarray(display["node_y"])
    nx = {int(k):float(v) for k,v in display["node_x"].items()}
    segments = bundle["segments"]
    sx = np.asarray(display["segment_x"])
    coloring = bundle["segment_gradient_coloring"]
    rgb = np.asarray(coloring["color_rgb_uint8"])
    order = np.asarray(bundle["plot_order"])
    stages = ("E9.5","E10.5","E11.5","E13.5","E14.5","E15.5","E16.5","E17.5","E18.5")
    stage_colors = ("#f5b888","#f2a985","#ee9982","#e58780","#cd717e","#b55c7d","#914679","#653175","#3a1c71")
    palette = dict(zip(stages,stage_colors))
    stage_values = bundle["obs"].stage_standardized.astype(str).to_numpy()
    digest = lambda value: hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()
    frozen = {"cell_xy":digest(xy), "node_y":digest(ny), "rgb":digest(rgb),
              "F":digest(bundle["model"]["F"]), "B":digest(bundle["model"]["B"]),
              "t":digest(bundle["projection"]["t"])}
    maximum = float(ny.max())
    sy = tree_height/maximum
    previous_sy = (.57*8.5*72)/(1.365*maximum)
    uniform_scale = sy/previous_sy
    x_scale = (2.8*72/4)*uniform_scale
    assert len(segments)==11 and len(xy)==7355
    assert float(display["tau_css30"])==.25
    assert display["horizontal_layout"]["VGN_root_span_scale"]==.5
    radius = math.sqrt(point_area)/2
    scenes, leaders, gradient_records, physical = [], [], [], {}

    def color(value,alpha=1.):
        value=HexColor(value) if isinstance(value,str) else Color(*map(float,np.asarray(value)[:3]))
        value.alpha=alpha
        return value

    def dx(value): return 1.5+.5*(np.asarray(value)-1.5)

    def cell_map(panel,x,y):
        return panel_left[panel]+x_scale*(np.asarray(x)-.75), tree_top-sy*np.asarray(y)

    def node_map(panel,x,y): return cell_map(panel,dx(x),y)

    c = canvas.Canvas(str(pdf),pagesize=(width,height),pageCompression=1,invariant=1,
                      initialFontName="Arial",initialFontSize=8)
    c.setTitle("Segment pseudotime and developmental stage")
    c.setAuthor("Developmental map of auditory neurons")
    c.setSubject("Compact A/B presentation; accepted graph and original annotations preserved")

    def text(value,x,y,size=8.,align="left",font="Arial",role="label",panel=None,top=False):
        value=str(value); lines=value.split("\n"); leading=1.13*size
        asc,desc=pdfmetrics.getAscentDescent(font,size)
        baseline=y-asc if top else y
        widths=[pdfmetrics.stringWidth(v,font,size) for v in lines]
        t=c.beginText();t.setFont(font,size);t.setFillColor(color("#202020"))
        for j,(line,w) in enumerate(zip(lines,widths)):
            offset=0. if align=="left" else (-w/2 if align=="center" else -w)
            t.setTextOrigin(x+offset,baseline-j*leading);t.textOut(line)
        c.drawText(t)
        w=max(widths);left=x if align=="left" else (x-w/2 if align=="center" else x-w)
        rec={"text":value,"bbox":[float(left),float(baseline-(len(lines)-1)*leading+desc),
             float(left+w),float(baseline+asc)],"role":role,"panel":panel,"font":font,"size":size}
        scenes.append(rec);return rec

    def line(x0,y0,x1,y1,fill="#727782",lw=.7):
        c.setStrokeColor(color(fill));c.setLineWidth(lw)
        c.line(float(x0),float(y0),float(x1),float(y1))

    def dot(x,y,r,fill,stroke=None,lw=.5,alpha=1.):
        c.setFillColor(color(fill,alpha))
        if stroke is not None:c.setStrokeColor(color(stroke));c.setLineWidth(lw)
        c.circle(float(x),float(y),float(r),fill=1,stroke=int(stroke is not None))

    #3. =========== Two identical tree geometries with different cell colors
    for panel in ("A","B"):
        for k,seg in enumerate(segments):
            a,z=int(seg["start"]),int(seg["end"])
            xa,ya=node_map(panel,sx[k],ny[a]);xz,yz=node_map(panel,sx[k],ny[z])
            line(xa,ya,xz,yz)
            if nx[a]!=sx[k]:
                x0,y0=node_map(panel,nx[a],ny[a]);line(x0,y0,xa,ya)
        px,py=cell_map(panel,xy[:,0],xy[:,1])
        c.setFillAlpha(point_alpha)
        for idx in order:
            fill=rgb[idx]/255. if panel=="A" else palette[stage_values[idx]]
            dot(px[idx],py[idx],radius,fill,alpha=point_alpha)
        c.setFillAlpha(1.)
        root=int(bundle["projection"]["root"])
        rx,ry=node_map(panel,nx[root],ny[root]);dot(rx,ry,1.7,"#ffffff","#404040",.6)
        text("Root",rx,ry+6.,7.2,align="center",role="endpoint",panel=panel)
        vg=int(bundle["vgn_tip"])
        vx,vy=node_map(panel,nx[vg],ny[vg]);dot(vx,vy,1.7,"#ffffff","#404040",.6)
        text("VGN candidate-\nenriched",panel_left[panel]+11.,vy-9.,8.,align="center",top=True,role="endpoint",panel=panel)
        # The display labels identify the same existing ends; obs/state is never edited.
        positions={"Ib":(panel_left[panel]+11.,18.),"Ic":(panel_left[panel]+.5*x_scale,66.),
                   "Ia":(panel_left[panel]+x_scale,33.),"II":(panel_left[panel]+1.5*x_scale,66.)}
        names={"Ib":"Immature Ib","Ic":"Emerging Ic\nImmature Ic","Ia":"Immature Ia","II":"Immature II"}
        for fate in ("Ib","Ic","Ia","II"):
            tip=int(bundle["figures"][fate]["tip"])
            tx,ty=node_map(panel,nx[tip],ny[tip]);dot(tx,ty,1.7,"#ffffff","#404040",.6)
            lx,ly=positions[fate]
            c.setDash(.9,1.8);line(tx,ty-2.,lx,ly+3.,"#929292",.45);c.setDash()
            leaders.append({"panel":panel,"fate":fate,"points":[float(tx),float(ty-2),float(lx),float(ly+3)]})
            text(names[fate],lx,ly,label_font,align="center",top=True,role="endpoint",panel=panel)
        gx,gy=node_map(panel,np.asarray([nx[i] for i in range(len(ny))]),ny)
        physical[panel]={"cell_xy_pt_sha256":digest(np.column_stack([px,py])),
                         "node_xy_pt_sha256":digest(np.column_stack([gx,gy])),
                         "node_paper_coordinates":{str(i):[float(gx[i]),float(gy[i])] for i in range(len(ny))},
                         "normalized_display_sha256":digest(xy)}
    #4. =========== Segment gradients, not categorical solid swatches
    short_labels={"early_common":"CVG common","VGN_candidate":"VGN candidate-enriched",
                  "SGN_candidate":"SGN candidate-enriched","connector":"Component connection",
                  "SGN_common":"SGN common trajectory","IbIc":"Ib/Ic shared segment",
                  "Ib":"Ib-enriched terminal","Ic":"Emerging Ic-enriched terminal",
                  "IaII":"Ia/II shared segment","Ia":"Ia-associated terminal","II":"II-associated terminal"}
    for j,k in enumerate(coloring["legend_order"]):
        row,seg=coloring["segments"][k],segments[k]
        yy=legend_top-j*legend_step
        if seg["role"]=="connector":
            line(legend_x,yy,legend_x+swatch_width,yy,"#8b8b8b",.7)
            steps=0
        else:
            a=np.asarray([int(row["start_color"][i:i+2],16) for i in (1,3,5)])/255.
            z=np.asarray([int(row["end_color"][i:i+2],16) for i in (1,3,5)])/255.
            c.setFillAlpha(legend_alpha)
            for m,v in enumerate(np.linspace(a,z,gradient_steps)):
                c.setFillColor(color(v,legend_alpha))
                c.rect(legend_x+swatch_width*m/gradient_steps,yy-swatch_height/2,
                       swatch_width/gradient_steps+.003,swatch_height,fill=1,stroke=0)
            c.setFillAlpha(1.)
            c.setStrokeColor(color("#383838"));c.setLineWidth(.45)
            c.rect(legend_x,yy-swatch_height/2,swatch_width,swatch_height,fill=0,stroke=1)
            steps=gradient_steps
        text(short_labels[seg["role"]],legend_x+swatch_width+6.,yy-2.5,legend_font,role="legend",panel="A")
        gradient_records.append({"metric_segment":int(k),"role":seg["role"],"bands":steps,
                                 "start_color":row["start_color"],"end_color":row["end_color"],
                                 "alpha":legend_alpha,"bbox":[legend_x,yy-swatch_height/2,legend_x+swatch_width,yy+swatch_height/2]})
    #5. =========== Stage dot sequence and density curves on the same y scale
    _,yy=cell_map("B",0.,np.asarray(density["grid"]))
    for stage in reversed(stages):
        xx=density_left+density_width*np.asarray(density["display_values"][stage])
        path=c.beginPath();path.moveTo(density_left,float(yy[0]))
        for x,y in zip(xx,yy):path.lineTo(float(x),float(y))
        path.lineTo(density_left,float(yy[-1]));path.close()
        c.setFillColor(color(palette[stage]));c.setFillAlpha(.2);c.drawPath(path,fill=1,stroke=0);c.setFillAlpha(1.)
        path=c.beginPath();path.moveTo(float(xx[0]),float(yy[0]))
        for x,y in zip(xx[1:],yy[1:]):path.lineTo(float(x),float(y))
        c.setStrokeColor(color(palette[stage]));c.setLineWidth(.9);c.drawPath(path,fill=0,stroke=1)
    for j,stage in enumerate(stages):dot(stage_x,stage_top-j*stage_step,3.,palette[stage])
    text(stages[0],stage_x+6.,stage_top-2.5,8.4,role="legend",panel="B")
    bottom=stage_top-(len(stages)-1)*stage_step
    text(stages[-1],stage_x+6.,bottom-2.5,8.4,role="legend",panel="B")
    arrow_x=stage_x+24.
    line(arrow_x,stage_top-6.,arrow_x,bottom+6.,"#4b4b4b",.8)
    line(arrow_x,bottom+6.,arrow_x-2.,bottom+10.,"#4b4b4b",.8)
    line(arrow_x,bottom+6.,arrow_x+2.,bottom+10.,"#4b4b4b",.8)
    text("A",12.,522.,28.,role="panel_letter",panel="A")
    text("Segment pseudotime",39.,524.,15.2,role="panel_title",panel="A")
    text("B",232.,522.,28.,role="panel_letter",panel="B")
    text("Developmental stage",261.,524.,15.2,role="panel_title",panel="B")

    #6. =========== Layout and immutable-data checks
    for item in scenes:
        x0,y0,x1,y1=item["bbox"]
        assert min(x0,y0)>=.0 and x1<=width and y1<=height,("Text outside page",item)
    for i,a in enumerate(scenes):
        for b in scenes[i+1:]:
            x0,y0,x1,y1=a["bbox"];u0,v0,u1,v1=b["bbox"]
            assert x1<=u0 or u1<=x0 or y1<=v0 or v1<=y0,("Text overlap",a,b)
    for panel in ("A","B"):
        xs,ys=cell_map(panel,xy[:,0],xy[:,1])
        for item in scenes:
            if item["role"]!="legend" or item["panel"]!="A":continue
            x0,y0,x1,y1=item["bbox"]
            assert not np.any((xs>=x0-radius)&(xs<=x1+radius)&(ys>=y0-radius)&(ys<=y1+radius)),("Legend covers points",item)
    assert all(v["bands"]==256 for v in gradient_records if v["role"]!="connector")
    assert frozen["cell_xy"]==digest(xy) and frozen["node_y"]==digest(ny) and frozen["rgb"]==digest(rgb)
    assert frozen["F"]==digest(bundle["model"]["F"]) and frozen["B"]==digest(bundle["model"]["B"])
    assert frozen["t"]==digest(bundle["projection"]["t"])
    c.showPage();c.save()
    poppler=Path(os.environ.get("TREE_PDFTOPPM",str(runtime/"native/poppler/Library/bin/pdftoppm.exe")))
    subprocess.run([str(poppler),"-singlefile","-r",str(dpi),"-png",str(pdf),str(png.with_suffix(""))],check=True,capture_output=True)
    with Image.open(png) as image:
        pixels=list(image.size);actual_dpi=list(image.info["dpi"])
        assert all(abs(a-round(b*dpi))<=1 for a,b in zip(pixels,size_in))
        assert all(abs(v-dpi)<.05 for v in actual_dpi);image.verify()
    artifact={"pdf":str(pdf),"png":str(png),"inches":list(size_in),"pixels":pixels,"dpi":actual_dpi,
              "pdf_sha256":hashlib.sha256(pdf.read_bytes()).hexdigest(),"png_sha256":hashlib.sha256(png.read_bytes()).hexdigest(),
              "expected_clips":0,"expected_images":0,"expected_forms":0,"required_font_styles":["regular"],
              "semantic_text_objects":len(scenes)}
    spec={"layout":"Compact A/B, no axes or grid, unchanged accepted VGN/SGN topology",
          "whole_size":list(size_in),"tree_top_pt":tree_top,"tree_height_pt":tree_height,
          "x_scale_pt":x_scale,"y_scale_pt":sy,"uniform_scale_from_previous_tree":uniform_scale,
          "panel_left_pt":panel_left,"point_area":point_area,"point_alpha":point_alpha,
          "legend_alpha":legend_alpha,"legend_rendering":"Opaque nominal RGB gradients; no translucent strip seams",
          "display_xy_sha256":frozen["cell_xy"],"node_y_sha256":frozen["node_y"],
          "original_segment_rgb_sha256":frozen["rgb"],"original_t_sha256":frozen["t"],
          "VGN_preserved":True,"segment_legend":gradient_records,"scene":scenes,"leaders":leaders,
          "density_bandwidth_display":float(density["bandwidth_display"]),"density_peak_normalization":1,
          "expected_clips":0,"expected_images":0,"expected_forms":0,"fonts":["ArialMT"],"PNG_DPI":dpi,
          "other_four_tree_layouts":"Unchanged; shared normalized geometry, independent page layout"}
    return {"artifact":artifact,"scene":scenes,"physical":physical,"render_spec":spec}


In [ ]:
def render_composites(bundle, display, heatmaps, outdir, work):
    """Compose C-G with vector RGB runs and unchanged trees, strips and text."""
    from pathlib import Path
    import hashlib
    import io
    import math
    import re
    import subprocess
    import sys
    import numpy as np
    from PIL import Image

    # Physical geometry is shared across the five panels.
    DPI = 600
    UNDERPAINT_PT = .2  # Restore the previous PDF-only seam treatment.
    SIZE = (5.45, 4.6)
    HEAT_SHIFT = 1.65
    MATRIX = (HEAT_SHIFT+.20*3.8, .24*4.6, .50*3.8, .58*4.6)
    TREE_LEFT = .20
    TREE_HEIGHT = MATRIX[3]
    TREE_WIDTH = TREE_HEIGHT*2.8/4.845
    TREE_SCALE = TREE_HEIGHT/4.845
    STRIP_HEIGHT, STRIP_GAP = .014*4.6, .004*4.6
    XMIN, XMAX = -.5, 3.5
    TREE_POINT_DIAMETER = math.sqrt(1.4)*TREE_SCALE
    TREE_ALPHA = .88
    STAGES = ['E9.5', 'E10.5', 'E11.5', 'E13.5', 'E14.5', 'E15.5', 'E16.5', 'E17.5', 'E18.5']
    STAGE_COLORS = ['#f5b888', '#f2a985', '#ee9982', '#e58780', '#cd717e', '#b55c7d', '#914679', '#653175', '#3a1c71']
    FATE_BY_PANEL = {'D': 'Ia', 'E': 'Ib', 'F': 'Ic', 'G': 'II'}
    runtime = ROOT
    try:
        import reportlab
    except ImportError:
        sys.path.append(str(runtime/'python/Lib/site-packages'))
        import reportlab
    from reportlab.pdfgen import canvas
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.lib.colors import Color, HexColor
    for name, filename in [('Arial', 'arial.ttf'), ('Arial-Bold', 'arialbd.ttf'), ('Arial-Italic', 'ariali.ttf')]:
        if name not in pdfmetrics.getRegisteredFontNames():
            pdfmetrics.registerFont(TTFont(name, str(ROOT/filename)))
    outdir, work = Path(outdir), Path(work)
    outdir.mkdir(parents=True, exist_ok=True); work.mkdir(parents=True, exist_ok=True)
    pdftoppm = runtime/'native/poppler/Library/bin/pdftoppm.exe'
    assert pdftoppm.is_file()
    obs = bundle['obs']
    xy = np.asarray(display['cell_xy'], float)
    node_y = np.asarray(display['node_y'], float)
    node_x = display['node_x']
    segment_x = np.asarray(display['segment_x'], float)
    assert xy.shape == (len(obs), 2) and np.isfinite(xy).all()
    assert len(segment_x) == len(bundle['segments'])
    maximum = float(node_y.max())
    ymin, ymax = -.045*maximum, 1.32*maximum
    graph_x = lambda value: 1.5+.5*(np.asarray(value)-1.5)
    original_xy_sha = hashlib.sha256(xy.tobytes()).hexdigest()
    width, height = np.array(SIZE)*72
    artifacts, scenes, image_boxes, mappings, stripped_symbols = {}, {}, {}, {}, {}

    def as_rgb8(value):
        a = np.asarray(value)
        assert a.ndim >= 2 and a.shape[-1] == 3 and np.isfinite(a).all()
        if a.dtype == np.uint8:
            return np.ascontiguousarray(a)
        assert a.min() >= 0
        if a.max() <= 1:
            return np.ascontiguousarray(np.rint(a*255).astype(np.uint8))
        assert a.max() <= 255
        return np.ascontiguousarray(np.rint(a).astype(np.uint8))

    def vector_matrix(c, matrix, box):
        """Restore editable same-color runs without resampling rows or columns."""
        rows, columns = matrix.shape[:2]
        keys = ((matrix[..., 0].astype(np.uint32) << 16) |
                (matrix[..., 1].astype(np.uint32) << 8) | matrix[..., 2].astype(np.uint32))
        start = np.ones((rows, columns), dtype=bool)
        start[:, 1:] = keys[:, 1:] != keys[:, :-1]
        rr, cc = np.nonzero(start)
        next_c, next_r = np.r_[cc[1:], columns], np.r_[rr[1:], rows]
        end = np.where(next_r == rr, next_c, columns)
        run_keys = keys[rr, cc]
        order = np.argsort(run_keys, kind='stable')
        sorted_keys = run_keys[order]
        splits = np.r_[0, np.flatnonzero(sorted_keys[1:] != sorted_keys[:-1])+1, len(order)]
        mx, my, mw, mh = map(float, box)
        dx, dy = mw/columns, mh/rows
        groups = []
        for a, b in zip(splits[:-1], splits[1:]):
            chosen = order[a:b]
            key = int(sorted_keys[a])
            color = ((key >> 16)/255., ((key >> 8) & 255)/255., (key & 255)/255.)
            rectangles = '\n'.join(
                f'{mx+cc[i]*dx:.8f} {my+(rows-rr[i]-1)*dy:.8f} '
                f'{(end[i]-cc[i])*dx:.8f} {dy:.8f} re' for i in chosen)
            groups.append((color, rectangles))
        c._code.append('%%HEATMAP_VECTOR_BEGIN')
        c.saveState()
        c.setLineJoin(0)
        c.setLineCap(0)
        c.setLineWidth(UNDERPAINT_PT)
        c._code.append('%%HEATMAP_UNDERPAINT_BEGIN')
        for color, rectangles in groups:
            channels = ' '.join(f'{v:.10f}' for v in color)
            c._code.append(f'{channels} rg\n{channels} RG\n{rectangles}\nB')
        c._code.append('%%HEATMAP_MAIN_BEGIN')
        for color, rectangles in groups:
            channels = ' '.join(f'{v:.10f}' for v in color)
            c._code.append(f'{channels} rg\n{rectangles}\nf')
        c._code.append('%%HEATMAP_MAIN_END')
        c.restoreState()
        c._code.append('%%HEATMAP_VECTOR_END')
        return dict(runs=int(len(rr)), colors=len(groups), underpaint_pt=UNDERPAINT_PT,
                    rgb_sha256=hashlib.sha256(matrix.tobytes()).hexdigest())

    def rgb(value):
        if isinstance(value, str):
            return HexColor(value)
        a = np.asarray(value, float)
        if a.max() > 1:
            a = a/255
        return Color(*a)

    def text(c, value, x, y, size=7., font='Arial', fill='#252525', align='left', role='label'):
        lines = str(value).split('\n')
        widths = [pdfmetrics.stringWidth(line, font, size) for line in lines]
        asc, desc = pdfmetrics.getAscentDescent(font, size)
        leading = size*1.18
        obj = c.beginText(); obj.setFont(font, size); obj.setFillColor(rgb(fill))
        for i, (line, line_width) in enumerate(zip(lines, widths)):
            dx = 0 if align == 'left' else (-line_width/2 if align == 'center' else -line_width)
            obj.setTextOrigin(x+dx, y-i*leading); obj.textOut(line)
        c.drawText(obj)
        left = x if align == 'left' else x-max(widths)/(2 if align == 'center' else 1)
        c._scene.append(dict(text=str(value), role=role, font=font,
                            bbox=[left, y+desc-(len(lines)-1)*leading, left+max(widths), y+asc]))

    def line(c, x0, y0, x1, y1, fill='#777777', weight=.45):
        c.setStrokeColor(rgb(fill)); c.setLineWidth(weight); c.line(x0, y0, x1, y1)

    def circle(c, x, y, radius, fill, stroke=None, weight=.35):
        k = radius*4*(math.sqrt(2)-1)/3
        p = c.beginPath(); p.moveTo(x+radius, y)
        p.curveTo(x+radius, y+k, x+k, y+radius, x, y+radius)
        p.curveTo(x-k, y+radius, x-radius, y+k, x-radius, y)
        p.curveTo(x-radius, y-k, x-k, y-radius, x, y-radius)
        p.curveTo(x+k, y-radius, x+radius, y-k, x+radius, y); p.close()
        c.setFillColor(rgb(fill))
        if stroke is not None:
            c.setStrokeColor(rgb(stroke)); c.setLineWidth(weight)
        c.drawPath(p, fill=1, stroke=int(stroke is not None))

    def tree_map(x, y):
        return ((TREE_LEFT+TREE_WIDTH*(np.asarray(x)-XMIN)/(XMAX-XMIN))*72,
                (MATRIX[1]+TREE_HEIGHT*(ymax-np.asarray(y))/(ymax-ymin))*72)

    def graph_map(x, y):
        return tree_map(graph_x(x), y)

    def vector_strip(c, values, box, frame=True):
        values = as_rgb8(values)
        assert values.ndim == 2
        x, y, w, h = np.array(box)*72
        changes = np.r_[0, np.flatnonzero(np.any(values[1:] != values[:-1], axis=1))+1, len(values)]
        for start, end in zip(changes[:-1], changes[1:]):
            x0, x1 = x+w*start/len(values), x+w*end/len(values)
            c.setFillColor(rgb(values[start]))
            overlap = min(.005, x+w-x1)
            c.rect(x0, y, x1-x0+overlap, h, fill=1, stroke=0)
        if frame:
            c.setStrokeColor(rgb('#333333')); c.setLineWidth(.45); c.rect(x, y, w, h, fill=0, stroke=1)

    def clean_symbol(value):
        return re.sub(r'[\s*\u2020\u2021\u00b0]+$', '', str(value).strip())

    for panel in 'CDEFG':
        item = heatmaps[panel]
        matrix = as_rgb8(item['rgb'])
        assert matrix.ndim == 3
        n_genes, n_cells = matrix.shape[:2]
        cells = item['obs']
        labels = item['labels']
        assert len(cells) == n_cells
        cell_ids = cells['cell_id'].astype(str).to_numpy()
        assert len(np.unique(cell_ids)) == n_cells
        positions = obs.index.get_indexer(cell_ids)
        assert np.all(positions >= 0)
        age_rgb = as_rgb8(item['stage_strip_rgb'])
        time_rgb = as_rgb8(item['pseudotime_strip_rgb'])
        assert age_rgb.shape == time_rgb.shape == (n_cells, 3)
        selected_mask = np.zeros(len(obs), bool)
        selected_mask[positions] = True
        if 'tree_mask' in item:
            np.testing.assert_array_equal(selected_mask, np.asarray(item['tree_mask'], bool))
        tree_rgb = np.full((len(obs), 3), 255, np.uint8)
        tree_rgb[positions] = time_rgb
        if 'tree_rgb' in item:
            np.testing.assert_array_equal(tree_rgb, np.asarray(item['tree_rgb'], np.uint8))
        if 'tree_cell_ids' in item:
            assert np.array_equal(cell_ids, np.asarray(item['tree_cell_ids']).astype(str))
        if panel != 'C':
            expected = np.asarray(bundle['figures'][FATE_BY_PANEL[panel]]['mask'], bool)
            assert set(obs.index[expected]) == set(cell_ids)
        else:
            assert n_cells == 2332, 'Panel C must retain the original 2,332-cell set'
        pdf, png = outdir/f'Fig3{panel}.pdf', outdir/f'Fig3{panel}.png'
        c = canvas.Canvas(str(pdf), pagesize=(width, height), invariant=1, pageCompression=1,
                          initialFontName='Arial', initialFontSize=7)
        c.setTitle(f'Fig3{panel}: tree and gene dynamics')
        c.setAuthor('Developmental map of auditory neurons')
        c.setSubject('Vector RGB heatmap rectangles; unchanged tree, strips, leaders and live Arial gene labels')
        c._scene = []

        # The full accepted display is reused for all five trees at one uniform scale.
        for k, seg in enumerate(bundle['segments']):
            start, end = int(seg['start']), int(seg['end'])
            x0, y0 = graph_map(segment_x[k], node_y[start])
            x1, y1 = graph_map(segment_x[k], node_y[end])
            line(c, x0, y0, x1, y1, '#A7AAAC', .65*TREE_SCALE)
            if node_x[start] != segment_x[k]:
                xp, yp = graph_map(node_x[start], node_y[start])
                line(c, xp, yp, x0, y0, '#A7AAAC', .65*TREE_SCALE)
        c.setFillAlpha(TREE_ALPHA)
        # Heatmap pseudotime colors are aligned to the same cell IDs, never to row rank alone.
        source_order = np.asarray(bundle.get('plot_order', np.arange(len(obs))), int)
        for cell in source_order[selected_mask[source_order]]:
            xp, yp = tree_map(*xy[cell])
            circle(c, float(xp), float(yp), TREE_POINT_DIAMETER/2, tree_rgb[cell])
        c.setFillAlpha(1); c.setStrokeAlpha(1)
        root = int(bundle['projection']['root'])
        xp, yp = graph_map(node_x[root], node_y[root])
        circle(c, xp, yp, 1.0, '#FFFFFF', '#555555', .35)
        text(c, 'Root', xp, yp+4, 5.8, align='center', role='tree_label')
        if 'vgn_tip' in bundle:
            vg = int(bundle['vgn_tip'])
            xp, yp = graph_map(node_x[vg], node_y[vg])
            text(c, 'VGN candidate-\nenriched', xp, yp-6, 5.1, align='center', role='tree_label')
        for fate in ('Ib', 'Ic', 'Ia', 'II'):
            tip = int(bundle['figures'][fate]['tip'])
            xp, yp = graph_map(node_x[tip], node_y[tip])
            fill = '#333333' if FATE_BY_PANEL.get(panel) == fate else '#999999'
            circle(c, xp, yp, .9, '#FFFFFF', fill, .3)
            text(c, fate, xp, yp-7, 5.9, font='Arial-Bold' if FATE_BY_PANEL.get(panel) == fate else 'Arial',
                 fill=fill, align='center', role='tree_label')
        if panel in FATE_BY_PANEL:
            start = int(bundle['figures'][FATE_BY_PANEL[panel]]['start'])
            xp, yp = graph_map(node_x[start], node_y[start])
            circle(c, xp, yp, 1.25, '#FFFFFF', '#555555', .4)
        tree_heading = 'Original common cell set' if panel == 'C' else FATE_BY_PANEL[panel]+' path'
        text(c, tree_heading, TREE_LEFT*72, (MATRIX[1]+MATRIX[3]+.20)*72, 7.5,
             font='Arial-Bold', role='tree_title')
        circle(c, TREE_LEFT*72+2, 62, 1.8, time_rgb[len(time_rgb)//2])
        text(c, f'Selected cells ({n_cells:,})', TREE_LEFT*72+9, 60, 5.8, role='legend')
        line(c, TREE_LEFT*72, 49, TREE_LEFT*72+5, 49, '#A7AAAC', .6)
        text(c, 'Other tree segments', TREE_LEFT*72+9, 47, 5.8, role='legend')

        # Preserve the frozen RGB matrix as editable rectangle subpaths.
        mx, my, mw, mh = np.array(MATRIX)*72
        c.setFillAlpha(1); c.setStrokeAlpha(1)
        vector_report = vector_matrix(c, matrix, (mx, my, mw, mh))
        c.setStrokeColor(rgb('#333333')); c.setLineWidth(.5); c.rect(mx, my, mw, mh, fill=0, stroke=1)
        image_boxes[panel] = dict(x=mx, y=my, width=mw, height=mh, pixels=[n_cells, n_genes],
                                  color_space='DeviceRGB', alpha=False, image_count=0, vector_report=vector_report,
                                  rgb_sha256=hashlib.sha256(matrix.tobytes()).hexdigest())
        time_box = (MATRIX[0], MATRIX[1]+MATRIX[3]+STRIP_GAP, MATRIX[2], STRIP_HEIGHT)
        age_box = (MATRIX[0], MATRIX[1]+MATRIX[3]+2*STRIP_GAP+STRIP_HEIGHT, MATRIX[2], STRIP_HEIGHT)
        vector_strip(c, time_rgb, time_box); vector_strip(c, age_rgb, age_box)
        text(c, 'Pseudotime', mx-3, (time_box[1]+STRIP_HEIGHT/2)*72-1.7, 5.6, align='right', role='strip_label')
        text(c, 'Age', mx-3, (age_box[1]+STRIP_HEIGHT/2)*72-1.7, 5.6, align='right', role='strip_label')

        # One complete live italic text object per gene, with vector leader lines.
        label_rows = []
        if hasattr(labels, 'iterrows'):
            iterator = [row for _, row in labels.iterrows()]
        else:
            iterator = labels
        for row in iterator:
            symbol = clean_symbol(row['symbol'])
            index = int(row['display_row'])
            assert 0 <= index < n_genes and symbol
            label_rows.append((index, symbol, str(row['symbol'])))
        label_rows.sort(key=lambda x: x[0])
        assert len({x[1] for x in label_rows}) == len(label_rows)
        anchors = np.array([(row+.5)*mh/n_genes for row, _, _ in label_rows])
        locations = anchors.copy()
        min_gap, padding = 9., 4.
        if len(locations):
            for i in range(1, len(locations)):
                locations[i] = max(locations[i], locations[i-1]+min_gap)
            locations[-1] = min(locations[-1], mh-padding)
            for i in range(len(locations)-2, -1, -1):
                locations[i] = min(locations[i], locations[i+1]-min_gap)
            if locations[0] < padding:
                locations += padding-locations[0]
            assert locations[-1] <= mh-padding+1e-9
        for (row, symbol, original), anchor, target in zip(label_rows, anchors, locations):
            ya, yt = my+mh-anchor, my+mh-target
            line(c, mx+mw+.8, ya, mx+mw+4.5, ya, '#555555', .4)
            line(c, mx+mw+4.5, ya, mx+mw+10.5, yt, '#555555', .4)
            text(c, symbol, mx+mw+12.5, yt-2.3, 7., font='Arial-Italic', role='gene')
            if symbol != original:
                stripped_symbols.setdefault(panel, []).append(dict(original=original, displayed=symbol))
        text(c, panel.lower(), 9, height-15, 14, font='Arial-Bold', role='panel_letter')
        text(c, str(item.get('title', FATE_BY_PANEL.get(panel, 'Common early path'))), mx, height-14,
             10, font='Arial-Bold', role='title')
        stage_values = cells['stage'].astype(str) if 'stage' in cells else cells['stage_standardized'].astype(str)
        present = [s for s in STAGES if s in set(stage_values)]
        stage_text = (present[0]+' - '+present[-1]) if present else 'Observed stages'
        text(c, f'{stage_text} | {n_cells:,} cells | {n_genes:,} genes', mx, height-29, 6.4, fill='#555555', role='subtitle')
        for fraction in (0., .5, 1.):
            x = mx+mw*fraction
            line(c, x, my, x, my-2.2, '#333333', .45)
            text(c, str(int(round(n_cells*fraction))), x, my-9, 5.7, align='center', role='tick')
        text(c, 'Cells ordered by original pseudotime', mx+mw/2, my-20, 6.2, align='center', role='axis')

        # The two continuous colorbars are also made from independent vector strips.
        expr_cmap = item.get('expression_cmap_rgb')
        if expr_cmap is None:
            from matplotlib import colormaps
            expr_cmap = colormaps['RdBu_r'](np.linspace(0, 1, 256))[:, :3]
        expr_box = (MATRIX[0], .565, 1.12, .05)
        vector_strip(c, expr_cmap, expr_box, frame=False)
        low, high = map(float, item['color_limits'])
        text(c, 'Scaled expression', expr_box[0]*72, 51, 6., role='legend')
        for fraction, value in [(0., low), (1., high)]:
            text(c, f'{value:.2f}', (expr_box[0]+expr_box[2]*fraction)*72, expr_box[1]*72-7,
                 5.4, align='center', role='tick')
        pseudo_cmap = item.get('pseudotime_cmap_rgb')
        if pseudo_cmap is None:
            from matplotlib import colormaps
            name = item.get('pseudotime_cmap', {'C': 'viridis', 'D': 'Reds', 'E': 'Blues', 'F': 'Greens', 'G': 'Purples'}[panel])
            pseudo_cmap = colormaps[name](np.linspace(0, 1, 256))[:, :3]
        pseudo_box = (MATRIX[0]+1.53, .565, .70, .05)
        vector_strip(c, pseudo_cmap, pseudo_box, frame=False)
        text(c, 'Original path distance', pseudo_box[0]*72, 51, 6., role='legend')
        bounds = item.get('pt_limits')
        if bounds is None:
            column = 'path_t' if 'path_t' in cells else 't'
            data = cells[column].to_numpy(float)
            bounds = [float(np.percentile(data, 2)), float(data.max())]
        for fraction, value in [(0., bounds[0]), (1., bounds[1])]:
            text(c, f'{float(value):.2f}', (pseudo_box[0]+pseudo_box[2]*fraction)*72, pseudo_box[1]*72-7,
                 5.4, align='center', role='tick')
        text(c, 'Age', 14, 21, 5.8, font='Arial-Bold', role='legend')
        for j, stage in enumerate(present):
            x = 39+j*35
            circle(c, x, 22, 1.8, STAGE_COLORS[STAGES.index(stage)])
            text(c, stage, x+4.5, 20.5, 5.7, role='legend')
        text(c, 'Tree: accepted log1p display. Heatmap: original fitted expression and cell ordering.', 14, 7, 5.5,
             fill='#555555', role='note')

        for entry in c._scene:
            x0, y0, x1, y1 = entry['bbox']
            assert min(x0, y0) >= -.1 and x1 <= width+.1 and y1 <= height+.1, ('Clipped text', panel, entry)
        gene_boxes = [entry for entry in c._scene if entry['role'] == 'gene']
        for i, a in enumerate(gene_boxes):
            for b in gene_boxes[i+1:]:
                x0, y0, x1, y1 = a['bbox']; p0, q0, p1, q1 = b['bbox']
                assert x1 <= p0 or p1 <= x0 or y1 <= q0 or q1 <= y0, ('Overlapping genes', panel, a, b)
        scenes[panel] = c._scene
        c.showPage(); c.save()
        subprocess.run([str(pdftoppm), '-singlefile', '-r', str(DPI), '-png', str(pdf), str(png.with_suffix(''))],
                       check=True, capture_output=True)
        with Image.open(png) as rendered:
            assert all(abs(px-v*DPI) <= 1.01 for px, v in zip(rendered.size, SIZE))
            pixels, actual_dpi = list(rendered.size), list(rendered.info['dpi'])
        artifacts[panel] = dict(pdf=str(pdf), png=str(png), inches=list(SIZE), pixels=pixels, dpi=actual_dpi,
                                pdf_sha256=hashlib.sha256(pdf.read_bytes()).hexdigest(),
                                png_sha256=hashlib.sha256(png.read_bytes()).hexdigest(),
                                expected_images=0, expected_clips=0, expected_forms=0, heatmap_vector=vector_report, genes=n_genes,
                                cells=n_cells, gene_labels=[x[1] for x in label_rows])
        mapped = np.column_stack(tree_map(xy[:, 0], xy[:, 1]))
        nodes = np.column_stack(graph_map(np.array([node_x[i] for i in range(len(node_y))]), node_y))
        mappings[panel] = dict(cell_xy_pt_sha256=hashlib.sha256(mapped.tobytes()).hexdigest(),
                               node_xy_pt_sha256=hashlib.sha256(nodes.tobytes()).hexdigest(),
                               selected_ids_sha256=hashlib.sha256('\n'.join(cell_ids).encode()).hexdigest(),
                               tree_axes_inches=[TREE_LEFT, MATRIX[1], TREE_WIDTH, TREE_HEIGHT],
                               x_bounds=[XMIN, XMAX], y_bounds=[ymin, ymax],
                               uniform_scale_from_AB=TREE_SCALE, point_diameter_pt=TREE_POINT_DIAMETER)
    assert len({v['cell_xy_pt_sha256'] for v in mappings.values()}) == 1
    assert len({v['node_xy_pt_sha256'] for v in mappings.values()}) == 1
    assert hashlib.sha256(xy.tobytes()).hexdigest() == original_xy_sha
    return dict(artifacts=artifacts, scenes=scenes, image_bbox_by_panel=image_boxes,
                coordinate_mappings=mappings, cleaned_gene_labels=stripped_symbols,
                render_spec=dict(size_inches=SIZE, matrix_inches=list(MATRIX), tree_uniform_scale=TREE_SCALE,
                                 tree_point_diameter_pt=TREE_POINT_DIAMETER, expected_clips=0,
                                 expected_images_per_page=0, expected_forms=0,
                                 image_policy='Vector heatmap RGB runs and editable trees, strips and semantic text; no image objects',
                                  heatmap_underpaint_pt=UNDERPAINT_PT,
                                 fonts=['ArialMT', 'Arial-BoldMT', 'Arial-ItalicMT']))


In [ ]:
"""Strict publication checks; importing this module creates no artifacts.

Entry point: validate_publication(bundle, display, heatmaps, artifacts, project, work).
Each artifact supplies pdf, png, inches (or size_in), and CG additionally supplies
heatmap_matrix_bbox_pt=(x0,y0,x1,y1) (legacy heatmap_image_bbox_pt is also accepted),
plus heatmap_key if its key differs. CG matrices are audited as vector rectangles, never images.
artifacts['_meta']['protected_file_sha256'] supplies the protected old files;
bundle['publication_protected_file_sha256'] is the default manifest.
Optional metadata: python, pdftoppm, validator, expected_ab_count, expected_cg_count.
The function returns a report; it never saves a report or rewrites a deliverable.
"""

from __future__ import annotations

from collections import Counter
from copy import copy
import hashlib
import math
import os
from pathlib import Path
import subprocess
import sys

import numpy as np


class PublicationValidationError(AssertionError):
    pass


def _require(condition, message):
    if not condition:
        raise PublicationValidationError(message)


def _hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def _same(actual, expected, location):
    if isinstance(expected, np.ndarray):
        try:
            equal = np.array_equal(actual, expected, equal_nan=True)
        except TypeError:
            equal = np.array_equal(actual, expected)
    elif hasattr(expected, 'equals'):
        equal = hasattr(actual, 'equals') and actual.equals(expected)
    elif isinstance(expected, dict):
        _require(isinstance(actual, dict) and actual.keys() == expected.keys(), f'{location}: dictionary keys changed')
        for key in expected:
            _same(actual[key], expected[key], f'{location}.{key}')
        return
    elif isinstance(expected, (list, tuple)):
        _require(len(actual) == len(expected), f'{location}: length changed')
        for index, value in enumerate(expected):
            _same(actual[index], value, f'{location}[{index}]')
        return
    elif isinstance(expected, float) and math.isnan(expected):
        equal = math.isnan(actual)
    else:
        equal = actual == expected
    _require(bool(equal), f'{location}: differs from the accepted input')


def _array_hash(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()


def _validate_frozen_inputs(bundle, display, project):
    """Use the archived inputs and accepted hashes, without a Tmp dependency."""
    source = bundle['archived_inputs']['old']
    expected = bundle['accepted_display_snapshot']
    model, projection = bundle['model'], bundle['projection']
    hashes = {'source_F_sha256': _array_hash(model['F']),
              'source_B_sha256': _array_hash(model['B']),
              'original_t_sha256': _array_hash(projection['t']),
              'node_y_sha256': _array_hash(display['node_y']),
              'cell_xy_sha256': _array_hash(display['cell_xy'])}
    for key, value in hashes.items():
        _require(value == expected[key], f'{key}: differs from the accepted snapshot')
    for key in ('obs', 'X_css30', 'plot_order'):
        _same(bundle[key], source[key], key)
    for path in ('Ia', 'Ib', 'Ic', 'II'):
        _same(bundle['figures'][path]['mask'], source['figures'][path]['mask'], f'{path}.mask')
    _same(display, bundle['publication_display'], 'publication_display argument')
    _same(bundle['whole_display_geometry']['cell_xy'], display['cell_xy'], 'whole_display_geometry.cell_xy')
    _require(len(bundle['obs']) == 7355 and bundle['obs'].index.is_unique, 'Expected 7355 unique original cells')
    registration = bundle['registration']
    expected_parameters = {'method': 'conditional component graft',
                           'local_selected_key': registration['selected_key'],
                           'common_space': 'shared-cell barycentric CSS30 transfer',
                           'bridge_nodes': (int(bundle['local_sgn_exit']), int(bundle['interface_anchor']))}
    _same(model['parameters'], expected_parameters, 'model.parameters')
    _require(display['tau_css30'] == .25, 'Accepted complete-branch log tau changed')
    F, B = np.asarray(model['F']), np.asarray(model['B'])
    edges = np.asarray(projection['edges'], int)
    edge_id, alpha = np.asarray(projection['edge_id'], int), np.asarray(projection['alpha'], float)
    distance, node_y = np.asarray(projection['root_distance']), np.asarray(display['node_y'])
    _require(F.shape == (30,160) and B.shape == (160,160) and edges.shape == (159,2), 'Model topology dimensions changed')
    _require(np.array_equal(B, B.T) and int(np.sum(B != 0)) == 318, 'Tree adjacency is not 159 undirected edges')
    _require(edge_id.shape == alpha.shape == (7355,) and np.isfinite(alpha).all() and
             np.all((alpha >= 0) & (alpha <= 1)), 'Invalid original projection edge/alpha')
    _require(np.all((edge_id >= 0) & (edge_id < 159)), 'Projection edge index outside tree')
    lengths = np.linalg.norm(F[:,edges[:,1]]-F[:,edges[:,0]],axis=0)
    _require(np.all(B[edges[:,0],edges[:,1]] != 0), 'Projection edges differ from model adjacency')
    _require(np.allclose(distance[edges[:,1]]-distance[edges[:,0]],lengths,rtol=0,atol=2e-12),
             'Original root distances disagree with CSS30 edge lengths')
    reconstructed_t = distance[edges[edge_id,0]] + alpha*lengths[edge_id]
    _require(np.allclose(projection['t'],reconstructed_t,rtol=0,atol=2e-12), 'Original t/edge/alpha are inconsistent')
    _same(bundle['linear_display_geometry']['cell_xy'][:,1], projection['t'], 'stashed linear geometry t')
    for key, actual in [('original_t_sha256',projection['t']),('original_root_distance_sha256',distance),
                        ('original_model_F_sha256',F),('original_model_B_sha256',B)]:
        _require(display[key] == _array_hash(actual), f'display.{key}: source hash mismatch')
    _require(display['display_xy_sha256'] == hashes['cell_xy_sha256'], 'Display coordinate hash mismatch')
    branches = display['branches']
    _require(len(branches) == 9, 'Expected nine maximal degree-2 branches')
    degree = np.count_nonzero(B,axis=1)
    covered, branch_by_edge = [], np.full(159,-1,int)
    independent_y = np.full(160,np.nan); independent_y[int(projection['root'])] = 0
    pending = list(branches)
    while pending:
        ready = [branch for branch in pending if np.isfinite(independent_y[int(branch['start'])])]
        _require(bool(ready), 'Log display branch hierarchy is disconnected')
        for branch in ready:
            path = np.asarray(branch['path'],int); ids = np.asarray(branch['edge_ids'],int)
            _require(path[0] == branch['start'] and path[-1] == branch['end'] and len(ids)==len(path)-1,
                     'Invalid maximal branch path')
            _require(all(degree[n]==2 for n in path[1:-1]), 'Branch split depends on an internal degree-2 node')
            _require(all(n==projection['root'] or degree[n]!=2 for n in (path[0],path[-1])), 'Branch endpoint is not a root/fork/tip')
            _same(edges[ids],np.column_stack((path[:-1],path[1:])), 'branch edge coverage')
            length = float(np.sum(lengths[ids])); transformed = .25*np.log1p(length/.25)
            _require(np.isclose(branch['original_length'],length,rtol=0,atol=2e-12) and
                     np.isclose(branch['display_length'],transformed,rtol=0,atol=2e-12), 'Incorrect complete-branch log transform')
            independent_y[path] = independent_y[path[0]] + np.r_[0,np.cumsum(lengths[ids])]*(transformed/length)
            covered.extend(ids.tolist()); branch_by_edge[ids] = int(branch['id'])
            pending = [item for item in pending if item is not branch]
    _require(sorted(covered) == list(range(159)), 'Nine branches do not cover every edge exactly once')
    _require(np.allclose(node_y,independent_y,rtol=0,atol=2e-12), 'Display node y differs from complete-branch log transform')
    expected_y = node_y[edges[edge_id,0]]+alpha*(node_y[edges[edge_id,1]]-node_y[edges[edge_id,0]])
    _require(np.allclose(display['cell_xy'][:,1],expected_y,rtol=0,atol=2e-12), 'Display cell y is not original-alpha interpolation')
    _same(display['edge_branch'],branch_by_edge,'edge_branch')
    _same(display['cell_branch'],branch_by_edge[edge_id],'cell_branch')
    _require(display['metric_segment_to_complete_branch'][2] == display['metric_segment_to_complete_branch'][3] ==
             display['metric_segment_to_complete_branch'][4], 'Metric segments 2/3/4 incorrectly transformed as separate branches')
    layout = display['horizontal_layout']; moved = np.asarray(layout['moved_cell_mask'],bool)
    _require(int(moved.sum()) == 175 and layout['VGN_root_span_scale'] == .5 and layout['displayed_x_shift'] == .5,
             'Accepted VGN horizontal layout changed')
    expected_x = np.array(bundle['linear_display_geometry']['cell_xy'][:,0],copy=True); expected_x[moved] += .5
    _same(display['cell_xy'][:,0],expected_x,'VGN horizontal cell shift')
    moved_nodes = set(map(int,layout['moved_node_ids']))
    _require(len(moved_nodes)==6 and int(projection['root']) not in moved_nodes, 'VGN node move includes root or incorrect node count')
    _same(display['node_x'],{node:x+(1 if int(node) in moved_nodes else 0) for node,x in bundle['node_x'].items()},
          'VGN horizontal node shift')
    _same(display['segment_x'],np.asarray([s['x'] for s in bundle['segments']])+np.isin(np.arange(11),layout['moved_metric_segment_ids']),
          'VGN horizontal segment shift')
    colors = bundle['segment_gradient_coloring']
    _require(_array_hash(colors['color_rgb_uint8']) == colors['colors_sha256'], 'Original gradient colors changed')
    density = bundle['stage_density_display']; grid = np.asarray(density['grid'])
    _require(np.all(np.diff(grid)>0), 'Display KDE grid is not increasing')
    integrate = np.trapezoid if hasattr(np,'trapezoid') else np.trapz
    for stage, values in density['values'].items():
        values = np.asarray(values)
        _require(np.isfinite(values).all() and np.all(values>=0) and
                 np.isclose(integrate(values,grid),1,rtol=0,atol=2e-10), f'{stage}: transformed KDE is not unit area')
        _require(int(density['counts'][stage]) == int(np.sum(bundle['obs']['stage_standardized'].astype(str)==str(stage))),
                 f'{stage}: transformed KDE cell count differs from original obs')
    return {'status':'PASS','cells':7355,'nodes':160,'edges':159,'complete_branches':9,'accepted_hashes':hashes,
            'five_views_share_display_sha256':hashes['cell_xy_sha256'],'source_model_and_t_preserved':True,
            'display_only_transform':True,'VGN_horizontal_moved_cells':175}


def _object(value):
    return value.get_object() if hasattr(value, 'get_object') else value


def _arial_style(font):
    name = str(_object(font).get('/BaseFont', '')).split('+')[-1]
    normalized = ''.join(c for c in name.lower() if c.isalnum())
    _require(normalized.startswith('arial'), f'Unexpected PDF font: {name}')
    bold, italic = 'bold' in normalized, ('italic' in normalized or 'oblique' in normalized)
    _require(not (bold and italic), f'Unexpected Arial bold-italic font: {name}')
    return 'italic' if italic else 'bold' if bold else 'regular'


def _check_resources(page):
    resources = _object(page['/Resources'])
    seen = set()

    def inspect(value):
        value = _object(value)
        if not isinstance(value, (dict, list, tuple)) or id(value) in seen:
            return
        seen.add(id(value))
        if isinstance(value, dict):
            for key in ('/SMask', '/Mask', '/ImageMask'):
                _require(key not in value, f'Forbidden PDF mask key {key}')
            _require(str(value.get('/S', '')) != '/Transparency', 'Transparency group is forbidden')
            for key in ('/Pattern', '/Shading'):
                _require(not value.get(key), f'Forbidden PDF resource {key}')
            for item in value.values():
                inspect(item)
        else:
            for item in value:
                inspect(item)

    inspect(resources)
    if '/Group' in page:
        inspect(page['/Group'])
    fonts = _object(resources.get('/Font', {}))
    _require(bool(fonts), 'PDF has no font resources')
    font_report = {}
    for key, raw in fonts.items():
        font = _object(raw)
        style = _arial_style(font)
        cmap = _object(font.get('/ToUnicode'))
        _require(cmap is not None and hasattr(cmap, 'get_data') and b'begincmap' in cmap.get_data(),
                 f'{key}: missing usable ToUnicode CMap')
        descendants = _object(font.get('/DescendantFonts', []))
        faces = [_object(descendants[0])] if descendants else [font]
        for face in faces:
            descriptor = _object(face.get('/FontDescriptor', {}))
            _require(any(k in descriptor for k in ('/FontFile', '/FontFile2', '/FontFile3')),
                     f'{key}: font is not embedded')
        font_report[str(key)] = {'name': str(font.get('/BaseFont')), 'style': style,
                                 'embedded': True, 'ToUnicode': True}
    return resources, font_report


def _multiply(left, right):
    a,b,c,d,e,f = left
    g,h,i,j,k,l = right
    return (a*g+c*h, b*g+d*h, a*i+c*j, b*i+d*j, a*k+c*l+e, b*k+d*l+f)


def _text_records(page, reader, operations):
    """Decode individual real BT/ET objects with public pypdf text visitors."""
    from pypdf.generic import ContentStream, FloatObject, NameObject
    records, stack = [], []
    matrix = (1.,0.,0.,1.,0.,0.)
    block, block_matrix = None, None
    font_state = None
    inherited_font = None
    for operands, operator in operations:
        if operator == b'q':
            stack.append((matrix, font_state))
        elif operator == b'Q':
            _require(bool(stack), 'Unbalanced Q operator')
            matrix, font_state = stack.pop()
        elif operator == b'cm':
            matrix = _multiply(matrix, tuple(map(float, operands)))
        if operator == b'BT':
            _require(block is None, 'Nested BT text objects')
            block, block_matrix, inherited_font = [], matrix, font_state
        if block is not None:
            block.append((operands, operator))
        if operator == b'Tf':
            font_state = (str(operands[0]), float(operands[1]))
        if operator != b'ET':
            continue
        _require(block is not None, 'ET without BT')
        shows = [(args, op) for args, op in block if op in (b'Tj', b'TJ', b"'", b'"')]
        if shows:
            copied = list(block)
            if not any(op == b'Tf' for _, op in copied) and inherited_font:
                copied.insert(1, ([NameObject(inherited_font[0]), FloatObject(inherited_font[1])], b'Tf'))
            stream = ContentStream(None, reader)
            stream.operations = [([FloatObject(x) for x in block_matrix], b'cm')] + copied
            scratch = copy(page)
            scratch[NameObject('/Contents')] = stream
            visits = []

            def visitor(text, cm, tm, font_dict, font_size):
                if text.strip():
                    x, y = float(tm[4]), float(tm[5])
                    visits.append({'text': text, 'font': font_dict, 'font_size': float(font_size),
                                   'xy': [float(cm[0])*x+float(cm[2])*y+float(cm[4]),
                                          float(cm[1])*x+float(cm[3])*y+float(cm[5])]})

            text = scratch.extract_text(visitor_text=visitor).strip('\r\n')
            strings = []
            for args, op in shows:
                values = args[0] if op == b'TJ' else [args[-1]]
                strings.extend(v for v in values if isinstance(v, (str, bytes, bytearray)) and len(v))
            records.append({'text': text, 'visits': visits, 'text_operators': [op.decode() for _,op in shows],
                            'string_count': len(strings), 'BT_index': len(records)})
        block = None
    _require(block is None and not stack, 'Unbalanced PDF text/graphics state')
    return records


def _image_calls(operations, resources):
    objects = _object(resources.get('/XObject', {}))
    image_names = []
    for name, raw in objects.items():
        obj = _object(raw)
        subtype = str(obj.get('/Subtype', ''))
        _require(subtype != '/Form', f'Form XObject {name} is forbidden')
        _require(subtype == '/Image', f'Unexpected XObject subtype {subtype}')
        image_names.append(str(name))
    matrix, stack, calls = (1.,0.,0.,1.,0.,0.), [], []
    for operands, operator in operations:
        _require(operator not in (b'W', b'W*', b'INLINE IMAGE', b'BI'), f'Forbidden clip/inline-image operator {operator}')
        if operator == b'q':
            stack.append(matrix)
        elif operator == b'Q':
            _require(bool(stack), 'Unbalanced Q operator')
            matrix = stack.pop()
        elif operator == b'cm':
            matrix = _multiply(matrix, tuple(map(float, operands)))
        elif operator == b'Do':
            name = str(operands[0])
            _require(name in image_names, f'Unknown or non-image Do call {name}')
            a,b,c,d,e,f = matrix
            corners = [(e,f),(a+e,b+f),(c+e,d+f),(a+c+e,b+d+f)]
            calls.append({'name': name, 'matrix': list(matrix),
                          'bbox_pt': [min(x for x,y in corners),min(y for x,y in corners),
                                      max(x for x,y in corners),max(y for x,y in corners)]})
    _require(not stack, 'Unbalanced q/Q graphics state')
    return objects, image_names, calls


def _gene_audit(records, labels, artifact, heatmap, page):
    expected = Counter(map(str, labels))
    _require(expected and all(label and not any(c in label for c in '*\u2020\u00b0') for label in expected),
             'Expected gene labels contain blanks or forbidden annotation marks')
    found = Counter()
    gene_records = []
    width, height = float(page.mediabox.width), float(page.mediabox.height)
    for record in records:
        _require(not any(c in record['text'] for c in '*\u2020\u00b0'),
                 f'Forbidden annotation mark in PDF text object {record["BT_index"]}: {record["text"]!r}')
        label = record['text']
        if label not in expected:
            continue
        _require(record['text_operators'] in (['Tj'], ['TJ']) and record['string_count'] == 1,
                 f'{label}: gene must be one whole Tj/TJ string in its own BT/ET object')
        _require(bool(record['visits']), f'{label}: no decoded text visitor event')
        for visit in record['visits']:
            _require(visit['font'] is not None and _arial_style(visit['font']) == 'italic',
                     f'{label}: gene is not actual embedded ArialItalic text')
            x,y = visit['xy']
            _require(0 <= x <= width and 0 <= y <= height, f'{label}: text origin lies outside the page')
        found[label] += 1
        gene_records.append({'label': label, 'BT_index': record['BT_index'],
                             'xy': record['visits'][0]['xy'], 'font_size': record['visits'][0]['font_size'],
                             'font': str(record['visits'][0]['font'].get('/BaseFont'))})
    _require(found == expected, f'Gene-object count mismatch: missing={dict(expected-found)}, extra={dict(found-expected)}')
    axis = artifact.get('gene_axis', heatmap.get('gene_axis', 'rows'))
    if axis in ('rows','columns'):
        coordinate = 1 if axis == 'rows' else 0
        ordered = sorted(gene_records, key=lambda r:r['xy'][coordinate], reverse=axis=='rows')
        _require([r['label'] for r in ordered] == list(map(str, labels)),
                 f'Gene text positions do not follow matrix {axis} order')
        bbox = artifact.get('heatmap_matrix_bbox_pt') or artifact['heatmap_image_bbox_pt']
        lo,hi = (float(bbox[1]),float(bbox[3])) if axis=='rows' else (float(bbox[0]),float(bbox[2]))
        _require(all(lo-r['font_size'] <= r['xy'][coordinate] <= hi+r['font_size'] for r in gene_records),
                 'Gene text positions do not lie alongside the heatmap matrix')
    return gene_records


def _heatmap_vector_audit(page, reader, artifact, heatmap):
    """Reconstruct frozen RGB cells from exact, opaque PDF rectangle runs."""
    from pypdf.generic import ContentStream, DecodedStreamObject
    raw = page.get_contents().get_data()
    markers = (b'%%HEATMAP_VECTOR_BEGIN', b'%%HEATMAP_MAIN_BEGIN', b'%%HEATMAP_VECTOR_END')
    _require(all(raw.count(marker) == 1 for marker in markers),
             'Expected exactly one vector-begin, main-begin and vector-end marker')
    begin, main, end = (raw.index(marker) for marker in markers)
    _require(begin < main < end, 'Heatmap vector markers are out of order')

    def operations(data):
        stream = DecodedStreamObject()
        stream.set_data(data)
        return ContentStream(stream, reader).operations

    # Exact page coordinates require an identity inherited CTM.  A local q does
    # not reset alpha or blend state, so check their effective inherited values.
    matrix, fill_alpha, stroke_alpha, blend = (1.,0.,0.,1.,0.,0.), 1., 1., '/Normal'
    stack = []
    resources = _object(page.get('/Resources', {}))
    states = _object(resources.get('/ExtGState', {}))
    for operands, operator in operations(raw[:begin]):
        if operator == b'q':
            stack.append((matrix, fill_alpha, stroke_alpha, blend))
        elif operator == b'Q':
            _require(bool(stack), 'Unbalanced Q before vector heatmap')
            matrix, fill_alpha, stroke_alpha, blend = stack.pop()
        elif operator == b'cm':
            matrix = _multiply(matrix, tuple(map(float, operands)))
        elif operator == b'gs':
            name = operands[0]
            _require(name in states, f'Unknown graphics state before heatmap: {name}')
            state = _object(states[name])
            fill_alpha = float(state.get('/ca', fill_alpha))
            stroke_alpha = float(state.get('/CA', stroke_alpha))
            blend = str(state.get('/BM', blend))
    _require(np.allclose(matrix, (1.,0.,0.,1.,0.,0.), rtol=0, atol=1e-10),
             'Heatmap inherits a nonidentity page transform')
    _require(fill_alpha == stroke_alpha == 1. and blend == '/Normal',
             'Heatmap must inherit fully opaque normal blending')

    under_ops = operations(raw[begin + len(markers[0]):main])
    main_ops = operations(raw[main + len(markers[1]):end])
    _require(under_ops and under_ops[0][1] == b'q' and main_ops and main_ops[-1][1] == b'Q',
             'Heatmap vector block must start with q and end with Q')
    _require(sum(op == b'q' for _,op in under_ops + main_ops) == 1 and
             sum(op == b'Q' for _,op in under_ops + main_ops) == 1,
             'Heatmap vector block must contain exactly one q/Q pair')
    under_ops, main_ops = under_ops[1:], main_ops[:-1]
    expected = np.asarray(heatmap['rgb'])
    _require(expected.dtype == np.uint8 and expected.ndim == 3 and expected.shape[2] == 3 and
             expected.shape[0] > 0 and expected.shape[1] > 0, 'Frozen heatmap must be a nonempty uint8 RGB matrix')
    nrows, ncols = expected.shape[:2]
    bbox = list(map(float, artifact.get('heatmap_matrix_bbox_pt') or artifact['heatmap_image_bbox_pt']))
    _require(len(bbox) == 4 and np.isfinite(bbox).all() and bbox[2] > bbox[0] and bbox[3] > bbox[1],
             'Invalid heatmap matrix bounding box')
    mx, my, xmax, ymax = bbox
    dx, dy = (xmax-mx)/ncols, (ymax-my)/nrows
    geometry_atol = 2e-7
    expected_keys = ((expected[...,0].astype(np.uint32) << 16) |
                     (expected[...,1].astype(np.uint32) << 8) | expected[...,2].astype(np.uint32))
    palette = np.unique(expected_keys).tolist()
    reconstructed = np.empty_like(expected)
    coverage_delta = np.zeros((nrows,ncols+1), np.int64)
    rgb_delta = np.zeros((nrows,ncols+1), np.int64)

    def color(operands):
        values = np.asarray(list(map(float, operands)))
        _require(values.shape == (3,) and np.isfinite(values).all() and
                 np.all((values >= 0) & (values <= 1)), 'Invalid heatmap RGB operands')
        rgb = np.rint(values*255).astype(np.uint8)
        _require(np.allclose(values, rgb.astype(float)/255, rtol=0, atol=6e-11),
                 'Heatmap RGB is not the frozen 8-bit color at ten-decimal precision')
        return (int(rgb[0]) << 16) | (int(rgb[1]) << 8) | int(rgb[2]), tuple(map(int,rgb))

    def rectangles(operands):
        # Convert and validate each whole color group once; precision/tolerance
        # is identical to the former per-rectangle float64 check.
        values = np.asarray(operands, dtype=float)
        _require(values.ndim == 2 and values.shape[1] == 4 and len(values) > 0 and
                 np.isfinite(values).all() and np.all(values[:,2:] > 0), 'Invalid heatmap rectangle')
        with np.errstate(over='ignore', invalid='ignore'):
            row = np.rint((ymax-values[:,1]-values[:,3])/dy)
            start = np.rint((values[:,0]-mx)/dx)
            stop = np.rint((values[:,0]+values[:,2]-mx)/dx)
        _require(np.all((row >= 0) & (row < nrows) & (start >= 0) & (start < stop) & (stop <= ncols)),
                 'Heatmap rectangle lies outside frozen matrix axes')
        result = np.column_stack((row,start,stop)).astype(np.int64)
        row,start,stop = result.T
        exact = np.column_stack((mx+start*dx, my+(nrows-row-1)*dy, (stop-start)*dx, np.full(len(row),dy)))
        _require(np.allclose(values, exact, rtol=0, atol=geometry_atol),
                 'Heatmap rectangle is not on an exact frozen-matrix cell boundary')
        return result

    def scan(ops, underpaint):
        groups, pending = {}, []
        current, stroke, width = None, None, None
        previous = -1
        painted = False
        allowed = (b'rg',b'RG',b'w',b'j',b'J',b're',b'B',b'B*',b'n') if underpaint else (b'rg',b're',b'f',b'f*',b'n')
        for operands, operator in ops:
            _require(operator in allowed, f'Forbidden operator in heatmap {"underpaint" if underpaint else "main"} pass: {operator}')
            if operator == b'rg':
                _require(not pending and (current is None or painted), 'Unpainted or interrupted heatmap color group')
                current = color(operands)
                _require(current[0] > previous, 'Heatmap color groups are not unique ascending RGB keys')
                previous, painted = current[0], False
            elif operator == b'RG':
                stroke = color(operands)
            elif operator == b'w':
                _require(len(operands) == 1, 'Invalid underpaint stroke width')
                width = float(operands[0])
            elif operator in (b'j',b'J'):
                _require(len(operands) == 1 and float(operands[0]) == 0.,
                         'Heatmap underpaint requires miter joins and butt caps (j/J = 0)')
            elif operator == b're':
                _require(current is not None and not painted, 'Rectangle outside an active heatmap color group')
                pending.append(operands)
            elif operator == b'n':
                _require(not pending, 'Heatmap path discarded without painting')
            else:
                _require(current is not None and pending and not painted, 'Empty or repeated heatmap paint operation')
                key, rgb = current
                runs = rectangles(pending)
                if underpaint:
                    _require(stroke == current and width is not None and abs(width-.2) <= 1e-10,
                             'Heatmap underpaint must use matching RGB fill/stroke and exactly 0.2 pt width')
                else:
                    # Difference arrays preserve multiplicity (including repeated
                    # same-color intervals); no per-run slice or pixel averaging.
                    row,start,stop = runs.T
                    np.add.at(coverage_delta, (row,start), 1)
                    np.add.at(coverage_delta, (row,stop), -1)
                    np.add.at(rgb_delta, (row,start), key)
                    np.add.at(rgb_delta, (row,stop), -key)
                groups[key] = runs
                pending, painted = [], True
        _require(not pending and current is not None and painted, 'Incomplete heatmap vector pass')
        _require(list(groups) == palette, 'Heatmap vector palette differs from frozen RGB colors')
        return groups

    under_groups = scan(under_ops, True)
    main_groups = scan(main_ops, False)
    _require(under_groups.keys() == main_groups.keys() and
             all(np.array_equal(under_groups[key], main_groups[key]) for key in main_groups),
             'Underpaint and main pass rectangle runs differ')
    coverage = np.cumsum(coverage_delta, axis=1)[:,:ncols]
    _require(not np.any(coverage > 1), 'Heatmap main pass has overlapping cells')
    _require(np.all(coverage == 1), 'Heatmap main pass leaves gaps in one or more gene rows')
    rgb_keys = np.cumsum(rgb_delta, axis=1)[:,:ncols]
    reconstructed[...,0] = (rgb_keys >> 16).astype(np.uint8)
    reconstructed[...,1] = ((rgb_keys >> 8) & 255).astype(np.uint8)
    reconstructed[...,2] = (rgb_keys & 255).astype(np.uint8)
    _require(np.array_equal(reconstructed, expected), 'Reconstructed vector RGB matrix differs from frozen heatmap')
    runs = sum(len(value) for value in main_groups.values())
    maximal_runs = nrows + int(np.count_nonzero(np.any(expected[:,1:] != expected[:,:-1], axis=2)))
    _require(runs == maximal_runs, 'Heatmap rectangles are not maximal contiguous same-RGB runs')
    digest = _array_hash(reconstructed)
    _require(digest == _array_hash(expected), 'Vector-reconstructed RGB hash differs from frozen matrix')
    return {'status':'PASS','bbox_pt':bbox,'rows':nrows,'columns':ncols,'colors':len(palette),
            'main_rectangles':runs,'underpaint_rectangles':runs,'underpaint_width_pt':.2,
            'geometry_atol_pt':geometry_atol,'main_pass_fill_only':True,'row_coverage_complete':True,
            'frozen_rgb_exact':True,'rgb_sha256':digest,'reconstructed_rgb_sha256':digest,
            'source_rgb_sha256':_array_hash(expected),'inherited_identity_CTM':True,'opaque':True}


def _pdf_audit(path, artifact, heatmap):
    from pypdf import PdfReader
    from pypdf.generic import ContentStream
    reader = PdfReader(str(path))
    _require(len(reader.pages) == 1, f'{path.name}: expected one PDF page')
    page = reader.pages[0]
    resources, fonts = _check_resources(page)
    operations = ContentStream(page.get_contents(), reader).operations
    objects,names,calls = _image_calls(operations,resources)
    is_cg = heatmap is not None
    _require(len(names) == 0 and len(calls) == 0,
             f'{path.name}: expected zero Image XObjects and image calls; got {len(names)}/{len(calls)}')
    vector_report = _heatmap_vector_audit(page,reader,artifact,heatmap) if is_cg else None
    records = _text_records(page,reader,operations)
    _require(records, 'PDF has no semantic text objects')
    styles = {v['style'] for v in fonts.values()}
    required = set(artifact.get('required_font_styles',
                   ['regular','bold','italic'] if is_cg else ['regular','bold']))
    _require(required <= styles, f'Missing required Arial styles: {sorted(required-styles)}')
    if is_cg:
        labels = heatmap['labels']
        if hasattr(labels,'columns'):
            _require('symbol' in labels.columns, 'Heatmap label table lacks original symbol column')
            labels = labels['symbol'].astype(str).tolist()
        genes = _gene_audit(records,labels,artifact,heatmap,page)
    else:
        genes = []
    return {'status':'PASS','clips':0,'forms':0,'image_xobjects':len(names),'image_calls':len(calls),
            'image':None,'heatmap_vector':vector_report,'fonts':fonts,'semantic_text_objects':len(records),
            'gene_text_objects':genes,'page_size_pt':[float(page.mediabox.width),float(page.mediabox.height)]}


def validate_publication(bundle, display, heatmaps, artifacts, project, work):
    """Validate five vector-tree and five combined pairs; retain renders in work."""
    project,work = Path(project).resolve(),Path(work).resolve()
    _require(work.is_relative_to(ROOT), 'Validation/render directory must stay inside project Tmp')
    meta = artifacts.get('_meta', {})
    items = {k:v for k,v in artifacts.items() if not str(k).startswith('_') and isinstance(v,dict) and 'pdf' in v}
    _require(bool(items), 'No figure artifacts supplied')
    protected = meta.get('protected_file_sha256',bundle.get('publication_protected_file_sha256',{}))
    _require(bool(protected), 'Supply the protected non-overwritten old-file hash manifest')
    protected_resolved = {os.path.normcase(str(Path(path).resolve())) for path in protected}

    def protection_check():
        return {str(path):Path(path).is_file() and _hash(path)==expected for path,expected in protected.items()}

    protected_before = protection_check()
    _require(all(protected_before.values()), f'Protected old files changed: {[p for p,v in protected_before.items() if not v]}')
    frozen = _validate_frozen_inputs(bundle,display,project)
    primary = ROOT
    python = Path(meta.get('python',primary/'python/python.exe'))
    poppler = Path(meta.get('pdftoppm',primary/'native/poppler/Library/bin/pdftoppm.exe'))
    validator = ROOT
    for path in (python,poppler,validator):
        _require(path.is_file(), f'NOT CERTIFIED: validation dependency missing: {path}')
    try:
        import pypdf  # noqa: F401
    except ModuleNotFoundError:
        sys.path.append(str(primary/'python/Lib/site-packages'))
        import pypdf  # noqa: F401
    from PIL import Image
    render_dir = ROOT
    runtime = ROOT
    render_dir.mkdir(parents=True,exist_ok=True)
    runtime.mkdir(parents=True,exist_ok=True)
    env = dict(os.environ,PYTHONDONTWRITEBYTECODE='1',TMP=str(runtime),TEMP=str(runtime),MPLCONFIGDIR=str(runtime))
    reports = {}
    counts = Counter()
    for key,artifact in items.items():
        cg = 'heatmap_matrix_bbox_pt' in artifact or 'heatmap_image_bbox_pt' in artifact
        counts['CG' if cg else 'AB'] += 1
        heatmap = heatmaps[artifact.get('heatmap_key',key)] if cg else None
        size = artifact.get('inches',artifact.get('size_in'))
        _require(size is not None and len(size)==2, f'{key}: physical size is required')
        pdf,png = Path(artifact['pdf']),Path(artifact['png'])
        _require(pdf.is_file() and png.is_file(), f'{key}: missing PDF/PNG')
        _require(os.path.normcase(str(pdf.resolve())) not in protected_resolved and
                 os.path.normcase(str(png.resolve())) not in protected_resolved,
                 f'{key}: destination is in the protected old-file list')
        print(f'[publication validation] {key}: PDF structure, text, vector matrix and PNG',flush=True)
        report = _pdf_audit(pdf,artifact,heatmap)
        _require(np.allclose(report['page_size_pt'],np.asarray(size,float)*72,rtol=0,atol=.02), f'{key}: PDF physical size mismatch')
        with Image.open(png) as image:
            actual_pixels = list(image.size)
            expected_pixels = [round(float(v)*600) for v in size]
            _require(all(abs(a-b)<=1 for a,b in zip(actual_pixels,expected_pixels)), f'{key}: PNG pixel size mismatch')
            dpi = image.info.get('dpi',(0,0))
            _require(len(dpi)==2 and all(abs(float(v)-600)<1 for v in dpi), f'{key}: PNG is not 600 DPI')
            image.verify()
        command = [str(python),'-B',str(validator),str(pdf),'--expected-clips','0','--max-clips','0',
                   '--pdftoppm',str(poppler),'--render-dpi','150','--render-dir',str(render_dir)]
        result = subprocess.run(command,cwd=project,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=300)
        _require(result.returncode==0 and '[PASS]' in result.stdout,
                 f'{key}: mandatory PDF validator failed:\n{result.stdout[-16000:]}')
        png_command = [str(python),'-B',str(validator),str(png),'--target-dpi','600',
                       '--expected-size-in',f'{float(size[0])}x{float(size[1])}']
        png_result = subprocess.run(png_command,cwd=project,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=120)
        _require(png_result.returncode==0 and '[PASS]' in png_result.stdout,
                 f'{key}: mandatory PNG validator failed:\n{png_result.stdout[-16000:]}')
        reports[str(key)] = dict(report,pdf=str(pdf),png=str(png),png_pixels=actual_pixels,png_dpi=list(dpi),
                                  PDF_automatic_validation=result.stdout,PNG_automatic_validation=png_result.stdout)
    _require(counts['AB']==int(meta.get('expected_ab_count',5)) and counts['CG']==int(meta.get('expected_cg_count',5)),
             f'Expected five vector trees and five CG artifacts; got {dict(counts)}')
    protected_after = protection_check()
    _require(all(protected_after.values()), 'Protected old files changed during publication validation')
    return {'status':'PASS','frozen_numeric_contract':frozen,'artifacts':reports,
            'protected_old_files':protected_after,'render_dpi':150,'render_root':str(render_dir),
            'visual_review_required':True,'visual_review_note':'Inspect every retained PDF render; automatic checks do not certify visual layout.'}


In [ ]:
#9. =========== Export selected views; keep unrelated figure files unchanged
RENDER_TREE_ALL = True
RENDER_SUBTYPE_TREES = False
RENDER_COMPOSITES = False
SAVE_PUBLICATION = True

publication_display = bundle["publication_display"]
tree_render = deepcopy(bundle["publication_trees"])
composite_render = deepcopy(bundle["publication_composites"])
protected_figures = {}
changed_paths = set()
unavailable_prior_outputs = []
for key, record in {**tree_render["artifacts"], **composite_render["artifacts"]}.items():
    changed = ((key == "all" and RENDER_TREE_ALL) or
               (key in ("Ia", "Ib", "Ic", "II") and RENDER_SUBTYPE_TREES) or
               (key in "CDEFG" and RENDER_COMPOSITES))
    if not changed:
        for extension in ("pdf", "png"):
            path = record[extension]
            if Path(path).is_file():
                protected_figures[path] = sha_file(path)
            else:
                unavailable_prior_outputs.append(path)
    else:
        changed_paths.update(str(Path(record[k])) for k in ("pdf", "png"))
for folder in (TREE_DIR, COMPOSITE_DIR):
    for path in folder.iterdir():
        if path.suffix.lower() in (".pdf", ".png") and str(path) not in changed_paths:
            protected_figures[str(path)] = sha_file(path)

changed_artifacts = {}
if RENDER_SUBTYPE_TREES:
    # The original renderer reproduces the four accepted path layouts unchanged.
    assert RENDER_TREE_ALL, "The compact whole view must follow the legacy path render."
    tree_render = render_trees(bundle, publication_display, bundle["stage_density_display"], TREE_DIR, WORK)
    changed_artifacts.update({k:v for k,v in tree_render["artifacts"].items() if k != "all"})

if RENDER_TREE_ALL:
    compact_all = render_tree_all_compact(bundle, publication_display, bundle["stage_density_display"], TREE_DIR, WORK)
    tree_render["artifacts"]["all"] = compact_all["artifact"]
    tree_render["scenes"]["all"] = compact_all["scene"]
    tree_render["physical_coordinate_checks"]["all"] = compact_all["physical"]
    tree_render["physicalmetadata"]["all"] = compact_all["physical"]
    tree_render["render_spec"]["whole_size"] = compact_all["artifact"]["inches"]
    tree_render["render_spec"]["compact_all"] = compact_all["render_spec"]
    tree_render["render_spec"]["all_views_share_normalized_geometry"] = True
    tree_render["render_spec"]["all_views_share_physical_layout"] = False
    tree_render["render_spec"]["legacy_axes_apply_to"] = ["Ia", "Ib", "Ic", "II"]
    changed_artifacts["all"] = compact_all["artifact"]

if RENDER_COMPOSITES:
    previous_composites = deepcopy(composite_render)
    composite_render = render_composites(bundle, publication_display, bundle["heatmaps"], COMPOSITE_DIR, WORK)
    for panel, record in composite_render["artifacts"].items():
        box = composite_render["image_bbox_by_panel"][panel]
        record["heatmap_key"] = panel
        record["heatmap_matrix_bbox_pt"] = [box["x"], box["y"], box["x"]+box["width"], box["y"]+box["height"]]
        record["heatmap_image_bbox_pt"] = record["heatmap_matrix_bbox_pt"]
        assert composite_render["scenes"][panel] == previous_composites["scenes"][panel]
        _same(composite_render["coordinate_mappings"][panel],previous_composites["coordinate_mappings"][panel],
              f"{panel}.tree_coordinate_mappings")
    changed_artifacts.update(composite_render["artifacts"])

assert changed_artifacts, "Choose at least one view to export."
assert all(sha_file(path) == expected for path, expected in protected_figures.items())
meta = {"expected_ab_count":sum(k in ("all","Ia","Ib","Ic","II") for k in changed_artifacts),
        "expected_cg_count":sum(k in "CDEFG" for k in changed_artifacts)}
current_check = validate_publication(bundle,publication_display,bundle["heatmaps"],
                                     {**changed_artifacts,"_meta":meta},PROJECT,WORK)
report = deepcopy(bundle["publication_validation"])
report["artifacts"].update(current_check["artifacts"])
for key in ("status","frozen_numeric_contract","protected_old_files","render_dpi","render_root"):
    report[key] = current_check[key]
for key in ("notebook_execution","code_file_sha256","visual_review","render_only_layout_check",
            "unchanged_formal_inputs_and_trees","render_only_migration","temporary_cleanup"):
    report.pop(key,None)
report["visual_review_required"] = True
report["visual_review_note"] = "Review the changed PDF pages; unchanged figures retain their earlier review."
report["code_file"] = str(ROOT)
report["result_file"] = str(RESULT)
report["heatmap_gene_counts"] = {p:bundle["heatmaps"][p]["genes_count"] for p in "CDEFG"}
report["current_revision"] = {
    "views":list(changed_artifacts),"VGN_preserved":True,
    "only_layout_and_display_labels_changed":True,
    "unchanged_other_figure_sha256":protected_figures,
    "unavailable_prior_output_paths_not_regenerated":unavailable_prior_outputs,
    "accepted_display_sha256":publication_display["display_xy_sha256"],
    "segment_legend":"256 opaque vector color bands per non-connector segment; original nominal RGB endpoints",
    "other_views_share_normalized_geometry":True,
    "physical_page_layouts_may_differ":True}
if SAVE_PUBLICATION:
    bundle["publication_trees"] = tree_render
    bundle["publication_composites"] = composite_render
    bundle["render_publication"] = tree_render
    bundle["render_spec"] = tree_render["render_spec"]
    bundle["figures"]["all"]["size"] = tuple(tree_render["artifacts"]["all"]["inches"])
    bundle["publication_validation"] = report
    bundle["publication_result_files"] = [str(RESULT), str(CHECK)]
    with RESULT.open("wb") as handle:
        pickle.dump(bundle,handle,protocol=5)
    with CHECK.open("w",encoding="utf-8") as handle:
        json.dump(report,handle,ensure_ascii=False,indent=2,
                  default=lambda value:value.item() if hasattr(value,"item") else str(value))
print("PASS: selected figure views exported; accepted graph, cells, annotations, colors and unrelated figures preserved.")


In [ ]:
#10. =========== Preview the compact whole tree
from IPython.display import Image as NotebookImage
show(NotebookImage(filename=str(TREE_DIR / "tree_all.png"), width=850))
